# 👀 Nie wierz własnym oczom! — wersja dla prowadzącego
![PBS WTIiE logo CMYK poziom PL-01.png](https://pbs.edu.pl/upload/media/default/0001/01/b6cecfbda538ed95961609f41725eeecb1c0aaff.svg) <br>
**Prowadzący:** Szymon Piórkowski, Mateusz Leśniewski <br>
**Data:** 20.10.2026

To nagranie wygląda prawdziwie, ale nikt go nie nakręcił. Dziś staniecie po drugiej stronie: zbudujecie deepfake od zera, nadacie fałszywy news.

## Co Was czeka
- 🟩 **Zielona kartka**, która staje się oknem do innego świata
- 🔢 **Kod miny:** Wasza twarz jako 478 punktów i 52 liczby, czyli to, czym „patrzą” filtry ze Snapchata
- 🔁 **Zamiana twarzy na żywo** z osobą z zespołu
- 🎭 **Kapibara-aktor**, która w 3D gra Waszą miną i mówi Waszymi powiedzonkami
- 📺 **Studio Kapibara TV:** absurdalny news
- 🔎 **Detektyw AI:** sieć neuronowa ocenia znane fałszywki z internetu. Czy da się ją oszukać?

Po zajęciach będziecie wiedzieć, jak powstają deepfake'i, po czym je poznać i dlaczego nie warto ślepo ufać ani własnym oczom, ani aplikacji, która obiecuje wykryć każdy fałsz. Wszystko działa na Waszym laptopie, bez internetu.

## Jak pracujecie
1. **Live coding:** prowadzący pisze kod na rzutniku, Wy piszecie to samo u siebie w miejscach `____`. Jeśli uruchomicie komórkę z lukami, program powie, czego brakuje.
2. Pętla programisty: zamknijcie obraz (`q` albo ■ Stop) → zmieńcie kod → `Shift + Enter` → sprawdźcie efekt.

## 📜 Zasady

1. **Tylko fikcja.** Kapibary i kosmici **tak**. Prawdziwi ludzie, politycy i marki **nie**.
2. **Nagrywacie tylko siebie** i osoby z zespołu, które się zgadzają.
3. **Żółty pasek „MATERIAŁ SYNTETYCZNY” zostaje** na każdym nagraniu (AI Act wymaga oznaczania deepfake'ów).
4. **Nic nie wychodzi z sali.** Modele działają lokalnie.
5. **W razie pytań.** Pytamy prowadzącego

In [ ]:
# ▶ 0.1 INSTALACJA — uruchamia prowadzący
import os
from importlib import metadata as _metadata

def _wersja(pakiet):
    try:
        return _metadata.version(pakiet)
    except _metadata.PackageNotFoundError:
        return ""

KARTA_NVIDIA = True               # Windows z kartą NVIDIA: detektor AI liczy na karcie (torch z CUDA, raz ok. 2 GB)
if not (_wersja("mediapipe") and _wersja("pillow")):
    %pip install mediapipe pillow
if not (_wersja("transformers") and _wersja("torch")):
    %pip install transformers torch
if KARTA_NVIDIA and os.name == "nt" and "+cu" not in _wersja("torch"):
    %pip uninstall -y torch
    %pip install torch --index-url https://download.pytorch.org/whl/cu130

# ▶ 0.2 Sprawdzenie bibliotek i pobranie modeli AI (ok. 20 MB, tylko za pierwszym razem)
import os, shutil, urllib.request
import cv2, mediapipe, numpy, PIL

print("MediaPipe", mediapipe.__version__, "| OpenCV", cv2.__version__,
      "| NumPy", numpy.__version__, "| Pillow", PIL.__version__)

try:
    cv2.namedWindow("test")
    cv2.destroyWindow("test")
    cv2.waitKey(1)
    print("✅ OpenCV może otwierać okna")
except cv2.error:
    print("❌ OpenCV nie umie otwierać okien. Uruchom:  %pip uninstall -y opencv-python-headless  i zrestartuj kernel.")

FOLDER_MODELI = "modele"
FOLDER_ZDJEC = "zdjecia" 
MODELE = {
    "face_landmarker.task":
        "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task",
    "selfie_multiclass_256x256.tflite":
        "https://storage.googleapis.com/mediapipe-models/image_segmenter/selfie_multiclass_256x256/float32/latest/selfie_multiclass_256x256.tflite",
}

os.makedirs(FOLDER_MODELI, exist_ok=True)
for nazwa, adres in MODELE.items():
    sciezka = os.path.join(FOLDER_MODELI, nazwa)
    if os.path.exists(sciezka) and os.path.getsize(sciezka) > 100_000:
        print("✅ model jest już na dysku:", sciezka)
        continue
    try:
        print("⏬ pobieram:", nazwa, "…")
        zapytanie = urllib.request.Request(adres, headers={"User-Agent": "FakeNewsStudio-warsztaty/1.0 (lab edukacyjny)"})
        with urllib.request.urlopen(zapytanie, timeout=60) as odpowiedz, open(sciezka + ".czesc", "wb") as plik:
            shutil.copyfileobj(odpowiedz, plik)
        os.replace(sciezka + ".czesc", sciezka)
        print("✅ pobrano:", sciezka)
    except Exception as e:
        print(f"❌ Nie udało się pobrać {nazwa}: {e}")
        if "CERTIFICATE" in str(e).upper():
            print("   macOS: uruchom 'Install Certificates.command' z folderu Pythona (Programy → Python 3.x).")
        print(f"   Poproś prowadzącego o folder '{FOLDER_MODELI}' z plikami modeli (np. z pendrive'a).")

SCIEZKA_TWARZ = os.path.join(FOLDER_MODELI, "face_landmarker.task")
SCIEZKA_SEGMENTACJA = os.path.join(FOLDER_MODELI, "selfie_multiclass_256x256.tflite")
for plik in ("kapibara_garnitur.png", "kapibara_trawa.jpg"):
    sciezka = os.path.join(FOLDER_ZDJEC, plik)
    print(("✅ zdjęcie jest na dysku: " if os.path.exists(sciezka) else "❌ brak zdjęcia (poproś prowadzącego): ") + sciezka)

DETEKTOR_MODEL = "Ateeqq/ai-vs-human-image-detector"   # wstępnie nauczony detektor AI (Hugging Face, licencja Apache-2.0)
FOLDER_DETEKTORA = os.path.join(FOLDER_MODELI, "detektor_ai")
# Windows: równoległe pobieranie z Hugging Face (Xet) potrafi stanąć na 0 B — zwykłe HTTPS, plik po pliku, jest pewniejsze
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
try:
    import transformers, torch
    if not any(os.path.exists(os.path.join(FOLDER_DETEKTORA, n)) for n in ("model.safetensors", "pytorch_model.bin")):
        import huggingface_hub
        from huggingface_hub import hf_hub_download, list_repo_files
        if hasattr(huggingface_hub.constants, "HF_HUB_DISABLE_XET"):
            huggingface_hub.constants.HF_HUB_DISABLE_XET = True
        print("⏬ pobieram detektor AI (ok. 400 MB, tylko za pierwszym razem; przerwać można przyciskiem ■ / Interrupt) …")
        pliki = [n for n in list_repo_files(DETEKTOR_MODEL)
                 if n in ("config.json", "preprocessor_config.json") or n.endswith(".safetensors")]
        for n in pliki:
            hf_hub_download(DETEKTOR_MODEL, n, local_dir=FOLDER_DETEKTORA)
    print("✅ detektor AI (LIVE 6) jest na dysku:", FOLDER_DETEKTORA)
except ImportError:
    print("⚠️ brak transformers/torch — LIVE 6 (detektyw AI) nie zadziała (instalacja: komórka 0.1)")
except KeyboardInterrupt:
    print("⏭️ Przerwano pobieranie detektora AI — runda 6 nie zadziała, reszta tak. "
          "Pobieranie wznowi się przy następnym uruchomieniu silnika.")
except Exception as e:
    print(f"❌ Nie udało się pobrać detektora AI: {e}")
    print(f"   Poproś prowadzącego o folder '{FOLDER_DETEKTORA}' (np. z pendrive'a).")
if os.path.isdir("probki"):
    print(f"✅ próbki deepfake'ów (LIVE 6): folder probki/ (liczba plików: {len(os.listdir('probki'))})")
else:
    print("ℹ️ brak folderu probki/ — potrzebny w LIVE 6 (filmy od prowadzącego)")

# ▶ 0.3 SILNIK, część 1: kamera, modele, nagrywanie. Uruchom — nie musisz tego czytać (ale możesz!).
import os, time, math, glob, difflib, unicodedata
import numpy as np
import cv2
import mediapipe as mp
from mediapipe.tasks.python import BaseOptions, vision

SZER, WYS = 854, 480      # rozmiar obrazu: 16:9, jak w telewizji
KAMERA = 0                # zmień na 1, jeśli laptop ma dwie kamery i włącza się nie ta
MAX_FPS = 20              # limit klatek na sekundę: mniej = mniej pracy dla procesora (0 = bez limitu)
FOLDER_NAGRAN = "nagrania"
MAX_NAGRANIE_S = 15
KAMERA_W_PRZEGLADARCE = "auto"   # "auto" = obraz na tej stronie, kamera z przeglądarki albo z Pythona (np. PyCharm) · False = osobne okna OpenCV · True = zawsze kamera z przeglądarki

_ostatni_ts = 0

def znacznik_ms():
    global _ostatni_ts
    _ostatni_ts = max(int(time.monotonic() * 1000), _ostatni_ts + 1)
    return _ostatni_ts

def _do_mp(klatka_bgr):
    rgb = cv2.cvtColor(klatka_bgr, cv2.COLOR_BGR2RGB)
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

_modele = {}

def _sprawdz_plik(sciezka):
    if not os.path.exists(sciezka):
        raise FileNotFoundError(f"Brak pliku modelu: {sciezka}. Uruchom komórkę 0.2 albo skopiuj folder z modelami.")
    return sciezka

def _segmenter():
    if "segmentacja" not in _modele:
        opcje = vision.ImageSegmenterOptions(
            base_options=BaseOptions(model_asset_path=_sprawdz_plik(SCIEZKA_SEGMENTACJA)),
            running_mode=vision.RunningMode.VIDEO,
            output_confidence_masks=True, output_category_mask=False)
        _modele["segmentacja"] = vision.ImageSegmenter.create_from_options(opcje)
    return _modele["segmentacja"]

def _landmarker():
    if "twarz" not in _modele:
        opcje = vision.FaceLandmarkerOptions(
            base_options=BaseOptions(model_asset_path=_sprawdz_plik(SCIEZKA_TWARZ)),
            running_mode=vision.RunningMode.VIDEO,
            num_faces=1, output_face_blendshapes=True)
        _modele["twarz"] = vision.FaceLandmarker.create_from_options(opcje)
    return _modele["twarz"]

def zamknij_modele():
    for model in _modele.values():
        if hasattr(model, "close"):
            model.close()
    _modele.clear()

KATEGORIE = ["tlo", "wlosy", "skora_ciala", "skora_twarzy", "ubranie", "inne"]

def segmentuj(klatka):
    """Dla KAŻDEGO piksela model zwraca pewność (0..1), że to tło, włosy, skóra, ubranie…"""
    wynik = _segmenter().segment_for_video(_do_mp(klatka), znacznik_ms())
    h, w = klatka.shape[:2]
    maski = {}
    for nazwa, maska in zip(KATEGORIE, wynik.confidence_masks):
        m = np.squeeze(np.array(maska.numpy_view(), dtype=np.float32))
        if m.shape != (h, w):
            m = cv2.resize(m, (w, h))
        maski[nazwa] = m
    maski["osoba"] = 1.0 - maski["tlo"]
    return maski

class Mina(dict):
    """Słownik: nazwa miny -> liczba 0..1. Podpowiada, gdy pomylisz nazwę."""
    def __missing__(self, klucz):
        podobne = difflib.get_close_matches(str(klucz), list(self.keys()), n=3)
        raise KeyError(f"Nie ma miny '{klucz}'. Może chodziło o: {podobne}?")

OSTATNIE_PUNKTY_3D = None    # te same punkty z głębokością (x, y, z w pikselach) – do obracania kapibary 3D

def wykryj_twarz(klatka):
    """Zwraca (punkty, mina):"""
    global OSTATNIE_PUNKTY_3D
    wynik = _landmarker().detect_for_video(_do_mp(klatka), znacznik_ms())
    if not wynik.face_landmarks:
        OSTATNIE_PUNKTY_3D = None
        return None, Mina()
    h, w = klatka.shape[:2]
    OSTATNIE_PUNKTY_3D = np.array([(p.x * w, p.y * h, p.z * w) for p in wynik.face_landmarks[0]], dtype=np.float32)
    punkty = OSTATNIE_PUNKTY_3D[:, :2].copy()
    mina = Mina()
    if wynik.face_blendshapes:
        mina.update({k.category_name: float(k.score) for k in wynik.face_blendshapes[0]})
    return punkty, mina

ZNAK_KLUCZ = 2026          # „klucz” studia Kapibara TV: z niego powstaje wzór znaku
ZNAK_BLOK = 16             # znak to szachownica losowych bloków 16 × 16 px…
ZNAK_SILA = 2.5            # …jaśniejszych albo ciemniejszych o 2–3 poziomy jasności (na 255), z rozmytymi brzegami
_wzory_znaku = {}

def _wzor_znaku(h, w, klucz=ZNAK_KLUCZ):
    k = (h, w, klucz)
    if k not in _wzory_znaku:
        znaki = np.random.default_rng(klucz).choice([-1.0, 1.0], size=(h // ZNAK_BLOK + 1, w // ZNAK_BLOK + 1))
        bloki = np.kron(znaki, np.ones((ZNAK_BLOK, ZNAK_BLOK), np.float32))[:h, :w].astype(np.float32)
        _wzory_znaku[k] = (znaki.astype(np.float32), cv2.GaussianBlur(bloki, (0, 0), 1.5))
    return _wzory_znaku[k]

def wstaw_znak_odporny(obraz, klucz=ZNAK_KLUCZ, sila=ZNAK_SILA):
    """Dodaje do obrazu prawie niewidoczny wzór bloków. Przetrwa kompresję wideo i zmniejszenie obrazu."""
    _, wzor = _wzor_znaku(obraz.shape[0], obraz.shape[1], klucz)
    return np.clip(obraz.astype(np.float32) + sila * wzor[..., None], 0, 255).astype(np.uint8)

def wykryj_znak_odporny(obraz, klucz=ZNAK_KLUCZ):
    """Siła znaku: ok. 0 = brak, powyżej 4 = znak jest. Liczymy, jak bardzo jasność bloków „pasuje” do wzoru."""
    if obraz.shape[:2] != (WYS, SZER):
        obraz = cv2.resize(obraz, (SZER, WYS), interpolation=cv2.INTER_AREA)
    szary = cv2.cvtColor(obraz, cv2.COLOR_BGR2GRAY).astype(np.float32)
    nh, nw = WYS // ZNAK_BLOK, SZER // ZNAK_BLOK
    bloki = szary[:nh * ZNAK_BLOK, :nw * ZNAK_BLOK].reshape(nh, ZNAK_BLOK, nw, ZNAK_BLOK).mean(axis=(1, 3))
    znaki, _ = _wzor_znaku(WYS, SZER, klucz)
    znaki = znaki[:nh, :nw]
    r = bloki - cv2.blur(bloki, (3, 3), borderType=cv2.BORDER_REFLECT)
    r = np.clip(r, -1.5 * ZNAK_SILA, 1.5 * ZNAK_SILA)
    p = znaki - cv2.blur(znaki, (3, 3), borderType=cv2.BORDER_REFLECT)
    r, p = r - r.mean(), p - p.mean()
    korelacja = float((r * p).sum() / (np.sqrt((r * r).sum() * (p * p).sum()) + 1e-9))
    return korelacja * math.sqrt(r.size)

def _otworz_kamere():
    """Kamera przez OpenCV (osobne okna): próbuje kilku sterowników i numerów kamer."""
    if os.name == "nt":
        sposoby = [cv2.CAP_DSHOW, cv2.CAP_MSMF, cv2.CAP_ANY]
    elif sys.platform == "darwin":
        sposoby = [cv2.CAP_AVFOUNDATION, cv2.CAP_ANY]
    else:
        sposoby = [cv2.CAP_V4L2, cv2.CAP_ANY]
    for numer in [KAMERA] + [n for n in range(3) if n != KAMERA]:
        for sposob in sposoby:
            kamera = cv2.VideoCapture(numer, sposob)
            if kamera.isOpened():
                kamera.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
                kamera.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
                for _ in range(20):                       # niektóre kamery potrzebują chwili na start
                    ok, _ = kamera.read()
                    if ok:
                        if numer != KAMERA:
                            print(f"ℹ️ Kamera działa pod numerem {numer} — możesz ustawić KAMERA = {numer}.")
                        return kamera
                    time.sleep(0.05)
            kamera.release()
    return None

def dopasuj_kadr(obraz):
    """Przycina obraz do proporcji 16:9 (zostawiając więcej góry, czyli głowy) i skaluje do SZER × WYS."""
    h, w = obraz.shape[:2]
    cel = SZER / WYS
    if w / h > cel:
        nowa_w = int(h * cel)
        x0 = (w - nowa_w) // 2
        obraz = obraz[:, x0:x0 + nowa_w]
    else:
        nowa_h = int(w / cel)
        y0 = int((h - nowa_h) * 0.35)
        obraz = obraz[y0:y0 + nowa_h]
    return cv2.resize(obraz, (SZER, WYS))

def _nowe_nagranie(nazwa, fps):
    os.makedirs(FOLDER_NAGRAN, exist_ok=True)
    baza = os.path.join(FOLDER_NAGRAN, f"{nazwa}_{time.strftime('%H%M%S')}")
    klatek_na_s = int(round(min(max(fps, 5), 30)))
    for rozszerzenie, kodek in ((".mp4", "mp4v"), (".avi", "MJPG")):
        pisarz = cv2.VideoWriter(baza + rozszerzenie, cv2.VideoWriter_fourcc(*kodek), klatek_na_s, (SZER, WYS))
        if pisarz.isOpened():
            print(f"🔴 Nagrywam ({klatek_na_s} kl./s)… naciśnij r, żeby zakończyć (max {MAX_NAGRANIE_S} s).")
            return pisarz, baza + rozszerzenie
        pisarz.release()
    raise RuntimeError("Nie mogę utworzyć pliku wideo.")

def _pasek_info(fps, pomoc, nagrywanie, czas_nagrania):
    pasek = np.full((30, SZER, 3), 32, np.uint8)
    x = 10
    if czas_nagrania is not None:
        if int(time.time() * 2) % 2 == 0:
            cv2.circle(pasek, (x + 7, 15), 7, (0, 0, 255), -1, cv2.LINE_AA)
        x += 20
        x += napisz(pasek, f"REC {czas_nagrania:4.1f} / {MAX_NAGRANIE_S} s", x, 6, 15, (90, 90, 255)) + 16
    tekst = "q – koniec" + (" | r – nagrywanie" if nagrywanie else "") + (f" | {pomoc}" if pomoc else "")
    for rozmiar in (15, 14, 13, 12, 11):
        if render_napisu(tekst, rozmiar, (220, 220, 220)).shape[1] <= SZER - x - 85:
            break
    napisz(pasek, tekst, x, 6 + (15 - rozmiar) // 2, rozmiar, (220, 220, 220))
    licznik = render_napisu(f"{fps:.0f} FPS", 15, (150, 230, 150))
    wklej(pasek, licznik, SZER - licznik.shape[1] - 10, 6)
    return pasek

# ---- obraz z kamery na stronie (Jupyter w przeglądarce) ----
# Strona (JavaScript) bierze obraz z kamery przeglądarki i wysyła go do Pythona przez lokalny adres 127.0.0.1,
# a Python odsyła przerobiony obraz. Nic nie wychodzi poza ten komputer.
import sys, collections, threading, secrets, html, http.server, urllib.parse

_RADY_KAMERA = ("\n   1) Zezwól tej stronie na kamerę: ikona kamery albo kłódki w pasku adresu → Zezwalaj, potem uruchom komórkę jeszcze raz."
                "\n   2) Zamknij inne programy i karty, które używają kamery (Teams, Zoom, Meet)."
                "\n   3) Albo osobne okna OpenCV: KAMERA_W_PRZEGLADARCE = False i uruchom komórkę jeszcze raz.")

_stary = globals().get("_PRZEGLADARKA", {}).get("serwer")
if _stary is not None:                                    # silnik uruchomiony ponownie: stary serwer kończy pracę
    threading.Thread(target=_stary.shutdown, daemon=True).start()
def _ide_bez_kamery_na_stronie():
    """PyCharm i DataSpell pokazują wyniki komórek we wbudowanej przeglądarce, która nie daje stronie dostępu do kamery."""
    return any(k.startswith(("PYCHARM", "JETBRAINS", "DATASPELL")) for k in os.environ)

# kamera_pythona = True: obraz z kamery bierze Python (OpenCV), a wynik dalej jest na stronie pod komórką
_PRZEGLADARKA = {"serwer": None, "bez_kontaktu": False, "kamera_pythona": _ide_bez_kamery_na_stronie()}
_SESJE = {}

class _Sesja:
    """Wspólny stan strony i Pythona dla jednego okna z obrazem."""
    def __init__(self, kamera):
        self.warunek = threading.Condition()
        self.kamera = kamera
        self.klatka, self.nr_klatki = None, 0             # ostatnia klatka z kamery przeglądarki (JPEG)
        self.obraz, self.nr_obrazu = None, 0              # ostatni obraz z Pythona (JPEG)
        self.klawisze = collections.deque()
        self.blad, self.koniec, self.kontakt = "", False, None

class _Obsluga(http.server.BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def log_message(self, *argumenty):
        pass

    def _naglowki(self, kod, dlugosc=0, typ="text/plain", dodatkowe=()):
        self.send_response(kod)
        for klucz, wartosc in (("Access-Control-Allow-Origin", "*"), ("Access-Control-Allow-Methods", "GET, POST, OPTIONS"),
                               ("Access-Control-Allow-Headers", "Content-Type"), ("Access-Control-Allow-Private-Network", "true"),
                               ("Access-Control-Expose-Headers", "X-Nr, X-Koniec"), ("Access-Control-Max-Age", "600"),
                               ("Cache-Control", "no-store"), ("Content-Type", typ), ("Content-Length", str(dlugosc))):
            self.send_header(klucz, wartosc)
        for klucz, wartosc in dodatkowe:
            self.send_header(klucz, wartosc)
        self.end_headers()

    def _rozbierz(self):
        adres = urllib.parse.urlsplit(self.path)
        czesci = adres.path.strip("/").split("/")
        dlugosc = int(self.headers.get("Content-Length") or 0)
        cialo = self.rfile.read(dlugosc) if dlugosc > 0 else b""
        if len(czesci) != 3 or czesci[0] != "s":
            return None, "", {}, cialo
        return _SESJE.get(czesci[1]), czesci[2], urllib.parse.parse_qs(adres.query), cialo

    def _wyslij_obraz(self, s, ma):
        with s.warunek:
            s.warunek.wait_for(lambda: s.koniec or s.nr_obrazu > ma, timeout=1.0)
            koniec, nr, obraz = s.koniec, s.nr_obrazu, s.obraz
        if koniec:
            self._naglowki(200, 0, dodatkowe=[("X-Koniec", "1")])
        elif obraz is None or nr <= ma:
            self._naglowki(204, 0, dodatkowe=[("X-Nr", str(nr))])
        else:
            self._naglowki(200, len(obraz), "image/jpeg", [("X-Nr", str(nr))])
            self.wfile.write(obraz)

    def do_OPTIONS(self):
        self._naglowki(204)

    def _obsluz(self):
        s, akcja, pytanie, cialo = self._rozbierz()
        if s is None:
            self._naglowki(410, 0, dodatkowe=[("X-Koniec", "1")])
            return
        with s.warunek:
            s.kontakt = time.time()
            s.warunek.notify_all()
        ma = int((pytanie.get("ma") or ["0"])[0] or 0)
        if akcja == "klatka":
            with s.warunek:
                s.klatka, s.nr_klatki = cialo, s.nr_klatki + 1
                s.warunek.notify_all()
            self._wyslij_obraz(s, ma)
        elif akcja == "obraz":
            self._wyslij_obraz(s, ma)
        elif akcja == "klawisz":
            znak = (pytanie.get("k") or [""])[0][:1]
            with s.warunek:
                if znak:
                    s.klawisze.append(ord(znak))
                s.warunek.notify_all()
            self._naglowki(204)
        elif akcja == "blad":
            with s.warunek:
                s.blad = cialo.decode("utf-8", "replace")[:300] or "nieznany błąd"
                s.warunek.notify_all()
            self._naglowki(204)
        else:
            self._naglowki(204)

    do_GET = do_POST = _obsluz

def _serwer():
    if _PRZEGLADARKA["serwer"] is None:
        serwer = http.server.ThreadingHTTPServer(("127.0.0.1", 0), _Obsluga)
        serwer.daemon_threads = True
        threading.Thread(target=serwer.serve_forever, daemon=True).start()
        _PRZEGLADARKA["serwer"] = serwer
    return _PRZEGLADARKA["serwer"].server_address[1]

def _tryb_przegladarki():
    """Czy pokazywać obraz na stronie Jupytera (True) czy w osobnych oknach OpenCV (False)."""
    if KAMERA_W_PRZEGLADARCE is True:
        return True
    if KAMERA_W_PRZEGLADARCE is False or _PRZEGLADARKA["bez_kontaktu"]:
        return False
    try:
        from IPython import get_ipython
        return getattr(get_ipython(), "kernel", None) is not None
    except Exception:
        return False

def _przyciski(pomoc, nagrywanie=False):
    """Z tekstu pomocy („k – kalibracja | +/- – próg”) robi przyciski pod obrazem."""
    wynik = []
    for czesc in pomoc.split("|"):
        czesc = czesc.strip()
        if not czesc:
            continue
        klawisze, _, opis = czesc.partition(" – ") if " – " in czesc else czesc.partition(" ")
        klawisze = klawisze.strip()
        lista = [" "] if klawisze == "spacja" else [k for k in klawisze.split("/") if k]
        for k in lista:
            if len(k) == 1:
                wynik.append((k, f"{'␣' if k == ' ' else k} · {opis.strip()}"))
    if nagrywanie:
        wynik.append(("r", "r · nagrywanie"))
    wynik.append(("q", "■ Stop (q)"))
    return wynik

_EKRAN_HTML = """<div class="fns-ekran" id="fns-__ID__" tabindex="0" data-lm-suppress-shortcuts="true" data-p-suppress-shortcuts="true">
<div class="fns-tytul">__TYTUL__ · kliknij obraz, żeby sterować klawiaturą</div>
<img class="fns-obraz" alt="">
<div class="fns-info">Łączę się z Pythonem…</div>
<div class="fns-przyciski">__PRZYCISKI__</div>
</div>
<style>
.fns-ekran{display:inline-flex;flex-direction:column;gap:6px;max-width:100%;padding:10px;border-radius:12px;background:#0c110d;color:#eef3ec;border:2px solid #243127;font-family:system-ui,sans-serif;outline:none}
.fns-ekran:focus{border-color:#38e83a}
.fns-tytul{font-size:13px;color:#a9b8ac}
.fns-obraz{display:block;width:854px;max-width:100%;border-radius:8px;background:#151d17}
.fns-obraz:not([src]){aspect-ratio:854/480}
.fns-info{font-size:14px;color:#f2c14e;white-space:pre-wrap}
.fns-info:empty{display:none}
.fns-przyciski{display:flex;flex-wrap:wrap;gap:6px}
.fns-przyciski button{font:inherit;font-size:13px;padding:4px 10px;border-radius:6px;border:1px solid #38e83a;background:#151d17;color:#eef3ec;cursor:pointer}
.fns-przyciski button:hover{background:#1f4a24}
.fns-przyciski button[data-k="q"]{border-color:#e5484d}
.fns-zamkniete .fns-obraz{display:none}
</style>
<script>
(() => {
  const ADRES = "http://127.0.0.1:__PORT__/s/__ID__";
  const KAMERA = __KAMERA__;
  const root = document.getElementById("fns-__ID__");
  if (!root || root.dataset.start) return;
  root.dataset.start = "1";
  const obraz = root.querySelector(".fns-obraz"), info = root.querySelector(".fns-info"), przyciski = root.querySelector(".fns-przyciski");
  const video = document.createElement("video"), plotno = document.createElement("canvas");
  video.muted = true; video.playsInline = true;
  let zywy = true, nr = 0, url = null, strumien = null, bledy = 0;
  const czekaj = (ms) => new Promise((r) => setTimeout(r, ms));
  const wyslij = (akcja, cialo) => fetch(`${ADRES}/${akcja}`, { method: "POST", body: cialo, headers: { "Content-Type": "text/plain" } });
  const zatrzymajKamere = () => { if (strumien) { strumien.getTracks().forEach((t) => t.stop()); strumien = null; } video.srcObject = null; };
  const koniec = (tekst) => {
    if (!zywy) return;
    zywy = false; zatrzymajKamere();
    obraz.removeAttribute("src"); if (url) { URL.revokeObjectURL(url); url = null; }
    przyciski.replaceChildren(); root.classList.add("fns-zamkniete");
    info.textContent = tekst || "Zamknięte — uruchom komórkę jeszcze raz.";
  };
  const klawisz = (k) => { if (zywy) wyslij("klawisz?k=" + encodeURIComponent(k)).catch(() => {}); };
  root.addEventListener("keydown", (e) => {
    if (!zywy || e.ctrlKey || e.metaKey || e.altKey) return;
    const k = e.key === "Escape" ? "\\u001b" : e.key;
    if (k.length !== 1) return;
    e.preventDefault(); e.stopPropagation();
    klawisz(k.toLowerCase());
  });
  przyciski.querySelectorAll("button").forEach((b) => b.addEventListener("click", (e) => {
    e.preventDefault(); e.stopPropagation(); klawisz(b.dataset.k); root.focus({ preventScroll: true });
  }));
  obraz.addEventListener("click", () => root.focus({ preventScroll: true }));
  const odbierz = async (odp) => {
    bledy = 0;
    if (odp.headers.get("X-Koniec")) { koniec(); return; }
    const n = parseInt(odp.headers.get("X-Nr") || "0", 10);
    if (odp.status === 200 && n > nr) {
      const blob = await odp.blob();
      const nowy = URL.createObjectURL(blob);
      obraz.src = nowy; if (url) URL.revokeObjectURL(url); url = nowy; nr = n;
      if (info.textContent.startsWith("Łączę") || info.textContent.startsWith("📷")) info.textContent = "";
    }
  };
  const bladSieci = async () => {
    bledy += 1;
    if (bledy >= 6) { koniec("❌ Nie mogę połączyć się z Pythonem (127.0.0.1:__PORT__). Uruchom komórkę jeszcze raz."); return; }
    await czekaj(300);
  };
  const zdjecie = () => new Promise((gotowe) => {
    plotno.width = video.videoWidth; plotno.height = video.videoHeight;
    plotno.getContext("2d").drawImage(video, 0, 0);
    plotno.toBlob(gotowe, "image/jpeg", 0.85);
  });
  const petlaKamery = async () => {
    if (!navigator.mediaDevices || !navigator.mediaDevices.getUserMedia)
      throw new Error("ta strona nie może używać kamery (potrzebny adres https:// albo localhost)");
    info.textContent = "📷 Włączam kamerę… jeśli przeglądarka zapyta, kliknij „Zezwól”.";
    strumien = await navigator.mediaDevices.getUserMedia({ video: { width: { ideal: 640 }, height: { ideal: 480 } }, audio: false });
    if (!zywy) { zatrzymajKamere(); return; }
    video.srcObject = strumien;
    await video.play();
    while (zywy) {
      if (!video.videoWidth) { await czekaj(50); continue; }
      const blob = await zdjecie();
      try { await odbierz(await wyslij(`klatka?ma=${nr}`, blob)); } catch (e) { await bladSieci(); }
    }
  };
  const petlaObrazu = async () => {
    while (zywy) {
      try { await odbierz(await fetch(`${ADRES}/obraz?ma=${nr}`)); } catch (e) { await bladSieci(); }
    }
  };
  wyslij("witaj").then(
    () => (KAMERA ? petlaKamery() : petlaObrazu()),
    () => koniec("❌ Strona nie może połączyć się z Pythonem pod 127.0.0.1:__PORT__ — Python użyje osobnych okien OpenCV."),
  ).catch((e) => {
    const opisy = { NotAllowedError: "brak zgody na kamerę", NotReadableError: "kamera jest zajęta przez inny program",
                    NotFoundError: "nie znaleziono kamery", OverconstrainedError: "kamera nie obsługuje tego rozmiaru obrazu" };
    const tekst = `${opisy[e.name] || e.name || "błąd"} (${e.message || e})`;
    wyslij("blad", tekst).catch(() => {});
    koniec("❌ Kamera: " + tekst);
  });
  setTimeout(() => root.focus({ preventScroll: true }), 100);
})();
</script>"""

class _Ekran:
    """Jedno „okno” na stronie: obraz z Pythona, przyciski i klawisze, a w trybie kamery — obraz z kamery przeglądarki."""
    def __init__(self, tytul, pomoc="", kamera=False, nagrywanie=False):
        from IPython.display import display, HTML
        self.id, self.start, self.s = secrets.token_hex(8), time.time(), _Sesja(kamera)
        _SESJE[self.id] = self.s
        przyciski = "".join(f'<button data-k="{html.escape(k)}">{html.escape(etykieta)}</button>'
                            for k, etykieta in _przyciski(pomoc, nagrywanie))
        kod = (_EKRAN_HTML.replace("__ID__", self.id).replace("__PORT__", str(_serwer()))
               .replace("__KAMERA__", "true" if kamera else "false").replace("__TYTUL__", html.escape(tytul))
               .replace("__PRZYCISKI__", przyciski))
        display(HTML(kod))

    def czekaj_na_strone(self, sekundy):
        with self.s.warunek:
            return self.s.warunek.wait_for(lambda: self.s.kontakt is not None, timeout=sekundy)

    def czekaj_na_kamere(self, sekundy):
        """Czeka na pierwszą klatkę z kamery strony. Zwraca opis błędu kamery albo "" (jest obraz, koniec albo minął czas)."""
        s = self.s
        with s.warunek:
            s.warunek.wait_for(lambda: s.nr_klatki > 0 or s.blad or s.koniec or any(k in (ord("q"), 27) for k in s.klawisze),
                               timeout=sekundy)
            return s.blad or ""

    def pokaz(self, obraz):
        ok, jpg = cv2.imencode(".jpg", obraz, [cv2.IMWRITE_JPEG_QUALITY, 80])
        if ok:
            with self.s.warunek:
                self.s.obraz, self.s.nr_obrazu = jpg.tobytes(), self.s.nr_obrazu + 1
                self.s.warunek.notify_all()

    def klawisz(self, ms):
        with self.s.warunek:
            self.s.warunek.wait_for(lambda: self.s.klawisze or self.s.koniec, timeout=max(ms, 1) / 1000)
            return self.s.klawisze.popleft() if self.s.klawisze else -1

    def otwarte(self):
        ostatni = self.s.kontakt or self.start
        return not self.s.koniec and time.time() - ostatni < (12 if self.s.kontakt else 20)

    def zamknij(self):
        with self.s.warunek:
            self.s.koniec = True
            self.s.warunek.notify_all()
        if self.s.kontakt is None and not _PRZEGLADARKA["bez_kontaktu"] and KAMERA_W_PRZEGLADARCE == "auto":
            _PRZEGLADARKA["bez_kontaktu"] = True
            print("ℹ️ Strona nie połączyła się z Pythonem — dalej używam osobnych okien OpenCV.")
        sprzatanie = threading.Timer(5.0, _SESJE.pop, args=(self.id, None))
        sprzatanie.daemon = True
        sprzatanie.start()

class _KameraPrzegladarki:
    """Kamera przeglądarki widziana z Pythona tak jak cv2.VideoCapture: read() → (ok, klatka)."""
    def __init__(self, ekran):
        self.ekran, self.ostatnia, self.powod = ekran, 0, ""

    def isOpened(self):
        return True

    def release(self):
        pass

    def read(self):
        s = self.ekran.s
        stop = lambda: s.koniec or any(k in (ord("q"), 27) for k in s.klawisze)
        with s.warunek:
            s.warunek.wait_for(lambda: s.nr_klatki > self.ostatnia or s.blad or stop(),
                               timeout=45.0 if self.ostatnia == 0 else 5.0)
            if s.blad:
                self.powod = "❌ Przeglądarka nie dała obrazu z kamery — " + s.blad + _RADY_KAMERA
                return False, None
            if s.nr_klatki <= self.ostatnia:
                if stop():
                    self.powod = ""
                elif self.ostatnia == 0:
                    self.powod = "❌ Strona nie przysłała obrazu z kamery." + _RADY_KAMERA
                else:
                    self.powod = "⚠️ Kamera na stronie przestała wysyłać obraz (zamknięta karta albo wyczyszczony wynik?)."
                return False, None
            self.ostatnia, dane = s.nr_klatki, s.klatka
        klatka = cv2.imdecode(np.frombuffer(dane, np.uint8), cv2.IMREAD_COLOR)
        if klatka is None:
            self.powod = "⚠️ Uszkodzona klatka z przeglądarki."
            return False, None
        return True, klatka

class _Okna:
    """Te same polecenia co w cv2 (namedWindow, imshow, waitKey…), ale obraz trafia na stronę albo do okien OpenCV."""
    def __init__(self):
        self.aktywny = None

    def namedWindow(self, tytul, flagi=None, pomoc="", kamera=False, nagrywanie=False):
        if _tryb_przegladarki():
            self.destroyAllWindows()
            self.aktywny = _Ekran(tytul, pomoc, kamera, nagrywanie)
        else:
            cv2.namedWindow(tytul, cv2.WINDOW_NORMAL if flagi is None else flagi)

    def resizeWindow(self, tytul, szer, wys):
        if self.aktywny is None:
            cv2.resizeWindow(tytul, szer, wys)

    def imshow(self, tytul, obraz):
        if self.aktywny is None and _tryb_przegladarki():
            self.aktywny = _Ekran(tytul)
        if self.aktywny is not None:
            self.aktywny.pokaz(obraz)
        else:
            cv2.imshow(tytul, obraz)

    def waitKey(self, ms=0):
        if self.aktywny is not None:
            return self.aktywny.klawisz(ms)
        if _tryb_przegladarki():
            time.sleep(max(ms, 1) / 1000)
            return -1
        return cv2.waitKey(ms)

    def getWindowProperty(self, tytul, wlasnosc):
        if self.aktywny is not None:
            return 1.0 if self.aktywny.otwarte() else 0.0
        return cv2.getWindowProperty(tytul, wlasnosc)

    def destroyAllWindows(self):
        if self.aktywny is not None:
            self.aktywny.zamknij()
            self.aktywny = None
        elif not _tryb_przegladarki():
            cv2.destroyAllWindows()

okna = _Okna()

def uruchom(przetworz, tytul="Fake News Studio", pomoc="", nagrywanie=False, nazwa="nagranie"):
    """Włącza kamerę i dla każdej klatki woła przetworz(klatka, klawisz) -> obraz do pokazania.
    W Jupyterze obraz jest na tej stronie: kamera z przeglądarki, a gdy strona jej nie dostaje (np. PyCharm) — z Pythona.
    Poza Jupyterem (albo gdy strona nie łączy się z Pythonem, np. VS Code) — osobne okno OpenCV."""
    kamera = None
    kamera_strony = _tryb_przegladarki() and not (KAMERA_W_PRZEGLADARCE == "auto" and _PRZEGLADARKA["kamera_pythona"])
    if kamera_strony:
        okna.namedWindow(tytul, pomoc=pomoc, kamera=True, nagrywanie=nagrywanie)
        if not okna.aktywny.czekaj_na_strone(8):
            okna.destroyAllWindows()                      # strona nie odpowiada (np. VS Code) → okna OpenCV
        else:
            blad = okna.aktywny.czekaj_na_kamere(45)
            if blad and "zajęta" not in blad and KAMERA_W_PRZEGLADARCE == "auto":
                _PRZEGLADARKA["kamera_pythona"] = True       # np. PyCharm: strona nie dostaje kamery
                print(f"ℹ️ Strona nie dostała kamery ({blad}). Obraz z kamery bierze teraz Python (OpenCV), "
                      "a wynik dalej jest tutaj — tak już do końca pracy z notebookiem.")
                okna.destroyAllWindows()
            else:
                kamera = _KameraPrzegladarki(okna.aktywny)
    if kamera is None:
        kamera = _otworz_kamere()
        if kamera is None:
            print("❌ Nie mogę włączyć kamery przez OpenCV. Sprawdź, czy inna aplikacja (Teams, Zoom, inny notebook) jej nie używa "
                  "i czy Windows pozwala aplikacjom klasycznym na dostęp do kamery (Ustawienia → Prywatność → Kamera).")
            return None
        if _tryb_przegladarki():                          # kamera z Pythona, obraz na stronie (np. PyCharm)
            okna.namedWindow(tytul, pomoc=pomoc, nagrywanie=nagrywanie)
        else:
            okna.namedWindow(tytul, cv2.WINDOW_NORMAL)
            okna.resizeWindow(tytul, SZER, WYS + 30)
            try:
                cv2.setWindowProperty(tytul, cv2.WND_PROP_TOPMOST, 1)
            except cv2.error:
                pass
    klawisz, fps, t_poprz = -1, 15.0, time.perf_counter()
    odstep = 1.0 / MAX_FPS if MAX_FPS else 0.0
    stan = {"pisarz": None, "sciezka": None, "start": 0.0, "zapisane": None}

    def zakoncz_nagranie():
        if stan["pisarz"] is not None:
            stan["pisarz"].release()
            stan["pisarz"] = None
            stan["zapisane"] = stan["sciezka"]
            print(f"💾 Zapisano nagranie: {stan['sciezka']}")

    try:
        while True:
            t_klatki = time.perf_counter()
            ok, klatka = kamera.read()
            if not ok:
                powod = getattr(kamera, "powod", "⚠️ Kamera przestała wysyłać obraz.")
                if powod:
                    print(powod)
                break
            klatka = cv2.flip(dopasuj_kadr(klatka), 1)
            obraz = przetworz(klatka, klawisz)
            if obraz is None:
                obraz = klatka
            if obraz.shape[:2] != (WYS, SZER):
                obraz = cv2.resize(obraz, (SZER, WYS))
            teraz = time.perf_counter()
            fps = 0.9 * fps + 0.1 / max(teraz - t_poprz, 1e-3)
            t_poprz = teraz
            if stan["pisarz"] is not None:
                stan["pisarz"].write(wstaw_znak_odporny(obraz))
                if teraz - stan["start"] >= MAX_NAGRANIE_S:
                    zakoncz_nagranie()
            czas = (teraz - stan["start"]) if stan["pisarz"] is not None else None
            okna.imshow(tytul, np.vstack([obraz, _pasek_info(fps, pomoc, nagrywanie, czas)]))
            klawisz = okna.waitKey(max(1, int((t_klatki + odstep - time.perf_counter()) * 1000)))
            klawisz = -1 if klawisz < 0 else klawisz & 0xFF
            if klawisz in (ord("q"), 27):
                break
            if nagrywanie and klawisz == ord("r"):
                if stan["pisarz"] is None:
                    stan["pisarz"], stan["sciezka"] = _nowe_nagranie(nazwa, fps)
                    stan["start"] = time.perf_counter()
                else:
                    zakoncz_nagranie()
            try:
                if okna.getWindowProperty(tytul, cv2.WND_PROP_VISIBLE) < 1:
                    break
            except cv2.error:
                break
    finally:
        zakoncz_nagranie()
        kamera.release()
        okna.destroyAllWindows()
        for _ in range(5):
            okna.waitKey(1)
    return stan["zapisane"]

def odtworz(sciezka=None):
    """Odtwarza nagranie w pętli (q – koniec). Bez argumentu: najnowsze nagranie."""
    if sciezka is None:
        pliki = sorted(glob.glob(os.path.join(FOLDER_NAGRAN, "*.*")), key=os.path.getmtime)
        if not pliki:
            print("Brak nagrań w folderze", FOLDER_NAGRAN)
            return
        sciezka = pliki[-1]
    wideo = cv2.VideoCapture(sciezka)
    ok, klatka = wideo.read()
    if not ok:
        print("❌ Nie mogę odczytać pliku", sciezka)
        return
    opoznienie = max(int(1000 / (wideo.get(cv2.CAP_PROP_FPS) or 15)), 1)
    print(f"▶ {sciezka} — q kończy odtwarzanie")
    okna.namedWindow("Odtwarzanie", cv2.WINDOW_NORMAL)
    try:
        while True:
            okna.imshow("Odtwarzanie", klatka)
            if (okna.waitKey(opoznienie) & 0xFF) in (ord("q"), 27):
                break
            try:
                if okna.getWindowProperty("Odtwarzanie", cv2.WND_PROP_VISIBLE) < 1:
                    break
            except cv2.error:
                break
            ok, klatka = wideo.read()
            if not ok:
                wideo.set(cv2.CAP_PROP_POS_FRAMES, 0)
                ok, klatka = wideo.read()
                if not ok:
                    break
    finally:
        wideo.release()
        okna.destroyAllWindows()
        for _ in range(5):
            okna.waitKey(1)

print("✅ Silnik, część 1 gotowa")

# ▶ 0.4 SILNIK, część 2: napisy, tła, maski, green screen, rysowanie. Uruchom — nie musisz tego czytać.
from PIL import Image, ImageDraw, ImageFont

BIALY, CZARNY = (255, 255, 255), (15, 15, 15)
CZERWONY, ZOLTY = (45, 35, 205), (40, 210, 250)

def _znajdz_czcionke():
    kandydaci = []
    try:
        import matplotlib
        kandydaci.append(os.path.join(matplotlib.get_data_path(), "fonts", "ttf", "DejaVuSans-Bold.ttf"))
    except ImportError:
        pass
    kandydaci += ["C:/Windows/Fonts/arialbd.ttf", "/System/Library/Fonts/Supplemental/Arial Bold.ttf",
                  "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"]
    return next((k for k in kandydaci if os.path.exists(k)), None)

CZCIONKA = _znajdz_czcionke()
_fonty, _napisy = {}, {}

def _bez_ogonkow(tekst):
    tekst = tekst.replace("ł", "l").replace("Ł", "L")
    return unicodedata.normalize("NFKD", tekst).encode("ascii", "ignore").decode()

def render_napisu(tekst, rozmiar=20, kolor=BIALY, obrys=0, kolor_obrysu=CZARNY):
    """Zwraca obrazek BGRA z napisem na przezroczystym tle (gotowe napisy są zapamiętywane)."""
    kolor, kolor_obrysu = tuple(int(c) for c in kolor), tuple(int(c) for c in kolor_obrysu)
    klucz = (tekst, rozmiar, kolor, obrys, kolor_obrysu)
    if klucz not in _napisy:
        if len(_napisy) > 400:
            _napisy.clear()
        if CZCIONKA:
            if rozmiar not in _fonty:
                _fonty[rozmiar] = ImageFont.truetype(CZCIONKA, rozmiar)
            f = _fonty[rozmiar]
            wznios, zejscie = f.getmetrics()
            w = int(math.ceil(f.getlength(tekst))) + 2 * obrys + 2
            h = wznios + zejscie + 2 * obrys
            img = Image.new("RGBA", (max(w, 1), h), (0, 0, 0, 0))
            ImageDraw.Draw(img).text((obrys, obrys), tekst, font=f, fill=kolor[::-1] + (255,),
                                     stroke_width=obrys, stroke_fill=kolor_obrysu[::-1] + (255,))
            _napisy[klucz] = np.array(img)[:, :, [2, 1, 0, 3]].copy()
        else:
            tekst_a, skala = _bez_ogonkow(tekst), rozmiar / 30
            (w, h), linia = cv2.getTextSize(tekst_a, cv2.FONT_HERSHEY_SIMPLEX, skala, 2)
            img = np.zeros((h + linia + 4, w + 4, 4), np.uint8)
            cv2.putText(img, tekst_a, (2, h + 2), cv2.FONT_HERSHEY_SIMPLEX, skala, kolor + (255,), 2, cv2.LINE_AA)
            _napisy[klucz] = img
    return _napisy[klucz]

def wklej(tlo, nakladka, x, y, krycie=1.0):
    """Nakłada obrazek BGRA na obraz BGR w punkcie (x, y) — z przezroczystością i przycinaniem do krawędzi."""
    x, y = int(x), int(y)
    h, w = nakladka.shape[:2]
    H, W = tlo.shape[:2]
    x0, y0, x1, y1 = max(x, 0), max(y, 0), min(x + w, W), min(y + h, H)
    if x0 >= x1 or y0 >= y1:
        return tlo
    kawalek = nakladka[y0 - y:y1 - y, x0 - x:x1 - x]
    a = kawalek[:, :, 3:4].astype(np.float32) * (krycie / 255.0)
    obszar = tlo[y0:y1, x0:x1]
    obszar[:] = (kawalek[:, :, :3] * a + obszar * (1 - a)).astype(np.uint8)
    return tlo

def napisz(obraz, tekst, x, y, rozmiar=20, kolor=BIALY, obrys=0, kolor_obrysu=CZARNY):
    """Pisze tekst (z polskimi znakami); (x, y) to lewy górny róg. Zwraca szerokość napisu."""
    napis = render_napisu(tekst, rozmiar, kolor, obrys, kolor_obrysu)
    wklej(obraz, napis, x, y)
    return napis.shape[1]

def prostokat(obraz, x0, y0, x1, y1, kolor, krycie=1.0):
    x0, y0 = max(int(x0), 0), max(int(y0), 0)
    x1, y1 = min(int(x1), obraz.shape[1]), min(int(y1), obraz.shape[0])
    if x0 >= x1 or y0 >= y1:
        return
    obszar = obraz[y0:y1, x0:x1]
    obszar[:] = (obszar * (1 - krycie) + np.array(kolor, np.float32) * krycie).astype(np.uint8)

def kierunek(stopnie):
    """Wektor długości 1 w danym kierunku (0° = w prawo, 90° = w dół, bo w obrazie oś y rośnie w dół)."""
    r = math.radians(stopnie)
    return np.array([math.cos(r), math.sin(r)], np.float32)

def punkt_lok(srodek, kat, skala, x, y):
    """Zamienia współrzędne „lokalne” (w jednostkach skali, obrócone o kąt) na piksele obrazu."""
    c, s = math.cos(math.radians(kat)), math.sin(math.radians(kat))
    return np.array([srodek[0] + (x * c - y * s) * skala, srodek[1] + (x * s + y * c) * skala], np.float32)

def elipsa(obraz, srodek, osie, kat, kolor, grubosc=-1):
    cv2.ellipse(obraz, (int(round(srodek[0])), int(round(srodek[1]))),
                (max(1, int(round(osie[0]))), max(1, int(round(osie[1])))),
                float(kat), 0, 360, tuple(int(c) for c in kolor), grubosc, cv2.LINE_AA)

def geometria_twarzy(punkty):
    """Środek (czubek nosa), przechylenie głowy w stopniach i skala (szerokość twarzy w pikselach)."""
    oczy = punkty[263] - punkty[33]
    if oczy[0] < 0:
        oczy = -oczy
    kat = math.degrees(math.atan2(oczy[1], oczy[0]))
    skala = float(np.linalg.norm(punkty[454] - punkty[234]))
    return punkty[1], kat, skala

def zrob_tlo_studia():
    yy, xx = np.mgrid[0:WYS, 0:SZER].astype(np.float32)
    t = yy / WYS
    tlo = np.stack([75 + 55 * t, 28 + 22 * t, 12 + 10 * t], axis=-1)
    d = np.sqrt(((xx - SZER / 2) / (SZER * 0.42)) ** 2 + ((yy - WYS * 0.42) / (WYS * 0.65)) ** 2)
    tlo += (np.clip(1 - d, 0, 1) ** 1.5)[..., None] * np.array([150, 90, 30], np.float32)
    tlo = np.clip(tlo, 0, 255).astype(np.uint8)
    for x0 in (34, SZER - 204):
        prostokat(tlo, x0, 96, x0 + 170, 250, (140, 70, 25), 0.55)
        cv2.rectangle(tlo, (x0, 96), (x0 + 170, 250), (230, 160, 70), 2, cv2.LINE_AA)
        for i in range(5):
            cv2.line(tlo, (x0 + 16, 122 + 25 * i), (x0 + 154 - 34 * (i % 2), 122 + 25 * i), (200, 140, 60), 3)
    for i in range(6):
        y = int(WYS * 0.74 + i * i * 4)
        cv2.line(tlo, (0, y), (SZER, y), (120, 66, 28), 1, cv2.LINE_AA)
    return tlo

def wczytaj_tlo(plik=None):
    """Wczytuje obrazek tła (dowolny rozmiar) albo tworzy domyślne studio."""
    if plik:
        if os.path.exists(plik):
            img = cv2.imdecode(np.fromfile(plik, np.uint8), cv2.IMREAD_COLOR)
            if img is not None:
                return dopasuj_kadr(img)
        print(f"⚠️ Nie mogę wczytać obrazka '{plik}' — używam tła studia.")
    return zrob_tlo_studia()

def alfa_z_maski(osoba, prog=0.5, miekkosc=0.15):
    """Zamienia pewność „to osoba” (0..1) na przezroczystość: poniżej progu — tło, powyżej — osoba."""
    if miekkosc <= 0:
        return (osoba > prog).astype(np.float32)
    a = np.clip((osoba - (prog - miekkosc)) / (2 * miekkosc), 0, 1)
    return a * a * (3 - 2 * a)

def zloz(przod, tlo, alfa):
    a = alfa[..., None]
    return (przod.astype(np.float32) * a + tlo.astype(np.float32) * (1 - a)).astype(np.uint8)

def pokaz_maske(maska, opis="żółty = osoba · fioletowy = tło · inne kolory = model nie jest pewien"):
    obraz = cv2.applyColorMap((np.clip(maska, 0, 1) * 255).astype(np.uint8), cv2.COLORMAP_VIRIDIS)
    napisz(obraz, opis, 12, WYS - 34, 16, obrys=2)
    return obraz

def przefarbuj(klatka, maska, kolor, sila=0.65):
    szary = cv2.cvtColor(klatka, cv2.COLOR_BGR2GRAY).astype(np.float32)[..., None] / 255
    farba = np.clip((0.35 + 0.9 * szary) * np.array(kolor, np.float32), 0, 255)
    m = (np.clip(maska, 0, 1) * sila)[..., None]
    return (klatka * (1 - m) + farba * m).astype(np.uint8)

ZIELEN = {"prog": None, "miekkosc": 12.0, "auto_do": 0.0}    # próg ustawia kalibracja (klawisz k) albo Wy (+/-)

def zielonosc(klatka):
    """O ile kanał zielony jest jaśniejszy od czerwonego i niebieskiego (0 = piksel wcale nie jest zielony)."""
    b, g, r = cv2.split(cv2.GaussianBlur(klatka, (0, 0), 1.0).astype(np.int16))
    return np.clip(g - np.maximum(r, b), 0, 255).astype(np.float32)

def kalibruj_zielen(klatka, z=None):
    """Mierzy, jak zielony jest materiał w kadrze, i dobiera próg. Zwraca False, gdy nie widać zielonego tła."""
    z = zielonosc(klatka) if z is None else z
    material = z[z > 25]
    if material.size < 0.05 * z.size:
        return False
    typowa = float(np.percentile(material, 25))
    ZIELEN["prog"] = round(max(12.0, 0.5 * typowa))
    ZIELEN["miekkosc"] = max(6.0, 0.2 * typowa)
    return True

def klucz_koloru(klatka, z=None):
    """Przezroczystość osoby z koloru: zieloność powyżej progu → tło (0), poniżej → osoba (1), pomiędzy → półprzezroczyste."""
    z = zielonosc(klatka) if z is None else z
    if ZIELEN["prog"] is None:
        ZIELEN["auto_do"] = time.time() + 2.0
    if time.time() < ZIELEN["auto_do"]:
        kalibruj_zielen(klatka, z)
    p, m = float(ZIELEN["prog"] or 40), float(ZIELEN["miekkosc"])
    a = np.clip((z - (p - m)) / (2 * m), 0, 1)
    return 1 - a * a * (3 - 2 * a)

def usun_poswiate(klatka):
    """Zielone światło odbite od materiału barwi włosy i brzegi ubrania. Zielony kanał przycinamy"""
    b, g, r = cv2.split(klatka)
    return cv2.merge([b, np.minimum(g, np.maximum(r, b)), r])

def wytnij_osobe(klatka, metoda="oba", poswiata=True):
    """Zwraca (klatka, alfa) – osobę wyciętą z tła. metoda:"""
    if metoda != "ai":
        z = zielonosc(klatka)
        if metoda == "zielony" or (z > 25).mean() > 0.03:
            alfa = klucz_koloru(klatka, z)
            if metoda == "oba":
                osoba = (segmentuj(klatka)["osoba"] > 0.5).astype(np.uint8)
                osoba = cv2.dilate(osoba, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (41, 41)))
                alfa = alfa * cv2.GaussianBlur(osoba.astype(np.float32), (0, 0), 6)
            return (usun_poswiate(klatka) if poswiata else klatka), alfa
    return klatka, alfa_z_maski(segmentuj(klatka)["osoba"])

def pokaz_zielonosc(z):
    """Mapa zieloności: im jaśniej, tym bardziej zielony piksel; czerwone = dokładnie na progu."""
    p = float(ZIELEN["prog"] or 40)
    obraz = cv2.applyColorMap(np.clip(z * 128 / max(p, 1), 0, 255).astype(np.uint8), cv2.COLORMAP_INFERNO)
    obraz[np.abs(z - p) < 1.5] = (60, 60, 255)
    napisz(obraz, "jasne = bardzo zielone (tło) · ciemne = osoba · czerwone = dokładnie na progu", 12, WYS - 34, 16, obrys=2)
    return obraz

_SIATKA = np.array([(c.start, c.end) for c in vision.FaceLandmarksConnections.FACE_LANDMARKS_TESSELATION])
_KONTURY = np.array([(c.start, c.end) for c in vision.FaceLandmarksConnections.FACE_LANDMARKS_CONTOURS])
WAZNE_PUNKTY = {1: "nos", 10: "czoło", 152: "broda", 33: "oko", 263: "oko", 61: "usta", 291: "usta",
                234: "bok", 454: "bok", 103: "czoło L", 332: "czoło P"}

def rysuj_siatke(obraz, punkty, numery=True, siatka=True):
    p = np.round(punkty).astype(np.int32)
    if siatka:
        cv2.polylines(obraz, list(p[_SIATKA]), False, (120, 120, 120), 1, cv2.LINE_AA)
    cv2.polylines(obraz, list(p[_KONTURY]), False, (80, 230, 255), 2, cv2.LINE_AA)
    for x, y in p:
        cv2.circle(obraz, (int(x), int(y)), 1, BIALY, -1)
    if numery:
        for nr, opis in WAZNE_PUNKTY.items():
            x, y = int(p[nr][0]), int(p[nr][1])
            cv2.circle(obraz, (x, y), 4, (0, 0, 255), -1, cv2.LINE_AA)
            napisz(obraz, f"{nr} {opis}", x + 6, y - 9, 13, BIALY, obrys=2)

MINY_NA_PANELU = ["eyeBlinkLeft", "eyeBlinkRight", "jawOpen", "mouthSmileLeft", "mouthSmileRight",
                  "browInnerUp", "cheekPuff", "mouthPucker", "eyeWideLeft", "noseSneerLeft"]

def panel_min(obraz, mina, nazwy=None):
    """Rysuje słupki z wybranymi liczbami miny + 3 najbardziej aktywne w tej chwili."""
    nazwy = list(nazwy or MINY_NA_PANELU)
    inne = sorted((k for k in mina if k not in nazwy and k != "_neutral"), key=lambda k: -mina[k])[:3]
    x0, wiersze = SZER - 305, nazwy + inne
    prostokat(obraz, x0 - 10, 8, SZER - 8, 44 + 24 * len(wiersze) + (20 if inne else 0), (20, 20, 20), 0.7)
    napisz(obraz, "MINA W LICZBACH (0–1)", x0, 14, 15, ZOLTY)
    y = 40
    for i, nazwa in enumerate(wiersze):
        if i == len(nazwy):
            napisz(obraz, "najbardziej aktywne teraz:", x0, y, 12, (180, 180, 180))
            y += 20
        v = float(mina.get(nazwa, 0.0))
        napisz(obraz, nazwa, x0, y, 13, BIALY)
        bx = x0 + 150
        cv2.rectangle(obraz, (bx, y + 3), (bx + 100, y + 15), (70, 70, 70), -1)
        cv2.rectangle(obraz, (bx, y + 3), (bx + int(100 * v), y + 15), (80, 220, 120) if v < 0.5 else (60, 160, 255), -1)
        napisz(obraz, f"{v:.2f}", bx + 106, y, 13, BIALY)
        y += 24

def wektor_miny(mina):
    """52 liczby z modelu → jeden wektor (zawsze w tej samej kolejności)."""
    return np.array([mina[k] for k in sorted(mina) if k != "_neutral"], dtype=np.float32)

def przeskaluj(x, od, do):
    """Liczba z przedziału [od, do] → [0, 1] (poza przedziałem przycinana)."""
    return float(np.clip((x - od) / (do - od), 0, 1))

def kolko(obraz, punkt, promien, kolor=CZERWONY, grubosc=-1):
    """Kółko o środku w punkcie (x, y). grubosc=-1 → wypełnione."""
    cv2.circle(obraz, (int(punkt[0]), int(punkt[1])), int(promien), kolor, int(grubosc), cv2.LINE_AA)

def linia(obraz, a, b, kolor=BIALY, grubosc=3):
    """Odcinek od punktu a do punktu b."""
    cv2.line(obraz, (int(a[0]), int(a[1])), (int(b[0]), int(b[1])), kolor, int(grubosc), cv2.LINE_AA)

# ---- live coding: sprawdzanie luk ____ i próba Waszego kodu przed włączeniem kamery ----
import inspect, traceback

NAZWY_MIN = ["_neutral", "browDownLeft", "browDownRight", "browInnerUp", "browOuterUpLeft", "browOuterUpRight",
             "cheekPuff", "cheekSquintLeft", "cheekSquintRight", "eyeBlinkLeft", "eyeBlinkRight", "eyeLookDownLeft",
             "eyeLookDownRight", "eyeLookInLeft", "eyeLookInRight", "eyeLookOutLeft", "eyeLookOutRight", "eyeLookUpLeft",
             "eyeLookUpRight", "eyeSquintLeft", "eyeSquintRight", "eyeWideLeft", "eyeWideRight", "jawForward", "jawLeft",
             "jawOpen", "jawRight", "mouthClose", "mouthDimpleLeft", "mouthDimpleRight", "mouthFrownLeft",
             "mouthFrownRight", "mouthFunnel", "mouthLeft", "mouthLowerDownLeft", "mouthLowerDownRight",
             "mouthPressLeft", "mouthPressRight", "mouthPucker", "mouthRight", "mouthRollLower", "mouthRollUpper",
             "mouthShrugLower", "mouthShrugUpper", "mouthSmileLeft", "mouthSmileRight", "mouthStretchLeft",
             "mouthStretchRight", "mouthUpperUpLeft", "mouthUpperUpRight", "noseSneerLeft", "noseSneerRight"]

_PODPOWIEDZI_LIVE = {
    "truth value of an array": "na tablicach NumPy piszcie np.maximum(a, b) zamiast max(a, b), a w warunkach & i | zamiast and i or",
    "is not defined": "literówka w nazwie albo nie uruchomiono silnika (pierwsza komórka z kodem)",
    "NoneType": "któraś funkcja nic nie zwraca — sprawdźcie, czy na końcu jest return",
    "Nie ma miny": "literówka w nazwie miny — wyżej jest podpowiedź poprawnej nazwy",
    "could not be broadcast": "wynik ma zły kształt — sprawdźcie nawiasy i to, co mnożycie",
    "not 'list'": "funkcja ma oddać jedną liczbę, a oddaje całą listę — policzcie z niej np. średnią",
}

def gotowe(*funkcje, proba=None):
    """Live coding: sprawdza, czy w Waszych funkcjach nie zostały luki ____, i raz próbuje je wywołać.
    Zwraca True, gdy można włączyć kamerę."""
    braki = []
    for f in funkcje:
        try:
            linie = inspect.getsource(f).split("\n")
        except (OSError, TypeError):
            continue
        braki += [f"   {f.__name__}:  {l.strip()}" for l in linie if "____" in l and not l.strip().startswith("#")]
    if braki:
        print("🧑‍💻 Jeszcze nie wszystko wpisane — uzupełnijcie luki ____ razem z prowadzącym:")
        print("\n".join(braki))
        return False
    if proba is not None:
        try:
            proba()
        except Exception as e:
            nazwy = {f.__name__ for f in funkcje}
            slady = [s for s in traceback.extract_tb(e.__traceback__) if s.name in nazwy]
            print(f"❌ Próba Waszego kodu się nie udała — {type(e).__name__}: {e}")
            if slady:
                print(f"   w funkcji {slady[-1].name}:  {slady[-1].line}")
            for klucz, rada in _PODPOWIEDZI_LIVE.items():
                if klucz in f"{type(e).__name__}: {e}":
                    print("💡 Podpowiedź:", rada)
                    break
            print("Poprawcie kod (porównajcie z rzutnikiem) i uruchomcie komórkę jeszcze raz.")
            return False
    return True

def sprawdz_mape(wynik, klatka, nazwa):
    """Live coding: Wasza funkcja ma oddać jedną liczbę (albo True/False) dla każdego piksela klatki."""
    if wynik is None:
        raise ValueError(f"{nazwa} nic nie zwraca — brakuje return?")
    mapa = np.asarray(wynik)
    if mapa.shape != klatka.shape[:2]:
        jest = "jedną wartość" if mapa.ndim == 0 else "tablicę " + " × ".join(map(str, mapa.shape))
        raise ValueError(f"{nazwa} ma zwrócić jedną liczbę dla każdego piksela (tablicę {klatka.shape[0]} × "
                         f"{klatka.shape[1]}), a zwraca {jest}.")
    return mapa.astype(np.float32)

def sprawdz_obraz(wynik, wzor, nazwa):
    """Live coding: Wasza funkcja ma oddać obraz tego samego rozmiaru co wzór."""
    if wynik is None:
        raise ValueError(f"{nazwa} nic nie zwraca — brakuje return?")
    if np.shape(wynik) != np.shape(wzor):
        raise ValueError(f"{nazwa} ma zwrócić obraz rozmiaru {np.shape(wzor)}, a zwraca {np.shape(wynik)}.")
    return wynik

def _klatka_proby():
    """Sztuczna klatka do próby kodu: szare tło, „osoba” i zielona kartka."""
    k = np.full((WYS, SZER, 3), (90, 110, 120), np.uint8)
    cv2.ellipse(k, (SZER // 2, WYS // 2), (90, 120), 0, 0, 360, (130, 160, 210), -1)
    cv2.rectangle(k, (60, 120), (260, 380), (60, 170, 70), -1)
    return k

def _punkty_proby():
    """Sztuczne punkty twarzy (478 × 2) do próby kodu."""
    kat = np.linspace(0, 2 * np.pi, 478)
    return np.stack([SZER / 2 + 80 * np.cos(kat), WYS / 2 + 100 * np.sin(kat)], 1).astype(np.float32)

def _mina_proby():
    """Sztuczna mina do próby kodu: wszystkie 52 liczby, usta otwarte, uśmiech, uniesione brwi."""
    mina = Mina({n: 0.1 for n in NAZWY_MIN})
    mina.update(jawOpen=0.9, mouthSmileLeft=0.9, mouthSmileRight=0.9, browInnerUp=0.9)
    return mina

def _miny_proby():
    """Trzy sztuczne miny do próby kodu: zwykła, wszystko na maksa i wszystko na zero — żeby sprawdzić każdy if."""
    return [_mina_proby(), Mina({n: 0.95 for n in NAZWY_MIN}), Mina({n: 0.0 for n in NAZWY_MIN})]

def _proba_twarzy():
    """Sztuczne (twarz, tlo, alfa) do próby funkcji wtop."""
    twarz = np.full((8, 8, 3), 200, np.float32)
    tlo = np.full((8, 8, 3), 50, np.float32)
    alfa = np.linspace(0, 1, 64, dtype=np.float32).reshape(8, 8, 1)
    return twarz, tlo, alfa

print("✅ Silnik, część 2 gotowa", "(czcionka: " + (os.path.basename(CZCIONKA) if CZCIONKA else "brak — napisy bez ogonków") + ")")

# ▶ 0.5 SILNIK, część 3: oprawa telewizyjna. Uruchom — nie musisz tego czytać.
MANDARYNKA_NA_GLOWIE = True

def dymek(obraz, punkt, tekst, skala=1.0):
    napis = render_napisu(tekst, int(np.clip(22 * skala, 14, 34)), CZARNY)
    h, w = napis.shape[:2]
    x = int(np.clip(punkt[0], 10, SZER - w - 30))
    y = int(np.clip(punkt[1], 30, WYS - h - 30))
    ogon = np.array([[x + 14, y + h + 12], [x + 34, y + h + 12], [x + 6, y + h + 30]], np.int32)
    cv2.fillPoly(obraz, [ogon], BIALY, cv2.LINE_AA)
    cv2.rectangle(obraz, (x - 4, y - 4), (x + w + 18, y + h + 14), BIALY, -1)
    cv2.rectangle(obraz, (x - 4, y - 4), (x + w + 18, y + h + 14), CZARNY, 2)
    wklej(obraz, napis, x + 7, y + 5)

def podglad_kamery(obraz, klatka, szer=200):
    h = int(szer * WYS / SZER)
    maly = cv2.resize(klatka, (szer, h))
    x0, y0 = SZER - szer - 12, WYS - h - 12
    obraz[y0:y0 + h, x0:x0 + szer] = maly
    cv2.rectangle(obraz, (x0 - 2, y0 - 2), (x0 + szer + 1, y0 + h + 1), BIALY, 2)

KANAL, NAGLOWEK = "KAPIBARA TV", "PILNE"
TYTUL = "Kapibary przejęły Politechnikę Bydgoską"
PASEK = "Rektor negocjuje dostawy arbuzów • Wykłady przeniesione do jacuzzi • Studenci zachowują spokój"
ZNAK_WODNY = "MATERIAŁ SYNTETYCZNY · FIKCJA · WARSZTATY"

def nakladki_tv(obraz, t=None):
    t = time.time() if t is None else t
    prostokat(obraz, 0, 0, SZER, 22, ZOLTY, 0.9)
    znak = render_napisu(ZNAK_WODNY, 13, CZARNY)
    wklej(obraz, znak, (SZER - znak.shape[1]) // 2, 11 - znak.shape[0] // 2)
    logo = render_napisu(KANAL, 20, BIALY)
    prostokat(obraz, 14, 32, 34 + logo.shape[1], 68, CZERWONY)
    wklej(obraz, logo, 24, 50 - logo.shape[0] // 2)
    na_zywo = render_napisu("NA ŻYWO  " + time.strftime("%H:%M"), 16, BIALY)
    x0 = SZER - na_zywo.shape[1] - 50
    prostokat(obraz, x0, 32, SZER - 14, 68, (20, 20, 20), 0.75)
    if int(t * 2) % 2 == 0:
        cv2.circle(obraz, (x0 + 15, 50), 6, (40, 40, 230), -1, cv2.LINE_AA)
    wklej(obraz, na_zywo, x0 + 28, 50 - na_zywo.shape[0] // 2)
    y0 = WYS - 92
    tag, tyt = render_napisu(NAGLOWEK, 20, BIALY), render_napisu(TYTUL, 22, (40, 25, 15))
    w_tag = tag.shape[1] + 20
    prostokat(obraz, 14, y0, 14 + w_tag, y0 + 40, CZERWONY)
    wklej(obraz, tag, 24, y0 + 20 - tag.shape[0] // 2)
    prostokat(obraz, 14 + w_tag, y0, min(SZER - 14, 38 + w_tag + tyt.shape[1]), y0 + 40, BIALY, 0.95)
    wklej(obraz, tyt, 26 + w_tag, y0 + 20 - tyt.shape[0] // 2)
    prostokat(obraz, 0, WYS - 38, SZER, WYS, (60, 28, 12), 0.95)
    tekst = render_napisu(PASEK + "     •     ", 17, BIALY)
    x = -(int(t * 90) % tekst.shape[1])
    while x < SZER:
        wklej(obraz, tekst, x, WYS - 19 - tekst.shape[0] // 2)
        x += tekst.shape[1]
    return obraz

def mapuj_mine(mina):
    usmiech = (mina["mouthSmileLeft"] + mina["mouthSmileRight"]) / 2
    return {"oko_lewe": mina["eyeBlinkRight"] > 0.45, "oko_prawe": mina["eyeBlinkLeft"] > 0.45,
            "pysk": przeskaluj(mina["jawOpen"], 0.05, 0.6), "usmiech": usmiech > 0.5,
            "rumience": usmiech > 0.7, "brwi": mina["browInnerUp"] > 0.4}

print("✅ Silnik, część 3 gotowa")

# ▶ 0.6 SILNIK, część 4: kapibara 3D ze zdjęcia. Uruchom — nie musisz tego czytać.
FOTY = {
    "garnitur": dict(
        plik="kapibara_garnitur.png", opis="prezenter w studiu", zrodlo="",
        kadr=(0.5, 40 / 1484, 960 / 1484), tlo="jednolite", garnitur=True,
        czaszka=(437, 190, 121, 152, 80), pysk3d=(437, 215, 40, 55, 48), obrot=(437, 300, -70), szyja=None,
        szyja_skret=(265, 345, 0.25),
        nos=(436, 192), usta=(440, 262, 12, 30, 4, 18, 60),
        oko_lewe=(364, 142, 8.5, 6, -5), oko_prawe=(508, 141, 8.5, 6, 5),
        ucho_lewe=(357, 95, 352, 55), ucho_prawe=(503, 92, 508, 52),
        mandarynka=(434, 56, 25), dymek=(548, 70)),
    "trawa": dict(
        plik="kapibara_trawa.jpg", opis="korespondentka w terenie", zrodlo="",
        kadr=(0.5, 0.0, 1.0), tlo="zielone", garnitur=False,
        czaszka=(418, 200, 140, 165, 85), pysk3d=(416, 235, 42, 60, 50), obrot=(418, 330, -70), szyja=(330, 400),
        szyja_skret=(300, 400, 0.2),
        nos=(414, 205), usta=(418, 295, 12, 30, 4, 18, 75),
        oko_lewe=(341, 142, 8.5, 7, 0), oko_prawe=(492, 142, 8.5, 7, 0),
        ucho_lewe=(345, 95, 342, 58), ucho_prawe=(488, 95, 491, 58),
        mandarynka=(416, 70, 24), dymek=(538, 75)),
}
OGNISKOWA = 1300                              # „odległość kamery” w pikselach – mniejsza = mocniejsza perspektywa
SILA_SWIATLA = 0.45                           # jak mocno zmienia się oświetlenie przy obrocie głowy (0 = wcale)
MAKS_SKRET, MAKS_KIWNIECIE, MAKS_PRZECHYLENIE = 32, 18, 15   # największe kąty głowy kapibary (w stopniach)
ODWROC_SKRET, ODWROC_KIWNIECIE = False, False  # True, jeśli kapibara skręca/kiwa głową w odwrotną stronę niż Ty
_fk = {"dane": {}, "foto": None}
KP = None                                     # punkty i parametry aktualnego zdjęcia

def dostepne_foty():
    return [n for n, f in FOTY.items() if os.path.exists(os.path.join(FOLDER_ZDJEC, f["plik"]))]

def nastepne_foto(teraz):
    foty = dostepne_foty() or list(FOTY)
    return foty[(foty.index(teraz) + 1) % len(foty)] if teraz in foty else foty[0]

def _ss(x):
    """Płynne przejście 0 → 1 (smoothstep)."""
    x = np.clip(x, 0.0, 1.0)
    return x * x * (3 - 2 * x)

def _zrob_mandarynke(r):
    """Mandarynka narysowana „fotograficznie”: cieniowana kula z fakturą skórki (obrazek BGRA)."""
    n = 2 * r + 8
    yy, xx = np.mgrid[0:n, 0:n].astype(np.float32)
    c = n / 2
    u, v = (xx - c) / r, (yy - c) / (r * 0.9)
    d2 = u * u + v * v
    alfa = np.clip((1 - np.sqrt(d2)) * r * 0.9, 0, 1)
    z = np.sqrt(np.clip(1 - d2, 0, 1))
    swiatlo = np.clip(-0.45 * u - 0.6 * v + 0.66 * z, 0, 1)
    rng = np.random.default_rng(7)
    faktura = cv2.GaussianBlur(rng.normal(0, 1, (n, n)).astype(np.float32), (0, 0), 0.7)
    faktura += 0.6 * cv2.GaussianBlur(rng.normal(0, 1, (n, n)).astype(np.float32), (0, 0), 2.0)
    kolor = np.array([18, 120, 240], np.float32)
    obraz = kolor * (0.4 + 0.75 * swiatlo[..., None]) * (1 + 0.09 * faktura[..., None])
    obraz += np.array([10, 18, 10]) * (1 - z[..., None]) * 0.8
    blysk = np.clip(1 - np.hypot(u + 0.36, v + 0.4) / 0.32, 0, 1) ** 2 * (0.6 + 0.4 * (faktura > 0))
    obraz += blysk[..., None] * 90
    obraz = np.clip(obraz, 0, 255)
    sx, sy = int(c + 0.05 * r), int(c - 0.86 * r)
    cv2.circle(obraz, (sx, sy + 2), max(2, r // 7), (40, 70, 60), -1, cv2.LINE_AA)
    cv2.ellipse(obraz, (sx + r // 3, sy - 1), (r // 2, max(2, r // 6)), -25, 0, 360, (40, 105, 45), -1, cv2.LINE_AA)
    cv2.ellipse(alfa, (sx + r // 3, sy - 1), (r // 2, max(2, r // 6)), -25, 0, 360, 1.0, -1, cv2.LINE_AA)
    return np.dstack([obraz, alfa * 255]).astype(np.uint8)

def _obrot(X, Y, cx, cy, kat, w):
    """Odwrotny obrót punktów (X, Y) wokół (cx, cy) o kąt × w (w = waga 0..1 dla każdego piksela)."""
    a = np.float32(-math.radians(kat)) * w
    c, s = np.cos(a), np.sin(a)
    return cx + (X - cx) * c - (Y - cy) * s, cy + (X - cx) * s + (Y - cy) * c

def _geometria_pyska(X, pysk):
    """Górna krawędź otworu pyska (lekki łuk) i jego wysokość w każdej kolumnie."""
    mx, my, pelna, zero, luk, maks, _ = KP["usta"]
    dx = X - mx
    gora = my - luk * np.minimum((dx / zero) ** 2, 1)
    return gora, maks * pysk * (1 - _ss((np.abs(dx) - pelna) / (zero - pelna)))

def _uklad_oka(oko, X, Y):
    """Współrzędne względem oka: v – w poprzek oka (w dół); profil = połowa wysokości oka w danym miejscu."""
    ex, ey, rx, ry, kat = oko
    a = math.radians(kat)
    vx, vy = -math.sin(a), math.cos(a)
    if vy < 0:
        vx, vy = -vx, -vy
    ux, uy = vy, -vx
    u, v = (X - ex) * ux + (Y - ey) * uy, (X - ex) * vx + (Y - ey) * vy
    rxm, rym = rx + 2, ry + 2
    return v, np.sqrt(np.clip(1 - (u / rxm) ** 2, 0, 1)) * rym, vx, vy

def _wycinek(ex, ey, R):
    x0, y0 = _fk["roi"][:2]
    return np.s_[max(int(ey) - R - y0, 0):int(ey) + R - y0, max(int(ex) - R - x0, 0):int(ex) + R - x0]

def _deformacje_lokalne(X, Y, pysk, oczy, zdziw, zadow, ucho):
    """Mapa: z którego miejsca zdjęcia wziąć kolor każdego piksela — część 1: pysk, powieki, oczy, uszy."""
    SX, SY = X.copy(), Y.copy()
    if pysk > 0.01:
        gora, h = _geometria_pyska(X, pysk)
        SY -= h * (Y >= gora) * (1 - _ss((Y - gora) / KP["usta"][6]))
    for nazwa, b in zip(("oko_lewe", "oko_prawe"), oczy):
        if b < 0.02:
            continue
        w = _wycinek(*KP[nazwa][:2], 34)
        v, profil, vx, vy = _uklad_oka(KP[nazwa], X[w], Y[w])
        gorna, dolna = profil * (-1 + 1.3 * b), profil * (1 - 0.7 * b)
        przes = np.where(v < gorna, -1.3 * b * profil * (1 - _ss((gorna - v) / 22)), 0)
        przes = np.where(v > dolna, 0.7 * b * profil * (1 - _ss((v - dolna) / 16)), przes)
        SX[w] += przes * vx
        SY[w] += przes * vy
    if zdziw > 0.01:
        for nazwa in ("oko_lewe", "oko_prawe"):
            ex, ey = KP[nazwa][:2]
            k = 0.32 * zdziw * (1 - _ss(np.hypot(X - ex, Y - ey) / 19))
            SX -= (X - ex) * k / (1 + k)
            SY -= (Y - ey) * k / (1 + k)
        for nazwa in ("ucho_lewe", "ucho_prawe"):
            bx, by, tx, ty = KP[nazwa]
            SY += 6 * zdziw * (1 - _ss(np.hypot(X - (bx + tx) / 2, Y - (by + ty) / 2) / 30))
    if zadow > 0.01:
        for nazwa in ("oko_lewe", "oko_prawe"):
            ex, ey = KP[nazwa][:2]
            w = 1 - _ss(np.hypot((X - ex - (KP["nos"][0] - ex) * 0.15) / 1.4, Y - ey - 22) / 24)
            SY += 3 * zadow * w
    kat_ucha, ktore, _ = ucho
    if abs(kat_ucha) > 0.3:
        bx, by, tx, ty = KP["ucho_lewe" if ktore == 0 else "ucho_prawe"]
        w = 1 - _ss(np.hypot(X - (bx + tx) / 2, Y - (by + ty) / 2) / 26)
        SX, SY = _obrot(SX, SY, bx, by, kat_ucha * (1 if ktore == 0 else -1), w)
    return SX, SY

def _rysuj_rzesy(roi, oko, b):
    """Ciemna linia na krawędzi powieki i lekki cień pod nią."""
    if b < 0.05:
        return
    w = _wycinek(*oko[:2], 16)
    v, profil, _, _ = _uklad_oka(oko, _fk["X"][w], _fk["Y"][w])
    gorna, dolna = profil * (-1 + 1.3 * b), profil * (1 - 0.7 * b)
    sila = min(1.0, 2 * b) * (profil > 0.3)
    gora_oka = -profil - 4
    t = np.clip((v - gora_oka) / np.maximum(gorna - gora_oka, 1), 0, 1)
    powieka = _ss((v - gora_oka) / 4) * (1 - _ss((v - gorna + 0.5) / 1.0)) * (0.06 + 0.3 * t ** 1.5)
    powieka += _ss((v - dolna + 0.5) / 1.0) * (1 - _ss((v - profil - 0.5) / 3)) * 0.2
    ciemniej = (0.5 * np.exp(-((v - gorna) / 0.9) ** 2) + powieka) * sila
    obszar = roi[w][..., :3]
    obszar[:] = (obszar * (1 - ciemniej[..., None]) + np.array([18, 16, 20]) * ciemniej[..., None]).astype(np.uint8)

def _rysuj_pysk(roi, pysk):
    """Wnętrze otwartego pyska: ciemna jama, pomarańczowe siekacze (jak u prawdziwej kapibary) i język."""
    if pysk < 0.06:
        return
    mx, my, _, zero, _, maks, _ = KP["usta"]
    w = _wycinek(mx, my + 8, int(zero + 12))
    X, Y = _fk["X"][w], _fk["Y"][w]
    gora, h = _geometria_pyska(X, pysk)
    h = np.maximum(h, 1e-3)
    t = np.clip((Y - gora) / h, 0, 1)
    maska = np.clip(np.minimum(Y - gora + 0.5, gora + h - Y + 0.5), 0, 1) * (h > 0.6)
    kolor = np.dstack([16 + 14 * t, 14 + 12 * t, 20 + 24 * t])
    jezyk = np.clip(1 - np.hypot((X - mx) / 10, (t - 1) / 0.45), 0, 1) * np.clip((pysk - 0.35) / 0.3, 0, 1)
    kolor = kolor * (1 - 0.8 * jezyk[..., None]) + np.array([70, 76, 128]) * 0.8 * jezyk[..., None]
    D = maks * pysk
    dx = np.abs(X - mx)
    dlugosc = min(0.5 * D, 8.5) * (1 - 0.35 * ((dx - 3.2) / 2.6) ** 2)
    zab = (dx > 0.6) & (dx < 5.8) & (Y - gora < dlugosc)
    tz = np.clip((Y - gora) / np.maximum(dlugosc, 1e-3), 0, 1)
    bok = np.clip(1 - np.abs(dx - 3.2) / 2.6, 0, 1)
    zab_kolor = np.array([75, 160, 215]) * (0.62 + 0.3 * tz[..., None] + 0.15 * bok[..., None])
    dolne = (dx > 0.6) & (dx < 4.6) & (gora + h - Y < min(0.28 * D, 4.5))
    kolor = np.where(zab[..., None], zab_kolor, kolor)
    kolor = np.where(dolne[..., None], kolor * 0.6 + np.array([62, 135, 185]) * 0.4, kolor)
    kolor = cv2.GaussianBlur(kolor.astype(np.float32), (0, 0), 0.55)
    maska = cv2.GaussianBlur(maska.astype(np.float32), (0, 0), 0.6)[..., None]
    obszar = roi[w][..., :3]
    pod = np.clip(1 - np.abs(Y - (gora + h + 1.5)) / 2.5, 0, 1) * (h > 2) * 0.25
    obszar[:] = np.clip(obszar * (1 - pod[..., None]) * (1 - maska) + kolor * maska, 0, 255).astype(np.uint8)

def _kadruj(img, kadr, wypelnienie):
    """Skaluje i przycina zdjęcie do 854 × 480; brakujące boki wypełnia podanym kolorem."""
    H, W = img.shape[:2]
    cx, ya, yb = kadr[0] * W, kadr[1] * H, kadr[2] * H
    s = WYS / (yb - ya)
    mniejsze = cv2.resize(img, (max(1, round(W * s)), max(1, round(H * s))),
                          interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC)
    wynik = np.empty((WYS, SZER) + img.shape[2:], img.dtype)
    wynik[:] = wypelnienie
    ox, oy = round((cx - (yb - ya) * SZER / WYS / 2) * s), round(ya * s)
    h, w = mniejsze.shape[:2]
    x0, y0, x1, y1 = max(ox, 0), max(oy, 0), min(ox + SZER, w), min(oy + WYS, h)
    wynik[y0 - oy:y1 - oy, x0 - ox:x1 - ox] = mniejsze[y0:y1, x0:x1]
    return wynik

def _najwiekszy_ksztalt(maska):
    """Zostawia największy spójny kształt i wypełnia w nim dziury (np. szary nos podobny do tła)."""
    n, etykiety, staty, _ = cv2.connectedComponentsWithStats(maska.astype(np.uint8))
    if n <= 1:
        return maska.astype(bool)
    ksztalt = (etykiety == 1 + int(np.argmax(staty[1:, cv2.CC_STAT_AREA]))).astype(np.uint8)
    tlo = (1 - ksztalt).copy()
    pom = np.zeros((tlo.shape[0] + 2, tlo.shape[1] + 2), np.uint8)
    for x, y in ((0, 0), (tlo.shape[1] - 1, 0), (0, tlo.shape[0] - 1), (tlo.shape[1] - 1, tlo.shape[0] - 1)):
        if tlo[y, x] == 1:
            cv2.floodFill(tlo, pom, (x, y), 2)
    return tlo != 2

def _alfa_postaci(img, rodzaj):
    """Przezroczystość postaci (0 = tło, 1 = kapibara) i kolory oczyszczone z poświaty tła."""
    if rodzaj == "jednolite":
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).astype(np.float32)
        d = lab - np.concatenate([lab[:40, :40].reshape(-1, 3), lab[:40, -40:].reshape(-1, 3)]).mean(0)
        alfa = _ss((np.sqrt((0.5 * d[..., 0]) ** 2 + (2.2 * d[..., 1]) ** 2 + (2.2 * d[..., 2]) ** 2) - 4) / 30)
    else:
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
        ziel = (np.clip((hsv[..., 1] - 50) / 40, 0, 1) * np.clip(1 - np.abs(hsv[..., 0] - 50) / 30, 0, 1)
                * np.clip((hsv[..., 2] - 40) / 40, 0, 1))
        alfa = cv2.GaussianBlur(1 - ziel, (0, 0), 1.0)
    ksztalt = _najwiekszy_ksztalt(alfa > 0.5).astype(np.uint8)
    elipsa_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    wnetrze = cv2.erode(ksztalt, elipsa_k)
    na_zewnatrz = 1 - cv2.dilate(ksztalt, elipsa_k)
    alfa = np.where(wnetrze > 0, 1.0, np.where(na_zewnatrz > 0, 0.0, alfa)).astype(np.float32)
    alfa = cv2.GaussianBlur(alfa, (0, 0), 0.7)
    kolor_tla = np.median(img[:30, :30].reshape(-1, 3), 0).astype(np.float32)
    czyste = np.clip(kolor_tla + (img - kolor_tla) / np.maximum(alfa, 0.05)[..., None], 0, 255)
    czyste = np.where((alfa > 0.97)[..., None], img, czyste)
    return alfa, czyste.astype(np.float32)

def _maska_garnituru(img, alfa):
    """Garnitur, koszula i krawat: mało nasycone albo niebieskawe piksele pod głową (szukane od dołu kadru)."""
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.int32)
    yy = np.mgrid[0:WYS, 0:SZER][0]
    surowa = ((yy > 280) & ((hsv[..., 1] < 40) | (hsv[..., 0] > 35)) & (alfa > 0.5)).astype(np.uint8)
    n, etykiety, staty, _ = cv2.connectedComponentsWithStats(surowa)
    dolne = np.zeros_like(surowa)
    for i in range(1, n):
        if staty[i, cv2.CC_STAT_TOP] + staty[i, cv2.CC_STAT_HEIGHT] >= WYS - 1 and staty[i, cv2.CC_STAT_AREA] > 500:
            dolne[etykiety == i] = 1
    dolne = cv2.morphologyEx(dolne, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
    pierwszy = np.where(dolne.any(0), dolne.argmax(0), WYS)
    maska = (yy >= pierwszy[None, :]).astype(np.float32)
    return cv2.GaussianBlur(maska, (0, 0), 0.8) * alfa

def _glebia(f):
    """Bryła 3D głowy: głębokość z (w pikselach, w stronę kamery) dla każdego piksela kadru."""
    Y, X = np.mgrid[0:WYS, 0:SZER].astype(np.float32)
    cx, cy, rx, ry, rz = f["czaszka"]
    r = (np.abs((X - cx) / rx) ** 2.6 + np.abs((Y - cy) / ry) ** 2.6) ** (1 / 2.6)
    z_czaszka = rz * np.clip(1 - r ** 2, 0, 1)
    px, py, sx, sy, wysuniecie = f["pysk3d"]
    pysk = wysuniecie * np.exp(-((X - px) ** 2 / (2 * sx ** 2) + (Y - py) ** 2 / (2 * sy ** 2)))
    return (z_czaszka + pysk * np.clip(1 - r, 0, 1) ** 0.3).astype(np.float32)

SWIATLO = np.array([-0.4, -0.5, 0.77], np.float32)            # skąd pada światło (z lewej, z góry, z przodu)

def _normalne(glebia):
    """Kierunek „na zewnątrz” powierzchni bryły w każdym pikselu (potrzebny do cieniowania)."""
    gy, gx = np.gradient(cv2.GaussianBlur(glebia, (0, 0), 3))
    n = np.dstack([-gx, -gy, np.ones_like(glebia)])
    return (n / np.linalg.norm(n, axis=2, keepdims=True)).astype(np.float32)

def _cieniowana_bryla(glebia):
    """Szara, oświetlona bryła — żeby zobaczyć sam model 3D bez zdjęcia."""
    jasnosc = np.clip(_normalne(glebia) @ SWIATLO, 0, 1) * 0.8 + 0.2
    return np.dstack([jasnosc * 205, jasnosc * 200, jasnosc * 190]).astype(np.float32)

def _siatka_na(kolor, alfa):
    siatka = np.zeros(kolor.shape[:2], np.uint8)
    for x in range(0, SZER, 12):
        cv2.line(siatka, (x, 0), (x, WYS - 1), 255, 1, cv2.LINE_AA)
    for y in range(0, WYS, 12):
        cv2.line(siatka, (0, y), (SZER - 1, y), 255, 1, cv2.LINE_AA)
    g = (siatka.astype(np.float32) / 255 * 0.6)[..., None]
    return kolor * (1 - g) + np.array([255, 255, 255]) * g

def _punkty_bryly(glowa, glebia, f):
    """Punkty bryły do rzutowania: co drugi piksel głowy (kolory i tak pobieramy później z pełnego zdjęcia)."""
    ys, xs = np.nonzero(glowa[::2, ::2] > 0.01)
    ys, xs = ys * 2, xs * 2
    ox, oy, oz = f["obrot"]
    s0, s1, minimum = f["szyja_skret"]
    return dict(punkty=np.stack([xs - ox, ys - oy, glebia[ys, xs] - oz], 1).astype(np.float32),
                waga_obrotu=(1 - (1 - minimum) * _ss((ys - s0) / (s1 - s0))).astype(np.float32)[:, None],
                zrodlo_xy=np.stack([xs, ys], 1).astype(np.float32))

def _przygotuj(nazwa):
    """Jednorazowo: warstwy, bryła 3D i punkty do rzutowania dla jednego zdjęcia."""
    f = FOTY[nazwa]
    plik = os.path.join(FOLDER_ZDJEC, f["plik"])
    if not os.path.exists(plik):
        raise FileNotFoundError(f"Brak zdjęcia kapibary: {plik}. Skopiuj zdjęcia do folderu {FOLDER_ZDJEC}/.")
    img = cv2.imdecode(np.fromfile(plik, np.uint8), cv2.IMREAD_COLOR)
    tlo_kolor = np.median(img[:30, :30].reshape(-1, 3), 0).astype(np.uint8).tolist()
    img = _kadruj(img, f["kadr"], tlo_kolor)
    alfa, kolor = _alfa_postaci(img, f["tlo"])
    Y = np.mgrid[0:WYS, 0:SZER][0].astype(np.float32)
    if f["garnitur"]:
        garnitur = _maska_garnituru(img, alfa)
        glowa = np.clip(alfa - garnitur, 0, 1)
        rdzen = cv2.dilate(_najwiekszy_ksztalt(glowa > 0.5).astype(np.uint8), np.ones((5, 5), np.uint8))
        glowa *= cv2.GaussianBlur(rdzen.astype(np.float32), (0, 0), 1.0)
        pod = (garnitur > 0.5) & (Y < Y[np.nonzero(glowa > 0.5)].max() + 30)
        pas = cv2.dilate((glowa > 0.5).astype(np.uint8), np.ones((61, 41), np.uint8)) > 0
        dorobic = (pod & pas).astype(np.uint8)
        warstwa_gora = np.dstack([kolor * garnitur[..., None], garnitur * 255])
        kolor = cv2.inpaint(np.clip(kolor, 0, 255).astype(np.uint8), dorobic, 5, cv2.INPAINT_TELEA).astype(np.float32)
        glowa = np.maximum(glowa, dorobic.astype(np.float32))
        warstwa_dol, tlo_zdjecia = None, None
    else:
        s0, s1 = f["szyja"]
        glowa = alfa * (1 - _ss((Y - s0) / (s1 - s0)))
        maly = cv2.resize(img, (SZER // 4, WYS // 4), interpolation=cv2.INTER_AREA)
        dziura = cv2.resize(cv2.dilate((alfa > 0.02).astype(np.uint8), np.ones((15, 15), np.uint8)),
                            (SZER // 4, WYS // 4), interpolation=cv2.INTER_NEAREST)
        tlo_zdjecia = cv2.resize(cv2.inpaint(maly, dziura, 7, cv2.INPAINT_TELEA), (SZER, WYS),
                                 interpolation=cv2.INTER_CUBIC).astype(np.float32)
        czesc_glowy = glowa / np.maximum(alfa, 1e-3)
        cialo = alfa * _ss((1 - czesc_glowy) / 0.08)
        warstwa_dol = np.dstack([kolor * cialo[..., None], cialo * 255])
        warstwa_gora = None
    glebia = _glebia(f)
    ys, xs = np.nonzero(glowa > 0.01)
    ramka_glowy = (xs.min(), ys.min(), xs.max(), ys.max())
    ramka = (max(int(xs.min()) - 2, 0), max(int(ys.min()) - 2, 0), min(int(xs.max()) + 3, SZER), min(int(ys.max()) + 3, WYS))
    ox, oy, oz = f["obrot"]
    tekstury = {}
    for tryb, kol in (("zdjecie", kolor), ("siatka", _siatka_na(kolor, glowa)), ("bryla", _cieniowana_bryla(glebia))):
        tekstury[tryb] = np.dstack([kol * glowa[..., None], glowa * 255]).clip(0, 255).astype(np.uint8)
    x0, y0, x1, y1 = ramka
    Yr, Xr = np.mgrid[y0:y1, x0:x1].astype(np.float32)
    return dict(
        f=f, tekstury=tekstury, warstwa_gora=warstwa_gora, warstwa_dol=warstwa_dol, tlo_zdjecia=tlo_zdjecia,
        roi=ramka, X=Xr, Y=Yr, glebia=glebia, normalne=_normalne(glebia),
        **_punkty_bryly(glowa, glebia, f),

        mandarynka=_zrob_mandarynke(f["mandarynka"][2]), kompozyt={})

def _macierz_obrotu(skret, kiwniecie, przechylenie):
    """Obrót w 3D (kąty w stopniach): skręt wokół osi pionowej, kiwnięcie wokół poziomej, przechylenie wokół osi patrzenia."""
    a, b, c = (math.radians(v) for v in (skret, kiwniecie, przechylenie))
    Ry = np.array([[math.cos(a), 0, math.sin(a)], [0, 1, 0], [-math.sin(a), 0, math.cos(a)]])
    Rx = np.array([[1, 0, 0], [0, math.cos(b), math.sin(b)], [0, -math.sin(b), math.cos(b)]])
    Rz = np.array([[math.cos(c), -math.sin(c), 0], [math.sin(c), math.cos(c), 0], [0, 0, 1]])
    return (Rz @ Rx @ Ry).astype(np.float32)

def _rzutuj(punkty, R, przes, waga=None):
    """Obraca punkty bryły i rzutuje je perspektywicznie na ekran. Zwraca x, y na ekranie i głębokość."""
    ox, oy, _ = KP["obrot"]
    Q = punkty @ R.T
    if waga is not None:
        Q = waga * Q + (1 - waga) * punkty
        przes = (waga[:, 0] * przes[0], waga[:, 0] * przes[1])
    s = OGNISKOWA / (OGNISKOWA - Q[:, 2])
    return ox + przes[0] + Q[:, 0] * s, oy + przes[1] + Q[:, 1] * s, Q[:, 2]

def _renderuj_glowe(tekstura, R, przes):
    """Rysuje obróconą bryłę z teksturą. Wynik: obraz BGRA (alfa przemnożona) i jego prostokąt na ekranie."""
    qx, qy, qz = _rzutuj(_fk["punkty"], R, przes, _fk["waga_obrotu"])
    bx0, by0 = int(np.floor(qx.min())) - 3, int(np.floor(qy.min())) - 3
    bx1, by1 = int(np.floor(qx.max())) + 4, int(np.floor(qy.max())) + 4
    bw, bh = bx1 - bx0, by1 - by0
    kolejnosc = np.argsort(np.clip(qz * 40, -32000, 32000).astype(np.int16), kind="stable")
    lin = (np.rint(qy[kolejnosc]).astype(np.int32) - by0) * bw + np.rint(qx[kolejnosc]).astype(np.int32) - bx0
    sasiedzi = np.array([-bw - 1, -bw, -bw + 1, -1, 0, 1, bw - 1, bw, bw + 1], np.int32)
    lin = (lin[:, None] + sasiedzi).ravel()
    bufor = np.full(bw * bh, -1, np.int32)
    bufor[lin] = np.repeat(kolejnosc, 9)
    jest = bufor >= 0
    nr = np.where(jest, bufor, 0)
    zr = _fk["zrodlo_xy"]
    siatka_x = np.arange(bx0, bx1, dtype=np.float32)
    siatka_y = np.arange(by0, by1, dtype=np.float32)
    mapa_x = (zr[nr, 0] + np.tile(siatka_x, bh) - qx[nr]).reshape(bh, bw)
    mapa_y = (zr[nr, 1] + np.repeat(siatka_y, bw) - qy[nr]).reshape(bh, bw)
    jest = jest.reshape(bh, bw)
    waga = jest.astype(np.float32)
    suma = cv2.boxFilter(waga, -1, (3, 3), normalize=False)
    dziury = (~jest) & (suma >= 2)
    if dziury.any():
        mapa_x[dziury] = cv2.boxFilter(mapa_x * waga, -1, (3, 3), normalize=False)[dziury] / suma[dziury]
        mapa_y[dziury] = cv2.boxFilter(mapa_y * waga, -1, (3, 3), normalize=False)[dziury] / suma[dziury]
        jest |= dziury
    mapa_x[~jest] = -10
    mapa_y[~jest] = -10
    glowa = cv2.remap(tekstura, mapa_x, mapa_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    if SILA_SWIATLA > 0:
        n = cv2.remap(_fk["normalne"], mapa_x[::3, ::3], mapa_y[::3, ::3], cv2.INTER_LINEAR,
                      borderMode=cv2.BORDER_REPLICATE)
        przed = np.clip(n @ SWIATLO, 0, 1)
        po = np.clip(n @ (R.T @ SWIATLO), 0, 1)
        mnoznik = (1 - SILA_SWIATLA + SILA_SWIATLA * po) / (1 - SILA_SWIATLA + SILA_SWIATLA * przed)
        mnoznik = cv2.resize(mnoznik, (bw, bh), interpolation=cv2.INTER_LINEAR)
        glowa[..., :3] = np.clip(glowa[..., :3] * mnoznik[..., None], 0, 255).astype(np.uint8)
    return glowa, (bx0, by0, bx1, by1)

def _punkt_3d(x, y, dz=0.0):
    """Gdzie na ekranie trafi punkt (x, y) zdjęcia leżący na bryle (plus dz nad powierzchnią)."""
    ox, oy, oz = KP["obrot"]
    p = np.array([[x - ox, y - oy, _fk["glebia"][int(y), int(x)] + dz - oz]], np.float32)
    qx, qy, qz = _rzutuj(p, _fk["R"], _fk["przes"])
    return np.array([qx[0], qy[0]], np.float32), float(OGNISKOWA / (OGNISKOWA - qz[0]))

def _kompozyt(prem, tlo):
    """Obraz z przemnożoną alfą (BGRA) na tle: kolor + tło × (1 − alfa)."""
    a = prem[..., 3:].astype(np.float32) / 255
    return np.clip(prem[..., :3] + tlo * (1 - a), 0, 255).astype(np.uint8)

def _poza_z_twarzy(punkty):
    """Kąty głowy użytkownika: skręt i kiwnięcie z punktów 3D twarzy, przechylenie z linii oczu."""
    srodek, kat, skala = geometria_twarzy(punkty)
    p3 = globals().get("OSTATNIE_PUNKTY_3D")
    if p3 is not None and len(p3) == len(punkty):
        prawo, gora = p3[454] - p3[234], p3[10] - p3[152]
        n = np.cross(prawo, gora)
        n /= max(np.linalg.norm(n), 1e-6)
        if np.dot(n, p3[1] - (p3[234] + p3[454]) / 2) < 0:
            n = -n
        skret = math.degrees(math.atan2(n[0], -n[2]))
        kiw = math.degrees(math.atan2(n[1], -n[2]))
    else:
        srodek_twarzy = (punkty[234] + punkty[454]) / 2
        skret = float((punkty[1][0] - srodek_twarzy[0]) / max(skala / 2, 1)) * 45
        oczy_y = (punkty[33][1] + punkty[263][1]) / 2
        kiw = float((punkty[1][1] - oczy_y) / max(punkty[152][1] - oczy_y, 1)) * 60
    skret, kiw = (-skret if ODWROC_SKRET else skret), (-kiw if ODWROC_KIWNIECIE else kiw)
    _fk["kiw_sr"] = kiw if _fk.get("kiw_sr") is None else 0.995 * _fk["kiw_sr"] + 0.005 * kiw
    px = float(np.clip((srodek[0] - SZER / 2) / (SZER / 2), -1, 1)) * 20
    py = float(np.clip((srodek[1] - WYS / 2) / (WYS / 2), -1, 1)) * 10
    return np.array([np.clip(skret, -MAKS_SKRET, MAKS_SKRET), np.clip(kiw - _fk["kiw_sr"], -MAKS_KIWNIECIE, MAKS_KIWNIECIE),
                     np.clip(kat, -MAKS_PRZECHYLENIE, MAKS_PRZECHYLENIE), px, py], np.float32)

def _wczytaj_foto(nazwa):
    global KP
    if nazwa not in _fk["dane"]:
        _fk["dane"][nazwa] = _przygotuj(nazwa)
    if _fk["foto"] != nazwa:
        _fk["foto"] = nazwa
        KP = FOTY[nazwa]
        _fk.update(_fk["dane"][nazwa])
    if "poza" not in _fk:
        _fk.update(poza=np.zeros(5, np.float32), predkosc=np.zeros(5, np.float32), oczy=np.zeros(2, np.float32),
                   pysk=0.0, zadow=0.0, zdziw=0.0,
                   nast_mrug=time.time() + 2, ucho=(0.0, 0, 0.0), nast_ucho=time.time() + 4)

def podpis_zdjecia(obraz, foto=None):
    """Jeśli znamy autora zdjęcia, podpis trafia też do nagrania."""
    zrodlo = FOTY.get(foto or _fk["foto"] or "", {}).get("zrodlo", "")
    for i, linia in enumerate(l for l in zrodlo.split("|") if l):
        napis = render_napisu(linia, 11, BIALY, obrys=2)
        wklej(obraz, napis, SZER - napis.shape[1] - 14, 74 + 15 * i)

_tlo_domyslne = {}

def kapibara_3d(punkty, kapibara, wygladzanie=0.6, widok="zdjecie", foto=None, tlo=None):
    """Zwraca klatkę z kapibarą 3D. kapibara = słownik z mapuj_mine() (ten sam co dla rysunku)."""
    _wczytaj_foto(foto or _fk["foto"] or (dostepne_foty() or list(FOTY))[0])
    t = time.time()
    kapibara = kapibara or {}
    if punkty is not None:
        cel_pozy = _poza_z_twarzy(punkty)
        cel_oczu = np.array([float(kapibara.get("oko_lewe", 0)), float(kapibara.get("oko_prawe", 0))], np.float32)
    else:
        cel_pozy = np.array([12 * math.sin(t * 0.45), 4 * math.sin(t * 0.7), 3 * math.sin(t * 0.33), 0, 0], np.float32)
        if t > _fk["nast_mrug"]:
            _fk["mrug"] = t
            _fk["nast_mrug"] = t + np.random.uniform(2.5, 6)
        cel_oczu = np.full(2, 1.0 if t - _fk.get("mrug", 0) < 0.15 else 0.0, np.float32)
    dt = float(np.clip(t - _fk.get("t_poprz", t - 0.04), 0.005, 0.1))
    _fk["t_poprz"] = t
    omega, tlumienie = 5 + 15 * (1 - wygladzanie), 0.85
    for _ in range(3):
        przysp = omega ** 2 * (cel_pozy - _fk["poza"]) - 2 * tlumienie * omega * _fk["predkosc"]
        _fk["predkosc"] = _fk["predkosc"] + przysp * dt / 3
        _fk["poza"] = _fk["poza"] + _fk["predkosc"] * dt / 3
    _fk["oczy"] = 0.35 * _fk["oczy"] + 0.65 * cel_oczu
    _fk["pysk"] = 0.35 * _fk["pysk"] + 0.65 * float(kapibara.get("pysk", 0))
    _fk["zadow"] = 0.8 * _fk["zadow"] + 0.2 * min(1.0, float(kapibara.get("usmiech", 0)))
    _fk["zdziw"] = 0.8 * _fk["zdziw"] + 0.2 * min(1.0, float(kapibara.get("brwi", 0)))
    if t > _fk["nast_ucho"]:
        _fk["nast_ucho"] = t + np.random.uniform(4, 9)
        _fk["ucho"] = (0.0, int(np.random.randint(2)), t)
    _, ktore, start = _fk["ucho"]
    faza = (t - start) / 0.35
    _fk["ucho"] = (14 * math.sin(faza * 2 * math.pi) * (1 - faza) if 0 <= faza < 1 else 0.0, ktore, start)
    x0, y0, x1, y1 = _fk["roi"]
    oczy = np.clip(np.maximum(_fk["oczy"], 0.45 * _fk["zadow"]) * (1 - 0.6 * _fk["zdziw"]), 0, 1)
    SX, SY = _deformacje_lokalne(_fk["X"], _fk["Y"], _fk["pysk"], oczy, _fk["zdziw"], _fk["zadow"], _fk["ucho"])
    tekstura = _fk["tekstury"][widok].copy()
    roi = cv2.remap(tekstura, SX.astype(np.float32), SY.astype(np.float32), cv2.INTER_LINEAR,
                    borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    if widok != "bryla":
        _rysuj_pysk(roi, _fk["pysk"])
        _rysuj_rzesy(roi, KP["oko_lewe"], float(oczy[0]))
        _rysuj_rzesy(roi, KP["oko_prawe"], float(oczy[1]))
    mandarynka = kapibara.get("mandarynka", MANDARYNKA_NA_GLOWIE) and widok != "bryla"
    if mandarynka:
        if "cien" not in _fk["kompozyt"]:
            mx, my, mr = KP["mandarynka"]
            cien = np.zeros(roi.shape[:2], np.float32)
            cv2.ellipse(cien, (int(mx + 3 - x0), int(my + mr * 0.8 - y0)), (int(mr * 1.1), int(mr * 0.35)), 0, 0, 360, 1.0, -1)
            _fk["kompozyt"]["cien"] = 1 - cv2.GaussianBlur(cien, (0, 0), 4)[..., None] * 0.45
        roi[..., :3] = (roi[..., :3] * _fk["kompozyt"]["cien"]).astype(np.uint8)
    tekstura[y0:y1, x0:x1] = roi
    skret, kiw, przech, px, py = (float(v) for v in _fk["poza"])
    _fk["R"], _fk["przes"] = _macierz_obrotu(skret, kiw, przech), (px, py)
    glowa, (gx0, gy0, gx1, gy1) = _renderuj_glowe(tekstura, _fk["R"], _fk["przes"])
    if _fk["tlo_zdjecia"] is not None:
        if "statyczne" not in _fk["kompozyt"]:
            _fk["kompozyt"]["statyczne"] = _kompozyt(_fk["warstwa_dol"], _fk["tlo_zdjecia"])
        klatka = _fk["kompozyt"]["statyczne"].copy()
    else:
        if tlo is None:
            if "studio" not in _tlo_domyslne:
                _tlo_domyslne["studio"] = zrob_tlo_studia()
            tlo = _tlo_domyslne["studio"]
        klatka = tlo.copy()
    cx0, cy0, cx1, cy1 = max(gx0, 0), max(gy0, 0), min(gx1, SZER), min(gy1, WYS)
    if cx1 > cx0 and cy1 > cy0:
        klatka[cy0:cy1, cx0:cx1] = _kompozyt(glowa[cy0 - gy0:cy1 - gy0, cx0 - gx0:cx1 - gx0],
                                             klatka[cy0:cy1, cx0:cx1].astype(np.float32))
    if mandarynka:
        mx, my, mr = KP["mandarynka"]
        srodek, skala = _punkt_3d(mx, my, dz=mr * 0.3)
        m = _fk["mandarynka"]
        if abs(skala - 1) > 0.02:
            m = cv2.resize(m, None, fx=skala, fy=skala, interpolation=cv2.INTER_LINEAR)
        wklej(klatka, m, srodek[0] - m.shape[1] / 2, srodek[1] - m.shape[0] / 2)
    if _fk["warstwa_gora"] is not None:
        if "gora" not in _fk["kompozyt"]:
            ys, xs = np.nonzero(_fk["warstwa_gora"][..., 3] > 0)
            r = (xs.min(), ys.min(), xs.max() + 1, ys.max() + 1)
            w = _fk["warstwa_gora"][r[1]:r[3], r[0]:r[2]].astype(np.float32)
            _fk["kompozyt"]["gora"] = (r, w[..., :3].copy(), 1 - w[..., 3:] / 255)
        (rx0, ry0, rx1, ry1), kolor_g, przez = _fk["kompozyt"]["gora"]
        klatka[ry0:ry1, rx0:rx1] = np.clip(kolor_g + klatka[ry0:ry1, rx0:rx1] * przez, 0, 255).astype(np.uint8)
    if kapibara.get("dymek"):
        dymek(klatka, _punkt_3d(*KP["dymek"])[0], str(kapibara["dymek"]), 1.0)
    return klatka

kapibara_foto = kapibara_3d

print(f"✅ Silnik, część 4 gotowa. Zdjęcia kapibar w folderze {FOLDER_ZDJEC}/:", ", ".join(dostepne_foty()) or f"brak – skopiuj je do {FOLDER_ZDJEC}/")
# ▶ 0.7 SILNIK, część 5: sowa 3D (puchacz). Uruchom — nie musisz tego czytać.
SOWA_MAKS_SKRET, SOWA_MAKS_KIWNIECIE, SOWA_MAKS_PRZECHYLENIE = 135, 35, 50   # sowa obraca głowę o 135° w każdą stronę
SOWA_WZMOCNIENIE = (3.0, 1.2, 1.8)            # Twój skręt / kiwnięcie / przechylenie × tyle = ruch głowy sowy
SOWA_SRODEK = (427, 158, 18)                  # środek głowy w kadrze (x, y, z)
SOWA_GLOWA_SKALA = 1.15                       # głowa puchacza jest duża w stosunku do tułowia
SOWA_SWIATLO = np.array([-0.45, -0.55, 0.7], np.float32)
SOWA_SWIATLO /= np.linalg.norm(SOWA_SWIATLO)
_sowa = {}

def _sdf_elipsoida(X, Y, Z, srodek, promienie):
    """Przybliżona odległość od elipsoidy (ujemna w środku)."""
    px, py, pz = (X - srodek[0]) / promienie[0], (Y - srodek[1]) / promienie[1], (Z - srodek[2]) / promienie[2]
    k0 = np.sqrt(px * px + py * py + pz * pz)
    k1 = np.sqrt((px / promienie[0]) ** 2 + (py / promienie[1]) ** 2 + (pz / promienie[2]) ** 2)
    return k0 * (k0 - 1) / np.maximum(k1, 1e-6)

def _smin(a, b, k):
    """Łagodne połączenie dwóch brył (bez ostrej krawędzi)."""
    h = np.clip(k - np.abs(a - b), 0, None) / k
    return np.minimum(a, b) - h * h * k / 4

def _sdf_kula(X, Y, Z, s, r):
    return np.sqrt((X - s[0]) ** 2 + (Y - s[1]) ** 2 + (Z - s[2]) ** 2) - r

def _smax(a, b, k):
    return -_smin(-a, -b, k)

def _obroc_xy(X, Y, kat):
    c, s = math.cos(math.radians(kat)), math.sin(math.radians(kat))
    return X * c - Y * s, X * s + Y * c

def _powierzchnia(sdf, zakres, krok):
    """Siatka punktów 3D → punkty tuż przy powierzchni bryły i normalne (kierunek „na zewnątrz”)."""
    xs, ys, zs = (np.arange(a, b, krok, dtype=np.float32) for a, b in zakres)
    Y, X, Z = np.meshgrid(ys, xs, zs, indexing="ij")
    d = sdf(X, Y, Z)
    w = d < 0
    brzeg = np.zeros_like(w)
    for os_ in range(3):
        for przes in (1, -1):
            brzeg |= w & ~np.roll(w, przes, axis=os_)
    iy, ix, iz = np.nonzero(brzeg)
    gy, gx, gz = np.gradient(d, krok)
    n = np.stack([gx[iy, ix, iz], gy[iy, ix, iz], gz[iy, ix, iz]], 1)
    n /= np.linalg.norm(n, axis=1, keepdims=True) + 1e-6
    p = np.stack([xs[ix], ys[iy], zs[iz]], 1) - n * d[iy, ix, iz][:, None]
    return p.astype(np.float32), n.astype(np.float32)

OCZY_SOWY = ((-40, -6, 49), (40, -6, 49))      # środki gałek ocznych
PROMIEN_OKA = 23

def _sdf_glowa(X, Y, Z):
    d = _sdf_elipsoida(X, Y, Z, (0, 0, 0), (100, 86, 90))
    d = _smax(d, Z - 58, 18)
    for s in (-1, 1):
        d = _smax(d, -_sdf_kula(X, Y, Z, (s * 40, -4, 98), 50), 12)
    for s in (-1, 1):
        xr, yr = _obroc_xy(X - s * 54, Y + 76, -s * 22)
        d = _smin(d, _sdf_elipsoida(xr, yr, Z - 8, (0, 0, 0), (13, 34, 9)), 14)
    return d

def _sdf_dziob_gorny(X, Y, Z):
    yr, zr = _obroc_xy(Y - 14, Z - 62, -28)
    return _smin(_sdf_elipsoida(X, yr, zr, (0, 0, 0), (10, 18, 11)),
                 _sdf_elipsoida(X, Y, Z, (0, 30, 64), (5, 6, 5)), 6)

def _sdf_dziob_dolny(X, Y, Z):
    return _sdf_elipsoida(X, Y, Z, (0, 30, 55), (8, 6, 9))

def _sdf_tulow(X, Y, Z):
    cx = SOWA_SRODEK[0]
    d = _sdf_elipsoida(X, Y, Z, (cx, 352, -5), (118, 142, 92))
    d = _smin(d, _sdf_elipsoida(X, Y, Z, (cx, 250, 0), (100, 55, 82)), 40)
    for s in (-1, 1):
        xr, yr = _obroc_xy(X - (cx + s * 104), Y - 352, s * 6)
        d = _smin(d, _sdf_elipsoida(xr, yr, Z + 15, (0, 0, 0), (40, 128, 82)), 12)
    galaz = np.sqrt((Y - 470) ** 2 + ((Z - 5) * 1.1) ** 2) - 16
    return np.minimum(d, galaz)
def _sowa_szum(P, skala, ziarno=0):
    """Gładki szum 3D (0..1) – z niego robimy pióra, pręgi i kreski."""
    q = P / np.asarray(skala, np.float32)
    i0 = np.floor(q)
    t = q - i0
    t = t * t * (3 - 2 * t)
    def los(ix, iy, iz):
        v = np.sin(ix * 12.9898 + iy * 78.233 + iz * 37.719 + ziarno * 4.123) * 43758.5453
        return v - np.floor(v)
    wynik = 0
    for dx in (0, 1):
        for dy in (0, 1):
            for dz in (0, 1):
                w = (t[:, 0] if dx else 1 - t[:, 0]) * (t[:, 1] if dy else 1 - t[:, 1]) * (t[:, 2] if dz else 1 - t[:, 2])
                wynik = wynik + w * los(i0[:, 0] + dx, i0[:, 1] + dy, i0[:, 2] + dz)
    return wynik.astype(np.float32)

def _kula_punktow(srodek, r, gestosc=1.3):
    """Punkty równomiernie na kuli (spirala Fibonacciego) i ich normalne."""
    n = max(int(4 * math.pi * r * r / gestosc ** 2), 300)
    k = np.arange(n) + 0.5
    fi = np.arccos(1 - 2 * k / n)
    th = math.pi * (1 + 5 ** 0.5) * k
    nn = np.stack([np.cos(th) * np.sin(fi), np.cos(fi), np.sin(th) * np.sin(fi)], 1).astype(np.float32)
    return np.asarray(srodek, np.float32) + r * nn, nn

def _kolory_glowy(P, N):
    """Upierzenie głowy puchacza: rdzawe pióra z czarnymi kreskami, jasna tarcza twarzowa z ciemnym obrzeżem,"""
    x, y, z = P[:, 0], P[:, 1], P[:, 2]
    kol = np.empty((len(P), 3), np.float32)
    kol[:] = (42, 84, 136)
    kol *= (0.82 + 0.32 * _sowa_szum(P, (9, 9, 9), 1))[:, None]
    kreski = _sowa_szum(P, (2.2, 11, 2.2), 2)
    kol *= (1 - 0.6 * _ss((kreski - 0.66) / 0.14))[:, None]
    kol *= (1 - 0.25 * np.clip((-y - 20) / 60, 0, 1))[:, None]
    for ex, ey, _ in OCZY_SOWY:
        r = np.hypot(x - ex, y - ey)
        kat = np.arctan2(y - ey, x - ex)
        przod = _ss((z - 15) / 20)
        promyki = _sowa_szum(np.stack([kat * 9, r / 4, z * 0], 1), (1, 1, 1), 3)
        tarcza = np.array([74, 124, 172], np.float32) * (0.8 + 0.35 * promyki[:, None])
        w = przod * (1 - _ss((r - 40) / 6))
        kol = kol * (1 - w[:, None]) + tarcza * w[:, None]
        obrzeze = przod * np.clip(1 - np.abs(r - 49) / 6, 0, 1) * (0.7 + 0.3 * promyki)
        kol = kol * (1 - obrzeze[:, None]) + np.array([18, 24, 34], np.float32) * obrzeze[:, None]
    brwi = _ss((z - 30) / 15) * np.clip(1 - np.abs(y + 26 + 0.3 * np.abs(x)) / 7, 0, 1) * (np.abs(x) < 52)
    kol = kol * (1 - brwi[:, None]) + np.array([165, 195, 215], np.float32) * brwi[:, None]
    ksztalt = np.hypot(x / 30, (y - 54) / 17) + 0.25 * (_sowa_szum(P, (3, 6, 3), 11) - 0.5)
    broda = _ss((z - 25) / 15) * (1 - _ss((ksztalt - 0.75) / 0.3)) * _ss((y - 34) / 6)
    kol = kol * (1 - broda[:, None]) + np.array([185, 200, 212], np.float32) * broda[:, None]
    pedzelki = _ss((-y - 78) / 10)
    kol = kol * (1 - 0.45 * pedzelki[:, None])
    return kol

def _kolory_oka(nn, b, kolor_powieki, szerokosc=1.0):
    """Oko puchacza: pomarańczowa tęczówka z promykami, czarna źrenica; powieka z piór schodzi od góry (b = 0..1)."""
    kat = np.arccos(np.clip(nn[:, 2], -1, 1))
    kol = np.zeros((len(nn), 3), np.float32) + (10, 10, 12)
    promyki = 0.8 + 0.3 * np.sin(np.arctan2(nn[:, 1], nn[:, 0]) * 23) ** 2
    teczowka = (kat < 0.78 * szerokosc)
    kol[teczowka] = np.array([0, 105, 245], np.float32) * promyki[teczowka, None] * (0.75 + 0.25 * (kat[teczowka] / 0.78))[:, None]
    kol[kat < 0.36 * szerokosc] = (6, 6, 8)
    if b > 0.02:
        krawedz = -1 + 2.15 * b
        kol[nn[:, 1] < krawedz] = kolor_powieki
        kol[np.abs(nn[:, 1] - krawedz) < 0.12] = kolor_powieki * 0.3
    return kol

def _kolory_tulowia(P, N, czesc):
    """Pierś: płowa z czarnymi kreskami i drobnymi prążkami; skrzydła: ciemne z jaśniejszymi pręgami; gałąź: kora."""
    kol = np.empty((len(P), 3), np.float32)
    piers = np.array([72, 126, 172], np.float32) * (0.85 + 0.3 * _sowa_szum(P, (10, 10, 10), 4))[:, None]
    kreski = _sowa_szum(P, (3.5, 24, 3.5), 5)
    gora = 1 - _ss((P[:, 1] - 300) / 120)
    piers *= (1 - 0.6 * _ss((kreski - (0.72 - 0.08 * gora)) / 0.12))[:, None]
    prazki = 0.92 + 0.08 * np.sin(P[:, 1] / 3.2 + 3 * _sowa_szum(P, (12, 12, 12), 6))
    piers *= prazki[:, None]
    skrzydlo = np.array([34, 62, 102], np.float32) * (0.8 + 0.35 * _sowa_szum(P, (6, 6, 6), 7))[:, None]
    pregi = np.sin(P[:, 1] / 7 + P[:, 2] / 25 + 2 * _sowa_szum(P, (15, 15, 15), 8))
    skrzydlo *= (1 + 0.55 * _ss((pregi - 0.45) / 0.3))[:, None]
    skrzydlo *= (1 - 0.5 * _ss((_sowa_szum(P, (3.5, 11, 3.5), 9) - 0.72) / 0.12))[:, None]
    kora = np.array([38, 52, 66], np.float32) * (0.7 + 0.5 * _sowa_szum(P, (40, 2.5, 2.5), 10))[:, None]
    kol[czesc == 0] = piers[czesc == 0]
    kol[czesc == 1] = skrzydlo[czesc == 1]
    kol[czesc == 2] = kora[czesc == 2]
    return kol

def _oswietl(kol, n, sila_rim=1.0):
    """Lampa z lewej-góry-przodu + słabe światło rozproszone + chłodna poświata na krawędziach (jak nocą)."""
    jasnosc = 0.38 + 0.75 * np.clip(n @ SOWA_SWIATLO, 0, 1)
    rim = np.clip(1 - np.abs(n[:, 2]), 0, 1) ** 3 * sila_rim
    return np.clip(kol * jasnosc[:, None] + rim[:, None] * np.array([70, 45, 25], np.float32), 0, 255)

def _siatka_na_punktach(P, kol):
    linie = np.zeros(len(P), np.float32)
    for os_ in range(3):
        u = P[:, os_] / 14.0
        linie = np.maximum(linie, np.clip(1 - np.abs(u - np.round(u)) * 14 / 1.2, 0, 1))
    return kol * (1 - 0.65 * linie[:, None]) + 255 * 0.65 * linie[:, None]

def _rozprysk(qx, qy, qz, kolory, ramka=None):
    """Punkty → obraz: każdy punkt na 3 × 3 piksele, od najdalszego do najbliższego (bufor głębokości),"""
    if ramka is None:
        ramka = (int(np.floor(qx.min())) - 3, int(np.floor(qy.min())) - 3, int(np.floor(qx.max())) + 4, int(np.floor(qy.max())) + 4)
    bx0, by0, bx1, by1 = ramka
    bw, bh = bx1 - bx0, by1 - by0
    kolejnosc = np.argsort(np.clip(qz * 40, -32000, 32000).astype(np.int16), kind="stable")
    lin = (np.rint(qy[kolejnosc]).astype(np.int32) - by0) * bw + np.rint(qx[kolejnosc]).astype(np.int32) - bx0
    sasiedzi = np.array([-bw - 1, -bw, -bw + 1, -1, 0, 1, bw - 1, bw, bw + 1], np.int32)
    lin = (lin[:, None] + sasiedzi).ravel()
    ok = (lin >= 0) & (lin < bw * bh)
    bufor = np.full(bw * bh, -1, np.int32)
    bufor[lin[ok]] = np.repeat(kolejnosc, 9)[ok]
    bufor = bufor.reshape(bh, bw)
    for _ in range(2):
        jest = bufor >= 0
        dziury = ~jest & (cv2.boxFilter(jest.astype(np.uint8), -1, (3, 3), normalize=False) >= 4)
        if not dziury.any():
            break
        for dy, dx in ((0, 1), (0, -1), (1, 0), (-1, 0)):
            sasiad = np.roll(np.roll(bufor, dy, 0), dx, 1)
            wez = dziury & (sasiad >= 0)
            bufor[wez] = sasiad[wez]
            dziury &= ~wez
    jest = bufor >= 0
    nr = np.where(jest, bufor, 0)
    obraz = kolory[nr].astype(np.uint8)
    glebia = np.where(jest, qz[nr], -1e9).astype(np.float32)
    return obraz, glebia, jest, ramka

def _puszysty_brzeg(jest, ramka, ziarno=5):
    """Na obrysie część pikseli robi się półprzezroczysta – brzeg wygląda jak pióra, a nie jak plastik."""
    rng = np.random.default_rng(ziarno)
    szum = cv2.GaussianBlur(rng.random(jest.shape, np.float32), (0, 0), 0.8)
    brzeg = jest & ~(cv2.erode(jest.astype(np.uint8), np.ones((5, 5), np.uint8)) > 0)
    alfa = jest.astype(np.float32)
    alfa[brzeg] = np.clip((szum[brzeg] - 0.35) * 4, 0, 1)
    return alfa

def _przygotuj_sowe():
    """Jednorazowo: bryły, kolory, tułów narysowany na stałe."""
    cx = SOWA_SRODEK[0]
    Pg, Ng = _powierzchnia(_sdf_glowa, ((-118, 118), (-128, 100), (-100, 80)), 2.0)
    Pdg, Ndg = _powierzchnia(_sdf_dziob_gorny, ((-15, 15), (-8, 44), (40, 82)), 1.0)
    Pdd, Ndd = _powierzchnia(_sdf_dziob_dolny, ((-12, 12), (20, 40), (42, 68)), 1.0)
    Pj, Nj = _kula_punktow((0, 27, 46), 11)
    oczy = [_kula_punktow(s, PROMIEN_OKA) for s in OCZY_SOWY]
    czesci = [Pg, Pdg, Pdd, Pj] + [o[0] for o in oczy]
    normalne = [Ng, Ndg, Ndd, Nj] + [o[1] for o in oczy]
    rodzaj = np.concatenate([np.full(len(p), i) for i, p in enumerate(czesci)])
    P = np.concatenate(czesci)
    N = np.concatenate(normalne)
    kol_glowy = _kolory_glowy(Pg, Ng)
    kol = np.zeros((len(P), 3), np.float32)
    kol[rodzaj == 0] = kol_glowy
    for r, jasny in ((1, (95, 100, 105)), (2, (80, 85, 92))):
        p = P[rodzaj == r]
        t = np.clip((p[:, 1] - 10) / 30, 0, 1)[:, None]
        kol[rodzaj == r] = np.array([30, 30, 36], np.float32) * (1 - t) + np.array(jasny, np.float32) * t
    kol[rodzaj == 3] = (25, 22, 70)
    kolor_powieki = np.array([62, 115, 172], np.float32)
    Pt, Nt = _powierzchnia(_sdf_tulow, ((cx - 250, cx + 250), (185, 500), (-110, 100)), 2.5)
    galaz = np.abs(np.sqrt((Pt[:, 1] - 470) ** 2 + ((Pt[:, 2] - 5) * 1.1) ** 2) - 16) < 3
    skrzydlo = np.zeros(len(Pt), bool)
    for s in (-1, 1):
        xr, yr = _obroc_xy(Pt[:, 0] - (cx + s * 104), Pt[:, 1] - 352, s * 6)
        skrzydlo |= _sdf_elipsoida(xr, yr, Pt[:, 2] + 15, (0, 0, 0), (40, 128, 82)) < 2.5
    czesc = np.where(galaz, 2, np.where(skrzydlo, 1, 0))
    kol_t = _kolory_tulowia(Pt, Nt, czesc)
    tulow = {}
    for widok in ("piora", "siatka", "bryla"):
        k = kol_t if widok == "piora" else (_siatka_na_punktach(Pt, kol_t) if widok == "siatka"
                                             else np.full_like(kol_t, 200))
        obraz, glebia, jest, ramka = _rozprysk(Pt[:, 0], Pt[:, 1], Pt[:, 2], _oswietl(k, Nt), (0, 0, SZER, WYS))
        if widok == "piora":
            obraz = np.where(jest[..., None], cv2.GaussianBlur(obraz, (0, 0), 0.7), obraz)
        tulow[widok] = (obraz, glebia, _puszysty_brzeg(jest, ramka, 11))
    _sowa.update(P=P * SOWA_GLOWA_SKALA, N=N, rodzaj=rodzaj, kol=kol, kolor_powieki=kolor_powieki,
                 nn_oczu=[o[1] for o in oczy], tulow=tulow, mandarynka=_zrob_mandarynke(24),
                 poza=np.zeros(5, np.float32), predkosc=np.zeros(5, np.float32), b=np.zeros(2, np.float32),
                 dziob=0.0, zdziw=0.0, zadow=0.0, nast_mrug=time.time() + 2, kompozyt={})

def _sowa_obroc(P, skret, kiw, przech):
    """Obrót głowy wokół szyi (punkt obrotu pod środkiem głowy)."""
    piv = np.array([0, 60, -15], np.float32)
    Q = P - piv
    a, b, c = (math.radians(v) for v in (skret, kiw, przech))
    x, y, z = Q[:, 0], Q[:, 1], Q[:, 2]
    x, z = x * math.cos(a) + z * math.sin(a), -x * math.sin(a) + z * math.cos(a)
    y, z = y * math.cos(b) + z * math.sin(b), -y * math.sin(b) + z * math.cos(b)
    x, y = x * math.cos(c) - y * math.sin(c), x * math.sin(c) + y * math.cos(c)
    return np.stack([x, y, z], 1) + piv

def _sowa_poza_z_twarzy(punkty):
    """Twoje kąty głowy (z punktów 3D twarzy), wzmocnione – sowa obraca głowę dużo dalej niż człowiek."""
    srodek, kat, skala = geometria_twarzy(punkty)
    p3 = globals().get("OSTATNIE_PUNKTY_3D")
    if p3 is not None and len(p3) == len(punkty):
        prawo, gora = p3[454] - p3[234], p3[10] - p3[152]
        n = np.cross(prawo, gora)
        n /= max(np.linalg.norm(n), 1e-6)
        if np.dot(n, p3[1] - (p3[234] + p3[454]) / 2) < 0:
            n = -n
        skret, kiw = math.degrees(math.atan2(n[0], -n[2])), math.degrees(math.atan2(n[1], -n[2]))
    else:
        srodek_twarzy = (punkty[234] + punkty[454]) / 2
        skret = float((punkty[1][0] - srodek_twarzy[0]) / max(skala / 2, 1)) * 45
        oczy_y = (punkty[33][1] + punkty[263][1]) / 2
        kiw = float((punkty[1][1] - oczy_y) / max(punkty[152][1] - oczy_y, 1)) * 60
    if globals().get("ODWROC_SKRET"):
        skret = -skret
    if globals().get("ODWROC_KIWNIECIE"):
        kiw = -kiw
    _sowa["kiw_sr"] = kiw if _sowa.get("kiw_sr") is None else 0.995 * _sowa["kiw_sr"] + 0.005 * kiw
    ws, wk, wp = SOWA_WZMOCNIENIE
    px = float(np.clip((srodek[0] - SZER / 2) / (SZER / 2), -1, 1)) * 15
    py = float(np.clip((srodek[1] - WYS / 2) / (WYS / 2), -1, 1)) * 8
    return np.array([np.clip(skret * ws, -SOWA_MAKS_SKRET, SOWA_MAKS_SKRET),
                     np.clip((kiw - _sowa["kiw_sr"]) * wk, -SOWA_MAKS_KIWNIECIE, SOWA_MAKS_KIWNIECIE),
                     np.clip(kat * wp, -SOWA_MAKS_PRZECHYLENIE, SOWA_MAKS_PRZECHYLENIE), px, py], np.float32)

def zrob_tlo_lasu():
    """Nocny las: granatowe niebo z gradientem, księżyc, sylwetki drzew."""
    yy, xx = np.mgrid[0:WYS, 0:SZER].astype(np.float32)
    tlo = np.dstack([70 + 50 * (1 - yy / WYS), 30 + 30 * (1 - yy / WYS), 15 + 15 * (1 - yy / WYS)])
    d = np.hypot(xx - 700, yy - 90)
    tlo += (np.clip(1 - d / 160, 0, 1) ** 2 * 60)[..., None]
    tlo[d < 38] = (200, 225, 235)
    rng = np.random.default_rng(3)
    for _ in range(90):
        x, y = rng.integers(0, SZER), rng.integers(0, WYS // 2)
        if np.hypot(x - 700, y - 90) > 60:
            tlo[y, x] = (200, 200, 210)
    for x0, sz, wys in ((40, 70, 330), (150, 50, 260), (640, 80, 300), (790, 60, 360), (260, 40, 200), (560, 45, 220)):
        pts = np.array([[x0, WYS], [x0 + sz, WYS - wys], [x0 + 2 * sz, WYS]], np.int32)
        cv2.fillPoly(tlo, [pts], (40, 32, 22))
    return cv2.GaussianBlur(np.clip(tlo, 0, 255).astype(np.uint8), (0, 0), 1.2)

def sowa_3d(punkty, kapibara, wygladzanie=0.6, widok="piora", tlo=None):
    """Zwraca klatkę z sową 3D. kapibara = słownik z mapuj_mine() – sowa reaguje na te same miny."""
    if "P" not in _sowa:
        _przygotuj_sowe()
    s = _sowa
    t = time.time()
    kapibara = kapibara or {}
    if punkty is not None:
        cel = _sowa_poza_z_twarzy(punkty)
        cel_b = np.array([float(kapibara.get("oko_lewe", 0)), float(kapibara.get("oko_prawe", 0))], np.float32)
    else:
        faza = (t * 0.15) % 1
        cel = np.array([120 * math.sin(faza * 2 * math.pi) if faza < 0.5 else 0, 6 * math.sin(t * 0.5),
                        25 * math.sin(t * 0.23), 0, 0], np.float32)
        if t > s["nast_mrug"]:
            s["mrug"], s["nast_mrug"] = t, t + np.random.uniform(2.5, 6)
        cel_b = np.full(2, 1.0 if t - s.get("mrug", 0) < 0.25 else 0.0, np.float32)
    dt = float(np.clip(t - s.get("t_poprz", t - 0.04), 0.005, 0.1))
    s["t_poprz"] = t
    omega, tlumienie = 5 + 15 * (1 - wygladzanie), 0.8
    for _ in range(3):
        przysp = omega ** 2 * (cel - s["poza"]) - 2 * tlumienie * omega * s["predkosc"]
        s["predkosc"] = s["predkosc"] + przysp * dt / 3
        s["poza"] = s["poza"] + s["predkosc"] * dt / 3
    s["b"] = 0.5 * s["b"] + 0.5 * cel_b
    s["dziob"] = 0.4 * s["dziob"] + 0.6 * float(kapibara.get("pysk", 0))
    s["zdziw"] = 0.8 * s["zdziw"] + 0.2 * min(1.0, float(kapibara.get("brwi", 0)))
    s["zadow"] = 0.8 * s["zadow"] + 0.2 * min(1.0, float(kapibara.get("usmiech", 0)))
    P = s["P"].copy()
    r = s["rodzaj"]
    if s["dziob"] > 0.01:
        dol = (r == 2) | (r == 3)
        kat = math.radians(32 * s["dziob"])
        zy, zz = 22 * SOWA_GLOWA_SKALA, 44 * SOWA_GLOWA_SKALA
        y, z = P[dol, 1] - zy, P[dol, 2] - zz
        P[dol, 1], P[dol, 2] = zy + y * math.cos(kat) + z * math.sin(kat), zz - y * math.sin(kat) + z * math.cos(kat)
    if s["zdziw"] > 0.01:
        g = SOWA_GLOWA_SKALA
        pedzelki = (r == 0) & (P[:, 1] < -70 * g)
        P[pedzelki, 1] -= 9 * s["zdziw"] * _ss((-P[pedzelki, 1] - 70 * g) / 30)
        for i, srodek_oka in enumerate(OCZY_SOWY):
            oko = r == 4 + i
            c = np.array(srodek_oka, np.float32) * g
            P[oko] = c + (P[oko] - c) * (1 + 0.1 * s["zdziw"])
    kol = s["kol"].copy() if widok != "bryla" else np.full_like(s["kol"], 200)
    zamkniecie = np.clip(np.maximum(s["b"], 0.5 * s["zadow"]) * (1 - 0.7 * s["zdziw"]), 0, 1)
    if widok != "bryla":
        for i in (0, 1):
            kol[r == 4 + i] = _kolory_oka(s["nn_oczu"][i], float(zamkniecie[i]), s["kolor_powieki"])
    if widok == "siatka":
        kol = _siatka_na_punktach(s["P"], kol)
    skret, kiw, przech, px, py = (float(v) for v in s["poza"])
    Q = _sowa_obroc(P, skret, kiw, przech)
    Nr = _sowa_obroc(s["N"], skret, kiw, przech) - _sowa_obroc(np.zeros((1, 3), np.float32), skret, kiw, przech)
    kol = _oswietl(kol, Nr)
    if widok != "bryla":
        oczy = r >= 4
        odbicie = 2 * (Nr[oczy] @ SOWA_SWIATLO)[:, None] * Nr[oczy] - SOWA_SWIATLO
        polysk = np.clip(odbicie[:, 2], 0, 1) ** 40 * (zamkniecie[(r[oczy] - 4)] < 0.6)
        kol[oczy] = np.clip(kol[oczy] + polysk[:, None] * 255, 0, 255)
    sx, sy, sz = SOWA_SRODEK
    glowa, glebia, jest, (bx0, by0, bx1, by1) = _rozprysk(Q[:, 0] + sx + px, Q[:, 1] + sy + py, Q[:, 2] + sz, kol)
    alfa = _puszysty_brzeg(jest, (bx0, by0, bx1, by1))
    if tlo is None:
        if "las" not in s["kompozyt"]:
            s["kompozyt"]["las"] = zrob_tlo_lasu()
        tlo = s["kompozyt"]["las"]
    klucz = (id(tlo), widok)
    if klucz not in s["kompozyt"]:
        if len(s["kompozyt"]) > 12:
            s["kompozyt"] = {"las": s["kompozyt"]["las"]} if "las" in s["kompozyt"] else {}
        obraz_t, glebia_t, alfa_t = s["tulow"][widok]
        s["kompozyt"][klucz] = (tlo * (1 - alfa_t[..., None]) + obraz_t * alfa_t[..., None]).astype(np.uint8)
    klatka = s["kompozyt"][klucz].copy()
    glebia_t = s["tulow"][widok][1]
    cx0, cy0, cx1, cy1 = max(bx0, 0), max(by0, 0), min(bx1, SZER), min(by1, WYS)
    if cx1 > cx0 and cy1 > cy0:
        w = (slice(cy0 - by0, cy1 - by0), slice(cx0 - bx0, cx1 - bx0))
        widac = (glebia[w] > glebia_t[cy0:cy1, cx0:cx1]).astype(np.float32) * alfa[w]
        obszar = klatka[cy0:cy1, cx0:cx1]
        obszar[:] = (obszar * (1 - widac[..., None]) + glowa[w] * widac[..., None]).astype(np.uint8)
    if kapibara.get("mandarynka", MANDARYNKA_NA_GLOWIE) and widok != "bryla":
        czubek = _sowa_obroc(np.array([[0, -88 * SOWA_GLOWA_SKALA, -5]], np.float32), skret, kiw, przech)[0]
        m = s["mandarynka"]
        wklej(klatka, m, czubek[0] + sx + px - m.shape[1] / 2, czubek[1] + sy + py - m.shape[0] * 0.95)
    if kapibara.get("dymek"):
        bok = _sowa_obroc(np.array([[95 * SOWA_GLOWA_SKALA, -60, 0]], np.float32), skret, kiw, przech)[0]
        dymek(klatka, np.array([bok[0] + sx + px, bok[1] + sy + py - 30], np.float32), str(kapibara["dymek"]), 1.0)
    return klatka

print("✅ Silnik, część 5 gotowa: sowa 3D")

# ▶ 0.8 SILNIK, część 6: zamiana twarzy i detektyw AI. Uruchom — nie musisz tego czytać.
import base64, zlib
from collections import Counter, defaultdict, deque

_ZNAKI = "SZasYs6g9llVTqyoMVbZZTTtSo2muBPiW9pmwM5UTflad0MT+VR/9FV68AZZRNZ2l5UFrrIT7lkXJY1JRT51STTWWUrVyWY/YO0hLMn5jV2QNoxF52A1XAC6qFD/NNdlMloavpRT4s3R5W4Aef8M"   # która strona każdego trójkąta patrzy „na zewnątrz” twarzy
_LUSTRO = "eNoFwQOAEAAAwMAt29Zn27Zt27Zt27Zt27Zt2667wHwnOD+Iwk+i8oto/CY6f4jBX2Lyj1hgbDQOgYxLYOMRxACCGp9gJiC4CQlhIkKamFAmIbRJCWMywpqccKYgvCmJYCoimppIpiGyaYliOqKanmhmILoZiWEmYpqZWGYhtlmJYzbimp145iDAnMQ3FwnMTULzkMi8JDYfScxPUguQzIIktxApLExKi5DKoqS2GGksTlpLkM6SpLcUGSxNRsuQybJkthxZLE9WK5DNimS3EjmsTE6rkMuq5LYaeaxOXmuQz5rktxYFrE1B61DIehS2PkVsQFEbUsxGFLcxJWxCSZtSymaUtjllbEFZW1LOVpS3NRVsQ0XbUsl2VLY9VexAVTtSzU5UtzM17EJNu1LLbtS2O3XsQV17Us9e1Lc3DexDQ/vSyH40tj9NHEBTB9LMQTR3MC0cQkuH0sphtHY4bRxBW0fSzlG0dzQdHENHx9LJcXR2PF2cQFcn0s1JdHcyPZxCT6fSyxn0diZ9nEVfZ9PPOfR3LgOcx0DnM8gFDHYhQ1zEUJcwzKUMdxkjXMFIVzLKVYx2NWNcw1jXMs71jHcDE9zIRDcxyc1MdgtT3MpUtzHN7Ux3BzPcyUx3McvdzHYPc9zLXPcxz/3M9wALPMhCD7PIoyz2BEs8yVJPsczTLPcMKzzLSs+xyvOs9gJrvMhaL7HOy6z3Chu8ykavscnrbPYGW7zJVm+xzdts9w47vMtO77HL++z2AXt8yF4fsc/H7PcJB3zKQZ9xyOcc9gVHfMlRX3HM1xz3DSd8y0nfccr3nPYDZ/zIWT9xzs+c9wsX/MpFv3HJ/8K40GA="   # pary punktów lewa ↔ prawa strona twarzy (z modelu kanonicznego MediaPipe)

def _landmarker2():
    if "twarze2" not in _modele:
        opcje = vision.FaceLandmarkerOptions(
            base_options=BaseOptions(model_asset_path=_sprawdz_plik(SCIEZKA_TWARZ)),
            running_mode=vision.RunningMode.VIDEO, num_faces=2)
        _modele["twarze2"] = vision.FaceLandmarker.create_from_options(opcje)
    return _modele["twarze2"]

def wykryj_twarze(klatka):
    """Do dwóch twarzy naraz. Zwraca listę tablic 478 × 2 (od lewej do prawej strony kadru)."""
    wynik = _landmarker2().detect_for_video(_do_mp(klatka), znacznik_ms())
    h, w = klatka.shape[:2]
    twarze = [np.array([(p.x * w, p.y * h) for p in lm], np.float32) for lm in wynik.face_landmarks]
    return sorted(twarze, key=lambda p: float(p[:, 0].mean()))

def _siatka():
    """Siatka twarzy MediaPipe: ok. 850 trójkątów, „wachlarze” oczu wokół tęczówek, brzeg ust i odbicie lewo–prawo."""
    F = vision.FaceLandmarksConnections
    sasiedzi = defaultdict(set)
    for c in F.FACE_LANDMARKS_TESSELATION:
        sasiedzi[c.start].add(c.end)
        sasiedzi[c.end].add(c.start)
    trojkaty = sorted({tuple(sorted((a, b, c))) for a in sasiedzi for b in sasiedzi[a] for c in sasiedzi[a] & sasiedzi[b]})
    ile = Counter(k for t in trojkaty for k in ((t[0], t[1]), (t[0], t[2]), (t[1], t[2])))
    brzeg = [k for k, n in ile.items() if n == 1]

    def punkty(polaczenia):
        return {c.start for c in polaczenia} | {c.end for c in polaczenia}
    pary = np.frombuffer(zlib.decompress(base64.b64decode(_LUSTRO)), np.uint16).reshape(-1, 2).astype(np.int64)
    lustro, strona = np.arange(478), np.zeros(478, np.int8)
    lustro[pary[:, 0]], lustro[pary[:, 1]] = pary[:, 1], pary[:, 0]
    strona[pary[:, 0]], strona[pary[:, 1]] = -1, 1
    oczy, srodki = [], []
    for oko, teczowka in ((F.FACE_LANDMARKS_LEFT_EYE, F.FACE_LANDMARKS_LEFT_IRIS), (F.FACE_LANDMARKS_RIGHT_EYE, F.FACE_LANDMARKS_RIGHT_IRIS)):
        p, srodek = punkty(oko), min(punkty(teczowka)) - 1
        oczy += [(a, b, srodek) for a, b in brzeg if a in p and b in p]
        strona[srodek] = strona[next(iter(p))]
        srodki.append(srodek)
    lustro[srodki[0]], lustro[srodki[1]] = srodki[1], srodki[0]
    usta = punkty(F.FACE_LANDMARKS_LIPS)
    brzeg_ust = [(a, b) for a, b in brzeg if a in usta and b in usta]
    return np.array(trojkaty, np.int32), np.array(oczy, np.int32), np.array(brzeg_ust, np.int32), lustro, strona

TROJKATY, TROJKATY_OCZU, BRZEG_UST, LUSTRO, STRONA = _siatka()
ZNAKI = np.unpackbits(np.frombuffer(base64.b64decode(_ZNAKI), np.uint8)).astype(np.float32) * 2 - 1
if len(ZNAKI) < len(TROJKATY) + len(TROJKATY_OCZU):
    ZNAKI = np.ones(len(TROJKATY) + len(TROJKATY_OCZU), np.float32)
MIESZANIE = ["ostre", "miękkie", "Poisson"]
KOLORY = ["bez zmian", "średni kolor", "kolor i światło"]

def _strona_trojkatow(trojkaty):
    s = STRONA[trojkaty].astype(np.int32)
    return np.where((s <= 0).all(1) & (s < 0).any(1), -1, np.where((s >= 0).all(1) & (s > 0).any(1), 1, 0))

def _zwrot(p, trojkaty, znaki):
    """Pole trójkątów: dodatnie, gdy trójkąt patrzy w kamerę, ujemne, gdy jest odwrócony tyłem (zasłonięty)."""
    a, b, c = (p[trojkaty[:, i]] for i in range(3))
    pole = ((b[:, 0] - a[:, 0]) * (c[:, 1] - a[:, 1]) - (b[:, 1] - a[:, 1]) * (c[:, 0] - a[:, 0])) * znaki
    return pole if pole.sum() >= 0 else -pole

def _waga_lustra(p_zr, p_cel, trojkaty, znaki):
    """Obrócona głowa: dalsza połowa twarzy źródłowej jest ściśnięta i po rozciągnięciu byłaby rozmazana."""
    zr = np.clip(_zwrot(p_zr, trojkaty, znaki), 0, None)
    cel = np.clip(_zwrot(p_cel, trojkaty, znaki), 0, None)
    skala = cel.sum() / max(zr.sum(), 1e-6)
    strona = _strona_trojkatow(trojkaty)
    waga = np.zeros(len(trojkaty))
    for s in (-1, 1):
        if zr[strona == s].sum() < zr[strona == -s].sum():
            rozciagniecie = cel[strona == s].sum() / max(zr[strona == s].sum() * skala, 1e-6)
            waga[strona == s] = np.clip((rozciagniecie - 1.35) / 0.5, 0, 1)
    return waga

def _przenies_twarz(zrodlo, p_zr, klatka, p_cel, oczy=True):
    """Każdy trójkąt twarzy z obrazu `zrodlo` rozciągamy tak, żeby pasował do trójkąta twarzy w klatce."""
    h, w = klatka.shape[:2]
    x0, y0 = (np.floor(p_cel[:468].min(0)) - 2).astype(int)
    x1, y1 = (np.ceil(p_cel[:468].max(0)) + 3).astype(int)
    x0, y0, x1, y1 = max(x0, 0), max(y0, 0), min(x1, w), min(y1, h)
    if x1 - x0 < 12 or y1 - y0 < 12:
        return None
    trojkaty = np.concatenate([TROJKATY, TROJKATY_OCZU]) if oczy else TROJKATY
    znaki = ZNAKI[:len(trojkaty)]
    lok = np.round(p_cel - (x0, y0)).astype(np.int32)
    mapa = np.zeros((y1 - y0, x1 - x0), np.uint16)
    for i in np.argsort(_zwrot(p_cel, trojkaty, znaki) > 0, kind="stable"):
        cv2.fillConvexPoly(mapa, lok[trojkaty[i]], int(i) + 1)
    dziury = np.zeros(mapa.shape, np.uint8)
    srodek_ust = np.round(lok[BRZEG_UST].reshape(-1, 2).mean(0)).astype(np.int32)
    for a, b in BRZEG_UST:
        cv2.fillConvexPoly(dziury, np.array([lok[a], lok[b], srodek_ust]), 1)
    if not oczy:
        for t in TROJKATY_OCZU:
            cv2.fillConvexPoly(dziury, lok[t], 1)
    luki = (mapa == 0) & (dziury == 0) & (cv2.dilate((mapa > 0).astype(np.uint8), np.ones((3, 3), np.uint8)) > 0)
    mapa[luki] = cv2.dilate(mapa, np.ones((3, 3), np.uint8))[luki]
    ys, xs = np.nonzero(mapa)
    if len(ys) < 50:
        return None
    D = np.concatenate([p_cel[trojkaty], np.ones((len(trojkaty), 3, 1), np.float32)], 2)
    Dp = np.linalg.pinv(D)
    k = mapa[ys, xs].astype(np.int32) - 1
    XY = np.stack([xs + x0, ys + y0, np.ones_like(xs)], 1).astype(np.float32)

    def przenies(punkty_zr):
        M = (Dp @ punkty_zr[trojkaty])[k]
        uv = np.einsum("nj,njk->nk", XY, M)
        mx, my = np.full(mapa.shape, -1, np.float32), np.full(mapa.shape, -1, np.float32)
        mx[ys, xs], my[ys, xs] = uv[:, 0], uv[:, 1]
        return cv2.remap(zrodlo, mx, my, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE).astype(np.float32)

    wycinek = przenies(p_zr)
    lustro = np.zeros(mapa.shape, np.float32)
    waga = _waga_lustra(p_zr, p_cel, trojkaty, znaki)
    if waga.max() > 0:
        lustro[ys, xs] = waga[k]
        lustro = cv2.GaussianBlur(lustro, (0, 0), 0.03 * (x1 - x0))
        wycinek += (przenies(p_zr[LUSTRO]) - wycinek) * lustro[..., None]
    twarz = cv2.morphologyEx(((mapa > 0) | (dziury > 0)).astype(np.uint8), cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    return {"wycinek": np.clip(wycinek, 0, 255).astype(np.uint8), "siatka": mapa > 0, "twarz": twarz,
            "dziury": dziury, "lustro": lustro, "ramka": (x0, y0, x1, y1)}

def dopasuj_kolor(wycinek, cel, maska):
    """Wyrównuje średni kolor i jasność przeniesionej twarzy do twarzy w klatce (w przestrzeni LAB)."""
    m = maska > 0
    if m.sum() < 50:
        return wycinek
    a = cv2.cvtColor(wycinek, cv2.COLOR_BGR2LAB).astype(np.float32)
    b = cv2.cvtColor(cel, cv2.COLOR_BGR2LAB).astype(np.float32)
    for k in range(3):
        sa, sb = a[..., k][m], b[..., k][m]
        skala = float(np.clip(sb.std() / max(sa.std(), 1e-3), 0.5, 2.0))
        a[..., k] = (a[..., k] - sa.mean()) * skala + sb.mean()
    return cv2.cvtColor(np.clip(a, 0, 255).astype(np.uint8), cv2.COLOR_LAB2BGR)

def dopasuj_swiatlo(wycinek, cel, maska, sigma):
    """Przenosi „rozmyty” kolor i światło z twarzy w klatce: oświetlona strona zostaje jasna, cień zostaje cieniem."""
    h, w = maska.shape
    f = max(1.0, sigma / 4)
    male = (max(2, int(w / f)), max(2, int(h / f)))
    e = max(3, int(sigma / 2)) | 1
    m = cv2.erode(maska.astype(np.uint8), np.ones((e, e), np.uint8)).astype(np.float32)
    waga = cv2.GaussianBlur(cv2.resize(m, male, interpolation=cv2.INTER_AREA), (0, 0), sigma / f)[..., None] + 1e-3

    def srednia(obraz):
        o = cv2.resize(obraz.astype(np.float32) * m[..., None], male, interpolation=cv2.INTER_AREA)
        return cv2.GaussianBlur(o, (0, 0), sigma / f) / waga
    iloraz = np.clip((srednia(cel) + 4) / (srednia(wycinek) + 4), 0.5, 2.0)
    return np.clip(wycinek * cv2.resize(iloraz, (w, h)), 0, 255).astype(np.uint8)

def _wtop_domyslne(twarz, tlo, alfa):
    return twarz * alfa + tlo * (1 - alfa)

def wklej_twarz(zrodlo, p_zr, klatka, p_cel, wynik, mieszanie="miękkie", kolor="kolor i światło", wtapianie=None,
                rozmycie=25, oczy=True, pokaz_maske=False):
    """Wkleja twarz z obrazu `zrodlo` (punkty p_zr) na twarz w klatce (punkty p_cel). Rysuje na `wynik`."""
    r = _przenies_twarz(zrodlo, p_zr, klatka, p_cel, oczy)
    if r is None:
        return wynik
    x0, y0, x1, y1 = r["ramka"]
    szer, wycinek, siatka, twarz, dziury = x1 - x0, r["wycinek"], r["siatka"], r["twarz"], r["dziury"]
    cel, tlo = klatka[y0:y1, x0:x1], wynik[y0:y1, x0:x1]
    if kolor == "średni kolor":
        wycinek = dopasuj_kolor(wycinek, cel, siatka)
    elif kolor == "kolor i światło":
        wycinek = dopasuj_swiatlo(wycinek, cel, siatka, 0.18 * szer)
    wycinek[~siatka] = tlo[~siatka]
    skora = twarz & (1 - dziury)
    if mieszanie == "Poisson" and not pokaz_maske:
        m = cv2.erode(skora, np.ones((5, 5), np.uint8)) * 255
        bx, by, bw, bh = cv2.boundingRect(m)
        if bw > 10 and bh > 10 and x0 + bx > 1 and y0 + by > 1 and x0 + bx + bw < wynik.shape[1] - 1 and y0 + by + bh < wynik.shape[0] - 1:
            try:
                return cv2.seamlessClone(wycinek, wynik, m, (x0 + bx + bw // 2, y0 + by + bh // 2), cv2.NORMAL_CLONE)
            except cv2.error:
                pass
        mieszanie = "miękkie"
    if mieszanie == "ostre":
        alfa = skora.astype(np.float32)
    else:
        e = max(3, szer // 30) | 1
        alfa = cv2.GaussianBlur(cv2.erode(twarz, np.ones((e, e), np.uint8)).astype(np.float32), (0, 0), szer / max(rozmycie, 1))
        alfa *= 1 - cv2.GaussianBlur(dziury.astype(np.float32), (0, 0), 1.5)
    if pokaz_maske:
        kolory = np.zeros(tlo.shape, np.float32)
        kolory[:] = (80, 200, 80)
        kolory[r["lustro"] > 0] = (230, 140, 60)
        kolory[dziury > 0] = (60, 60, 230)
        widac = np.maximum(alfa, dziury.astype(np.float32))[..., None] * 0.6
        wynik[y0:y1, x0:x1] = (tlo * (1 - widac) + kolory * widac).astype(np.uint8)
        return wynik
    polaczone = (wtapianie or _wtop_domyslne)(wycinek.astype(np.float32), tlo.astype(np.float32), alfa[..., None])
    wynik[y0:y1, x0:x1] = np.clip(polaczone, 0, 255).astype(np.uint8)
    return wynik

def zamien_twarze(klatka, twarze, zapamietana=None, **opcje):
    """Dwie twarze w kadrze: zamienia je miejscami. Jedna twarz i twarz zapamiętana klawiszem z: wkleja zapamiętaną."""
    wynik = klatka.copy()
    if len(twarze) >= 2:
        for zr, cel in ((0, 1), (1, 0)):
            wynik = wklej_twarz(klatka, twarze[zr], klatka, twarze[cel], wynik, **opcje)
    elif len(twarze) == 1 and zapamietana is not None:
        wynik = wklej_twarz(zapamietana[0], zapamietana[1], klatka, twarze[0], wynik, **opcje)
    return wynik

def wygladz_twarze(twarze, poprzednie, sila=0.6):
    """Punkty z kamery drżą o 1–2 piksele. Mieszamy je z poprzednią klatką, a przy szybkim ruchu — mniej, żeby nie było opóźnienia."""
    if sila <= 0 or len(twarze) != len(poprzednie):
        return [t.copy() for t in twarze]
    wynik = []
    for nowe, stare in zip(twarze, poprzednie):
        ruch = float(np.linalg.norm(nowe - stare, axis=1).mean()) / (float(np.ptp(nowe[:, 0])) + 1)
        a = min(sila, 0.95) * float(np.clip(1 - ruch / 0.08, 0, 1))
        wynik.append(stare * a + nowe * (1 - a))
    return wynik

def zapamietaj_twarz(klatka, twarze):
    """Kopia klatki i punktów największej twarzy. Zostaje tylko w pamięci — nic nie trafia na dysk."""
    if not twarze:
        return None
    return klatka.copy(), max(twarze, key=lambda t: float(np.ptp(t[:, 0]))).copy()

def miniatura_twarzy(obraz, zapamietana, wys=96):
    zrodlo, p = zapamietana
    x0, y0 = np.maximum(np.floor(p[:468].min(0)).astype(int) - 10, 0)
    x1, y1 = np.ceil(p[:468].max(0)).astype(int) + 10
    wycinek = zrodlo[y0:y1, x0:x1]
    if wycinek.size == 0:
        return
    szer = max(1, int(wycinek.shape[1] * wys / wycinek.shape[0]))
    x, y = SZER - szer - 12, 12
    obraz[y:y + wys, x:x + szer] = cv2.resize(wycinek, (szer, wys))
    cv2.rectangle(obraz, (x - 2, y - 2), (x + szer + 1, y + wys + 1), ZOLTY, 2)
    napisz(obraz, "zapamiętana", min(x, SZER - 105), y + wys + 4, 13, ZOLTY, obrys=2)

def rysuj_trojkaty(obraz, punkty, kolor=(255, 230, 120)):
    cv2.polylines(obraz, list(np.round(punkty[TROJKATY]).astype(np.int32)), True, kolor, 1, cv2.LINE_AA)

FOLDER_DETEKTORA = os.path.join(FOLDER_MODELI, "detektor_ai")
ROZSZERZENIA_OBRAZOW = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
ROZSZERZENIA_WIDEO = (".mp4", ".mov", ".avi", ".mkv", ".webm", ".gif")

def dopasuj_do_okna(obraz):
    """Skaluje obraz tak, żeby zmieścił się w oknie 854 × 480 (z czarnymi pasami)."""
    h, w = obraz.shape[:2]
    s = min(SZER / w, WYS / h)
    maly = cv2.resize(obraz, (max(1, int(w * s)), max(1, int(h * s))), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR)
    wynik = np.zeros((WYS, SZER, 3), np.uint8)
    y0, x0 = (WYS - maly.shape[0]) // 2, (SZER - maly.shape[1]) // 2
    wynik[y0:y0 + maly.shape[0], x0:x0 + maly.shape[1]] = maly
    return wynik

def _klatki_pliku(sciezka, max_klatek=48):
    rozsz = os.path.splitext(sciezka)[1].lower()
    if rozsz in ROZSZERZENIA_OBRAZOW:
        obraz = cv2.imread(sciezka)
        return [] if obraz is None else [obraz]
    if rozsz == ".gif":
        from PIL import Image, ImageSequence
        with Image.open(sciezka) as gif:
            klatki = [cv2.cvtColor(np.array(k.convert("RGB")), cv2.COLOR_RGB2BGR) for k in ImageSequence.Iterator(gif)]
        return klatki[::max(1, len(klatki) // max_klatek)][:max_klatek]
    wideo = cv2.VideoCapture(sciezka)
    krok = max(1, int(wideo.get(cv2.CAP_PROP_FRAME_COUNT) or 0) // max_klatek)
    klatki, nr = [], 0
    while len(klatki) < max_klatek:
        ok, k = wideo.read()
        if not ok:
            break
        if nr % krok == 0:
            klatki.append(k)
        nr += 1
    wideo.release()
    return klatki

def _zawin(tekst, rozmiar, szerokosc):
    linie, linia = [], ""
    for slowo in tekst.split():
        proba = (linia + " " + slowo).strip()
        if linia and render_napisu(proba, rozmiar).shape[1] > szerokosc:
            linie.append(linia)
            linia = slowo
        else:
            linia = proba
    return linie + ([linia] if linia else [])

URZADZENIE_DETEKTORA = None   # None = samo wybiera: karta graficzna, jeśli jest, inaczej procesor

def _wybierz_urzadzenie():
    import torch
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

def _detektor():
    global URZADZENIE_DETEKTORA
    if "detektor_ai" not in _modele:
        if not os.path.exists(os.path.join(FOLDER_DETEKTORA, "config.json")):
            raise FileNotFoundError(f"Brak modelu detektora w folderze {FOLDER_DETEKTORA}. "
                                    "Uruchom komórkę 0.2 albo skopiuj ten folder od prowadzącego.")
        try:
            from transformers import pipeline
        except ImportError:
            raise ImportError("Brak bibliotek transformers i torch. Uruchom komórkę 0.1 i zrestartuj kernel.") from None
        URZADZENIE_DETEKTORA = URZADZENIE_DETEKTORA or _wybierz_urzadzenie()
        try:
            _modele["detektor_ai"] = pipeline("image-classification", model=FOLDER_DETEKTORA, device=URZADZENIE_DETEKTORA)
        except Exception:
            if URZADZENIE_DETEKTORA == "cpu":
                raise
            URZADZENIE_DETEKTORA = "cpu"
            _modele["detektor_ai"] = pipeline("image-classification", model=FOLDER_DETEKTORA, device="cpu")
        print("🔎 Detektor liczy na:", "procesorze (CPU)" if URZADZENIE_DETEKTORA == "cpu"
              else f"karcie graficznej ({URZADZENIE_DETEKTORA.upper()})")
    return _modele["detektor_ai"]

def _czy_ai(etykieta):
    e = etykieta.lower().replace("-", " ").replace("_", " ").strip()
    if e.startswith(("hum", "real", "authentic", "natur", "orig")):
        return False
    if e == "ai" or e.startswith(("ai ", "fake", "artific", "generat", "synth", "deepfake", "machine")):
        return True
    return None

def pewnosc_ai(klatki):
    """Dla każdej klatki: pewność wstępnie nauczonego modelu (0..1), że obraz zrobiła AI."""
    global URZADZENIE_DETEKTORA
    obrazy = [Image.fromarray(cv2.cvtColor(k, cv2.COLOR_BGR2RGB)) for k in klatki]
    try:
        wyniki = _detektor()(obrazy, top_k=None, batch_size=8)
    except (RuntimeError, NotImplementedError):
        if URZADZENIE_DETEKTORA == "cpu":
            raise
        print("⚠️ Karta graficzna nie dała rady — liczę na procesorze.")
        URZADZENIE_DETEKTORA = "cpu"
        _modele.pop("detektor_ai", None)
        wyniki = _detektor()(obrazy, top_k=None, batch_size=8)
    pewnosci = []
    for wynik in wyniki:
        ai = [w["score"] for w in wynik if _czy_ai(w["label"]) is True]
        prawda = [w["score"] for w in wynik if _czy_ai(w["label"]) is False]
        if not ai and not prawda:
            raise ValueError("Nie rozumiem etykiet modelu: " + ", ".join(w["label"] for w in wynik))
        pewnosci.append(float(sum(ai)) if ai else 1.0 - float(sum(prawda)))
    return pewnosci

def _pliki_do_zbadania(plik=None):
    if plik:
        return [plik] if os.path.exists(plik) else []
    return sorted((p for p in glob.glob(os.path.join(FOLDER_NAGRAN, "*"))
                   if os.path.splitext(p)[1].lower() in ROZSZERZENIA_OBRAZOW + ROZSZERZENIA_WIDEO), key=os.path.getmtime)

def _plansza(okno, tekst):
    obraz = np.full((WYS + 120, SZER, 3), 25, np.uint8)
    napisz(obraz, tekst, 30, WYS // 2, 22, ZOLTY)
    okna.imshow(okno, obraz)
    okna.waitKey(1)

def detektyw_ai(plik=None, klatek=16, prog=0.5, werdykt=None, przerobka=None):
    """Model ocenia Wasze nagrania z folderu nagrania/. Klawisze: spacja – start/stop, a/d – plik,"""
    pliki = _pliki_do_zbadania(plik)
    if plik and not pliki:
        print(f"❌ Nie ma pliku: {plik}")
        return
    if not pliki:
        print("❌ Nie ma czego badać: nagrajcie próbkę z kamery (komórka wyżej) albo nagranie w studiu (LIVE 5).")
        return
    try:
        _detektor()
    except Exception as e:
        print("❌", e)
        return
    pamiec, oceny, wyniki_koncowe = {}, {}, {}
    stan = {"i": 0, "nr": 0, "gra": True, "przerobka": False, "werdykt": True}
    okno = "LIVE 6b - detektyw AI"
    okna.namedWindow(okno, cv2.WINDOW_NORMAL, pomoc="spacja – start/stop | a/d – plik | p – przeróbka | w – werdykt")
    okna.resizeWindow(okno, SZER, WYS + 120)

    def przerob(k):
        if not (stan["przerobka"] and przerobka):
            return k
        try:
            wynik = np.asarray(przerobka(k.copy()), dtype=np.float32)
            if wynik.ndim == 2:
                wynik = np.repeat(wynik[..., None], 3, axis=2)
            return np.clip(np.nan_to_num(wynik), 0, 255).astype(np.uint8)
        except Exception as e:
            print("⚠️ błąd w przerobka:", repr(e))
            stan["przerobka"] = False
            return k

    try:
        while True:
            sciezka = pliki[stan["i"]]
            if sciezka not in pamiec:
                _plansza(okno, f"Wczytuję: {os.path.basename(sciezka)}")
                klatki = [k if k.shape[1] <= SZER else cv2.resize(k, (SZER, int(k.shape[0] * SZER / k.shape[1])))
                          for k in _klatki_pliku(sciezka, 64)]
                pamiec[sciezka] = klatki or [np.zeros((WYS, SZER, 3), np.uint8)]
            klatki = pamiec[sciezka]
            klucz = (sciezka, stan["przerobka"])
            if klucz not in oceny:
                ktore = sorted(set(np.linspace(0, len(klatki) - 1, min(klatek, len(klatki))).round().astype(int).tolist()))
                _plansza(okno, f"Model AI ocenia {len(ktore)} klatek…")
                try:
                    pewnosci = pewnosc_ai([przerob(klatki[j]) for j in ktore])
                except Exception as e:
                    print("❌ Model nie zadziałał:", repr(e))
                    return
                try:
                    ocena = float(werdykt(pewnosci)) if werdykt else float(np.mean(pewnosci))
                except Exception as e:
                    print("⚠️ błąd w werdykt:", repr(e), "— liczę średnią")
                    ocena = float(np.mean(pewnosci))
                oceny[klucz] = (ktore, pewnosci, ocena)
            ktore, pewnosci, ocena = oceny[klucz]
            tytul = os.path.basename(sciezka)
            wyniki_koncowe[klucz] = (tytul, ocena)
            nr = stan["nr"] % len(klatki)
            obraz = dopasuj_do_okna(przerob(klatki[nr]))
            linie_tytulu = _zawin(f"plik {stan['i'] + 1}/{len(pliki)}: {tytul}", 17, SZER - 370)
            napisz(obraz, linie_tytulu[0] + (" …" if len(linie_tytulu) > 1 else ""), 12, 10, 17, obrys=2)
            if stan["przerobka"]:
                napisz(obraz, "PRZERÓBKA WŁĄCZONA (p)", 12, 34, 14, ZOLTY, obrys=2)
            if stan["werdykt"]:
                jest_ai = ocena >= prog
                prostokat(obraz, SZER - 350, 8, SZER - 8, 76, (20, 20, 20), 0.8)
                napisz(obraz, "MODEL: " + ("AI" if jest_ai else "PRAWDZIWE"), SZER - 340, 14, 24,
                       (70, 70, 255) if jest_ai else (80, 220, 120))
                napisz(obraz, f"pewność „AI”: {ocena:.0%}", SZER - 340, 48, 16, BIALY)
            dol = np.full((120, SZER, 3), 32, np.uint8)
            napisz(dol, "spacja – start/stop | a/d – plik | p – przeróbka | w – werdykt | q – koniec", 10, 6, 13, (220, 220, 220))
            napisz(dol, f"pewność „AI” w {len(ktore)} klatkach (czerwone = powyżej progu {prog:.0%})", 10, 26, 13, (80, 180, 255))
            x0, x1, y0, y1 = 14, SZER - 14, 50, 112
            yp = int(y1 - prog * (y1 - y0))
            cv2.line(dol, (x0, yp), (x1, yp), (120, 120, 120), 1)
            szer_slupka = max(3, int((x1 - x0) / max(len(klatki), 1)))
            for j, p in zip(ktore, pewnosci):
                x = int(x0 + (x1 - x0) * j / max(len(klatki) - 1, 1))
                cv2.rectangle(dol, (x - szer_slupka // 2, int(y1 - p * (y1 - y0))), (x + szer_slupka // 2, y1),
                              (70, 70, 255) if p >= prog else (80, 220, 120), -1)
            xn = int(x0 + (x1 - x0) * nr / max(len(klatki) - 1, 1))
            cv2.line(dol, (xn, y0 - 4), (xn, y1 + 4), BIALY, 1)
            okna.imshow(okno, np.vstack([obraz, dol]))
            klawisz = okna.waitKey(80)
            klawisz = -1 if klawisz < 0 else klawisz & 0xFF
            if klawisz in (ord("q"), 27):
                break
            if klawisz == ord(" "):
                stan["gra"] = not stan["gra"]
            if klawisz in (ord("a"), ord("d")):
                stan.update(i=(stan["i"] + (1 if klawisz == ord("d") else -1)) % len(pliki), nr=0)
            for znak, nazwa in ((ord("p"), "przerobka"), (ord("w"), "werdykt")):
                if klawisz == znak:
                    stan[nazwa] = not stan[nazwa]
            if stan["gra"]:
                stan["nr"] += 1
            try:
                if okna.getWindowProperty(okno, cv2.WND_PROP_VISIBLE) < 1:
                    break
            except cv2.error:
                break
    finally:
        okna.destroyAllWindows()
        for _ in range(5):
            okna.waitKey(1)
    for (sciezka, z_przerobka), (tytul, ocena) in wyniki_koncowe.items():
        print(f"{'AI' if ocena >= prog else 'prawdziwe':9s} {ocena:4.0%}  {tytul}" + (" (z przeróbką)" if z_przerobka else ""))

def _jako_klatki(wynik):
    """Sprawdza, co zwróciła funkcja wczytaj_klatki. Zwraca (lista klatek, komunikat błędu albo None)."""
    if wynik is None:
        return [], "funkcja nic nie zwraca (brakuje return klatki?)"
    if isinstance(wynik, np.ndarray) and wynik.ndim == 3:
        wynik = [wynik]
    try:
        wynik = list(wynik)
    except TypeError:
        return [], f"zwraca {type(wynik).__name__} zamiast listy klatek"
    klatki = []
    for k in wynik:
        if isinstance(k, tuple) and len(k) == 2 and isinstance(k[1], np.ndarray):
            return [], "na liście są pary (ok, klatka) — dodawajcie samą klatkę: klatki.append(klatka)"
        if not isinstance(k, np.ndarray) or k.ndim not in (2, 3) or k.size == 0:
            return [], f"na liście jest {type(k).__name__}, a powinny być same klatki (obrazy z wideo.read())"
        if k.ndim == 2:
            k = cv2.cvtColor(k, cv2.COLOR_GRAY2BGR)
        klatki.append(np.clip(k[..., :3], 0, 255).astype(np.uint8))
    if not klatki:
        return [], "lista klatek jest pusta — uzupełnijcie lukę 2 (klatki.append)"
    return klatki, None

def _kafelek(wynik, w, h, prog):
    kafel = np.full((h, w, 3), 38, np.uint8)
    mh = int(w * 9 / 16)
    if wynik.get("miniatura") is not None:
        m = wynik["miniatura"]
        s = min(w / m.shape[1], mh / m.shape[0])
        m = cv2.resize(m, (max(1, int(m.shape[1] * s)), max(1, int(m.shape[0] * s))), interpolation=cv2.INTER_AREA)
        y0, x0 = (mh - m.shape[0]) // 2, (w - m.shape[1]) // 2
        kafel[y0:y0 + m.shape[0], x0:x0 + m.shape[1]] = m
    linie = _zawin(wynik["nazwa"], 13, w - 12)
    napisz(kafel, linie[0] + (" …" if len(linie) > 1 else ""), 6, mh + 6, 13, BIALY)
    if wynik.get("blad"):
        for k, linia in enumerate(_zawin("⚠️ " + wynik["blad"], 12, w - 12)[:3]):
            napisz(kafel, linia, 6, mh + 26 + 16 * k, 12, ZOLTY)
        return kafel
    jest_ai = wynik["ocena"] >= prog
    kolor = (70, 70, 255) if jest_ai else (80, 220, 120)
    cv2.rectangle(kafel, (6, mh + 30), (w - 6, mh + 40), (70, 70, 70), -1)
    cv2.rectangle(kafel, (6, mh + 30), (6 + int((w - 12) * wynik["ocena"]), mh + 40), kolor, -1)
    napisz(kafel, f"model: {'AI' if jest_ai else 'PRAWDZIWE'} ({wynik['ocena']:.0%})", 6, mh + 46, 14, kolor)
    if wynik.get("trafil") is not None:
        napisz(kafel, "detektor trafił" if wynik["trafil"] else "detektor się pomylił", 6, mh + 68, 13,
               (80, 220, 120) if wynik["trafil"] else ZOLTY)
    return kafel

def zbadaj_probki(folder, znajdz, wczytaj, klatek=8, czy_trafil=None, prog=0.5):
    """Uruchamia Wasze funkcje wczytywania (luki 1–2) i pokazuje, co detektor AI mówi o każdym filmie z folderu."""
    if not os.path.isdir(folder):
        print(f"❌ Nie ma folderu '{folder}' obok notebooka. Poproście prowadzącego o próbki albo zmieńcie FOLDER_PROBEK.")
        return
    try:
        filmy = znajdz(folder)
    except Exception as e:
        print("❌ Błąd w znajdz_filmy (luka 1):", repr(e))
        return
    if filmy is None:
        print("❌ znajdz_filmy nic nie zwraca — na końcu ma być return filmy.")
        return
    filmy = [f for f in filmy]
    if not filmy:
        print(f"❌ Lista filmów jest pusta, a w folderze '{folder}' są pliki (liczba: {len(os.listdir(folder))}) — uzupełnijcie lukę 1.")
        return
    for f in filmy:
        if not isinstance(f, str) or not os.path.isfile(f):
            if isinstance(f, str) and os.path.isfile(os.path.join(folder, f)):
                print(f"❌ Na liście jest sama nazwa '{f}' — dodajcie folder: os.path.join(folder, nazwa).")
            else:
                print(f"❌ Na liście jest {f!r}, a to nie jest ścieżka do pliku.")
            return
    try:
        _detektor()
    except Exception as e:
        print("❌", e)
        return
    print(f"📂 Wasza funkcja znalazła pliki (liczba: {len(filmy)}). Detektor ocenia klatki z każdego (do {klatek})…")
    okno = "LIVE 6 - probki z folderu"
    okna.namedWindow(okno, cv2.WINDOW_NORMAL, pomoc="a/d – strona")
    okna.resizeWindow(okno, SZER, WYS + 120)
    wyniki, ostrzezenie_trafien = [], False
    try:
        for n, sciezka in enumerate(filmy, 1):
            nazwa = os.path.basename(sciezka)
            wynik = {"nazwa": nazwa, "miniatura": None, "blad": None, "ocena": 0.0, "klatek": 0, "trafil": None}
            wyniki.append(wynik)
            _plansza(okno, f"Wczytuję {n}/{len(filmy)}: {nazwa}")
            try:
                klatki, blad = _jako_klatki(wczytaj(sciezka, klatek))
            except Exception as e:
                klatki, blad = [], f"błąd w wczytaj_klatki (luka 2): {e!r}"
            if blad:
                if "pusta" in blad and os.path.splitext(nazwa)[1].lower() not in ROZSZERZENIA_WIDEO:
                    blad = "to nie jest film — luka 1 ma przepuszczać tylko filmy"
                wynik["blad"] = blad
                continue
            if len(klatki) > 32:
                klatki = [klatki[j] for j in np.linspace(0, len(klatki) - 1, 32).round().astype(int)]
            klatki = [k if k.shape[1] <= SZER else cv2.resize(k, (SZER, max(1, int(k.shape[0] * SZER / k.shape[1]))))
                      for k in klatki]
            _plansza(okno, f"Model AI ocenia {nazwa} ({len(klatki)} klatek)…")
            try:
                pewnosci = pewnosc_ai(klatki)
            except Exception as e:
                print("❌ Model nie zadziałał:", repr(e))
                return
            wynik.update(ocena=float(np.mean(pewnosci)), klatek=len(klatki), miniatura=klatki[len(klatki) // 2])
            if czy_trafil is not None:
                try:
                    t = czy_trafil(nazwa, wynik["ocena"] >= prog)
                    wynik["trafil"] = None if t is None else bool(t)
                except Exception as e:
                    if not ostrzezenie_trafien:
                        print("⚠️ błąd w czy_trafil (luka 3):", repr(e))
                    ostrzezenie_trafien = True
        kol, rz, gora = 3, 2, 40
        w, h = (SZER - 40) // kol, (WYS + 120 - gora - 36) // rz
        strony = max(1, -(-len(wyniki) // (kol * rz)))
        strona = 0
        while True:
            ekran = np.full((WYS + 120, SZER, 3), 25, np.uint8)
            napisz(ekran, f"Próbki z folderu {folder}/ · strona {strona + 1}/{strony}", 12, 10, 18, BIALY)
            for j, wynik in enumerate(wyniki[strona * kol * rz:(strona + 1) * kol * rz]):
                x, y = 10 + (j % kol) * (w + 10), gora + (j // kol) * (h + 6)
                ekran[y:y + h, x:x + w] = _kafelek(wynik, w, h, prog)
            napisz(ekran, "a/d – strona | q – koniec", 12, WYS + 120 - 26, 13, (220, 220, 220))
            okna.imshow(okno, ekran)
            klawisz = okna.waitKey(100)
            klawisz = -1 if klawisz < 0 else klawisz & 0xFF
            if klawisz in (ord("q"), 27):
                break
            if klawisz in (ord("a"), ord("d")):
                strona = (strona + (1 if klawisz == ord("d") else -1)) % strony
            try:
                if okna.getWindowProperty(okno, cv2.WND_PROP_VISIBLE) < 1:
                    break
            except cv2.error:
                break
    finally:
        okna.destroyAllWindows()
        for _ in range(5):
            okna.waitKey(1)
    print(f"{'plik':34s} {'klatek':>6s}  {'model mówi':12s} {'pewność AI':>10s}  trafił?")
    for wynik in wyniki:
        if wynik["blad"]:
            print(f"{wynik['nazwa'][:34]:34s}   ⚠️ {wynik['blad']}")
            continue
        trafil = "—" if wynik["trafil"] is None else ("tak" if wynik["trafil"] else "NIE")
        print(f"{wynik['nazwa'][:34]:34s} {wynik['klatek']:6d}  {'AI' if wynik['ocena'] >= prog else 'prawdziwe':12s} "
              f"{wynik['ocena']:10.0%}  {trafil}")
    ocenione = [w_ for w_ in wyniki if w_["trafil"] is not None]
    if ocenione:
        dobrze = sum(w_["trafil"] for w_ in ocenione)
        print(f"🎯 Detektor miał rację: {dobrze} na {len(ocenione)} ({dobrze / len(ocenione):.0%}).")
    print("💡 Jeden film klatka po klatce obejrzycie w ⭐ LIVE 6b: PLIK = \"probki/nazwa.mp4\".")

print("✅ Silnik, część 6 gotowa: zamiana twarzy i detektyw AI")

# ▶ 0.9 Test kamery. Widzisz siebie? Zamknij obraz klawiszem q albo przyciskiem ■ Stop.
uruchom(lambda klatka, klawisz: klatka, "Test kamery")

## 🧑‍💻 Live coding

Prowadzący pisze na rzutniku, Wy piszecie to samo u siebie. W każdej komórce są luki `____` - tylko tam wpisujecie kod.

### 1. Zielona kartka - okno do innego świata 🟩

Każdy piksel to trzy liczby: niebieski, zielony, czerwony. Liczymy **zieloność = G − max(R, B)** jest duża tylko dla zielonych pikseli. Tam, gdzie jest większa od progu, pokazujemy nowe tło.

Potrzebujecie: zieloną kartkę (od prowadzącego). Trzymajcie ją przed sobą albo przyłóżcie do kamery.

⌨️ `+`/`-` próg · `m` widok: wynik → zieloność → maska

In [ ]:
PROG = 40                         # od jakiej zieloności piksel jest tłem (w oknie: klawisze +/-)
TLO_PLIK = "tlo.jpg"               

def moja_zielonosc(klatka):
    b, g, r = cv2.split(klatka.astype(np.int16))     # trzy tablice liczb: niebieski, zielony, czerwony
    # 🧑‍💻 1 · o ile zielony jest większy od WIĘKSZEGO z czerwonego i niebieskiego?
    z = g - np.maximum(r, b)                          # ✅ 1
    # z = b - np.maximum(r, g)                        # ✅ chętni: blue screen (zamiast linijki wyżej)
    return np.clip(z, 0, 255)                         # ujemne (wcale nie zielone) → 0

def czy_tlo(z, prog):
    # 🧑‍💻 2 · piksel jest tłem, gdy jego zieloność jest większa niż prog 
    return z > prog                                   # ✅ 2 (z < prog: zostaje tylko kartka)

tlo_l1 = wczytaj_tlo(TLO_PLIK)
WIDOKI_L1 = ["wynik", "zieloność", "maska"]
stanL1 = {"widok": 0, "prog": PROG}

def live1(klatka, klawisz):
    if klawisz in (ord("+"), ord("=")):
        stanL1["prog"] = min(200, stanL1["prog"] + 3)
    if klawisz == ord("-"):
        stanL1["prog"] = max(3, stanL1["prog"] - 3)
    if klawisz == ord("m"):
        stanL1["widok"] = (stanL1["widok"] + 1) % len(WIDOKI_L1)
    prog = stanL1["prog"]
    z = sprawdz_mape(moja_zielonosc(cv2.GaussianBlur(klatka, (0, 0), 1.0)), klatka, "moja_zielonosc")
    tlo = sprawdz_mape(czy_tlo(z, prog), klatka, "czy_tlo")
    alfa = cv2.GaussianBlur(1 - np.clip(tlo, 0, 1), (0, 0), 1.5)       # 1 = zostaje, 0 = nowe tło
    widok = WIDOKI_L1[stanL1["widok"]]
    if widok == "zieloność":
        obraz = cv2.applyColorMap(np.clip(z * 128 / prog, 0, 255).astype(np.uint8), cv2.COLORMAP_INFERNO)
        napisz(obraz, "jasne = bardzo zielone · ciemne = wcale nie zielone", 12, WYS - 34, 16, obrys=2)
    elif widok == "maska":
        obraz = pokaz_maske(alfa, "żółty = zostaje · fioletowy = tu wchodzi nowe tło")
    else:
        obraz = zloz(klatka, tlo_l1, alfa)
    napisz(obraz, f"próg: {prog} · nowe tło: {100 * tlo.mean():.0f}% kadru · widok: {widok}", 12, 12, 18, obrys=2)
    if tlo.mean() < 0.01 and widok == "wynik":
        napisz(obraz, "Pokażcie kamerze zieloną kartkę!", 12, 40, 22, ZOLTY, obrys=2)
    return obraz

if gotowe(moja_zielonosc, czy_tlo, proba=lambda: live1(_klatka_proby(), -1)):
    uruchom(live1, "LIVE 1 - green screen z kartki", pomoc="+/- – próg | m – widok")

### 2. Kod miny 🔢

Model widzi twarz jako **478 punktów** i **52 liczby miny** od 0 do 1. Tak działają filtry na Snapchacie i tak deepfake „kopiuje” Waszą minę na cudzą twarz.

⌨️ `n` numery punktów · `s` siatka · `p` panel miny

In [ ]:
def moja_mina(obraz, punkty, mina):
    # 🧑‍💻 1 · czerwony nos klauna na czubku nosa
    kolko(obraz, punkty[1], 18, CZERWONY)                                         # ✅ 1 · nos klauna
    # 🧑‍💻 2 · gdy usta są otwarte bardziej niż 0.5 — napis „AAA!” nad czołem
    if mina["jawOpen"] > 0.5:                                                     # ✅ 2
        napisz(obraz, "AAA!", punkty[10][0] - 40, punkty[10][1] - 90, 36, ZOLTY, obrys=3)
    return obraz

stanL2 = {"numery": False, "siatka": True, "panel": True}

def live2(klatka, klawisz):
    for klucz, nazwa in ((ord("n"), "numery"), (ord("s"), "siatka"), (ord("p"), "panel")):
        if klawisz == klucz:
            stanL2[nazwa] = not stanL2[nazwa]
    punkty, mina = wykryj_twarz(klatka)
    obraz = (klatka * 0.6).astype(np.uint8) if stanL2["siatka"] else klatka.copy()
    if punkty is None:
        napisz(obraz, "Nie widzę twarzy…", 20, 20, 26, obrys=2)
        return obraz
    if stanL2["siatka"]:
        rysuj_siatke(obraz, punkty, numery=stanL2["numery"])
    if stanL2["panel"]:
        panel_min(obraz, mina)
    obraz = sprawdz_obraz(moja_mina(obraz, punkty, mina), klatka, "moja_mina")
    napisz(obraz, f"{len(punkty)} punktów · {len(mina)} liczby miny", 12, 12, 20, ZOLTY, obrys=2)
    return obraz

if gotowe(moja_mina, proba=lambda: sprawdz_obraz(moja_mina(_klatka_proby(), _punkty_proby(), _mina_proby()),
                                                 _klatka_proby(), "moja_mina")):
    uruchom(live2, "LIVE 2 - kod miny", pomoc="n – numery | s – siatka | p – panel miny")

### 3. Zamiana twarzy 🔁

Twarz A jest pocięta na ok. 850 trójkątów, a każdy trójkąt rozciągamy na kształt twarzy B i odwrotnie. Potem wyrównujemy kolor i światło. Pozostaje do zrobienia wtapianie, czyli mieszanie przeniesionej twarzy z obrazem z kamery.

⌨️ `t` widok: zamiana → trójkąty → maska · `b` mieszanie · `c` kolor · `z` zapamiętaj twarz (gdy w kadrze jest jedna osoba) · `x` zapomnij

In [ ]:
ZAMIEN_OCZY = True                # False = każdy zostaje przy swoich oczach.
WYGLADZANIE = 0.6                 # 0 = punkty drżą, 0.9 = gładko, ale z opóźnieniem
ROZMYCIE_BRZEGU = 25              # 5 = szeroki, miękki brzeg · 60 = wąski, ostry brzeg

def wtop(twarz, tlo, alfa):
    # 🧑‍💻 1 · gdzie alfa = 1 — twarz, gdzie alfa = 0 — tło, a pomiędzy trochę jednego i drugiego
    # 🧑‍💻 2 · DUCH (twarz w połowie przezroczysta) i UFOLUDEK (twarz = twarz * (0.6, 1.3, 0.6))
    # twarz = twarz * (0.6, 1.3, 0.6)                              # ✅ 2 · UFOLUDEK (odkomentuj)
    # return twarz * alfa * 0.5 + tlo * (1 - alfa * 0.5)          # ✅ 2 · DUCH (odkomentuj zamiast linijki niżej)
    return twarz * alfa + tlo * (1 - alfa)                         # ✅ 1

WIDOKI_L3 = ["zamiana", "trójkąty", "maska"]
stanL3 = {"mieszanie": MIESZANIE.index("miękkie"), "kolor": KOLORY.index("kolor i światło"), "widok": 0,
          "poprzednie": [], "zapamietana": None}

def live3(klatka, klawisz):
    if klawisz == ord("b"):
        stanL3["mieszanie"] = (stanL3["mieszanie"] + 1) % len(MIESZANIE)
    if klawisz == ord("c"):
        stanL3["kolor"] = (stanL3["kolor"] + 1) % len(KOLORY)
    if klawisz == ord("t"):
        stanL3["widok"] = (stanL3["widok"] + 1) % len(WIDOKI_L3)
    twarze = wygladz_twarze(wykryj_twarze(klatka), stanL3["poprzednie"], WYGLADZANIE)
    stanL3["poprzednie"] = twarze
    if klawisz == ord("z") and twarze:
        stanL3["zapamietana"] = zapamietaj_twarz(klatka, twarze)
    if klawisz == ord("x"):
        stanL3["zapamietana"] = None
    widok = WIDOKI_L3[stanL3["widok"]]
    obraz = zamien_twarze(klatka, twarze, stanL3["zapamietana"], mieszanie=MIESZANIE[stanL3["mieszanie"]],
                          kolor=KOLORY[stanL3["kolor"]], wtapianie=wtop, rozmycie=ROZMYCIE_BRZEGU,
                          oczy=ZAMIEN_OCZY, pokaz_maske=widok == "maska")
    if widok == "trójkąty":
        for punkty in twarze:
            rysuj_trojkaty(obraz, punkty)
    if len(twarze) >= 2:
        opis = "zamiana: osoba z lewej ↔ osoba z prawej"
    elif twarze and stanL3["zapamietana"]:
        opis = "zamiana: zapamiętana twarz → osoba w kadrze"
    elif twarze:
        opis = "jedna twarz: z — zapamiętaj ją, potem przed kamerę siada druga osoba"
    else:
        opis = "nie widzę twarzy"
    napisz(obraz, opis, 12, 12, 18, obrys=2)
    napisz(obraz, f"mieszanie: {MIESZANIE[stanL3['mieszanie']]} · kolor: {KOLORY[stanL3['kolor']]} · widok: {widok}",
           12, 38, 14, ZOLTY, obrys=2)
    if widok == "maska":
        napisz(obraz, "zielone: wklejona twarz · czerwone: własne usta · niebieskie: odbicie lustrzane", 12, 60, 13, obrys=2)
    if stanL3["zapamietana"]:
        miniatura_twarzy(obraz, stanL3["zapamietana"])
    if len(twarze) >= 2 or (twarze and stanL3["zapamietana"]):
        podglad_kamery(obraz, klatka)
    prostokat(obraz, 0, WYS - 26, 362, WYS, ZOLTY, 0.9)
    napisz(obraz, "MATERIAŁ SYNTETYCZNY · TYLKO NA ŻYWO", 10, WYS - 22, 14, CZARNY)
    return obraz

if gotowe(wtop, proba=lambda: sprawdz_obraz(wtop(*_proba_twarzy()), _proba_twarzy()[0], "wtop")):
    uruchom(live3, "LIVE 3 - zamiana twarzy", pomoc="t – widok | b – mieszanie | c – kolor | z – zapamiętaj | x – zapomnij")
stanL3["zapamietana"] = None

### 4a. Kapibara-aktor 🎭

Kapibara ze zdjęcia naklejonego na bryłę 3D gra Waszą miną: liczby miny z zad 2 sterują teraz cudzą twarzą (*face reenactment*).

⌨️ `f` zdjęcie · `s` widok: zdjęcie → siatka → bryła · `p` podgląd · `m` mandarynka

In [ ]:
PROG_MRUGNIECIA = 0.45            # od jakiej liczby oko jest zamknięte
WYGLADZANIE = 0.6                 # 0 = drga, 0.9 = płynnie, ale z opóźnieniem

def mapuj_mine(mina):
    usmiech = (mina["mouthSmileLeft"] + mina["mouthSmileRight"]) / 2
    kapibara = {"pysk": przeskaluj(mina["jawOpen"], od=0.05, do=0.6), "usmiech": usmiech > 0.5,
                "rumience": usmiech > 0.7, "brwi": mina["browInnerUp"] > 0.4}
    # 🧑‍💻 1 · kapibara mruga razem z Wami (lustro: jej lewe oko to Wasze prawe)
    kapibara["oko_lewe"] = mina["eyeBlinkRight"] > PROG_MRUGNIECIA     # ✅ 1 (lustro)
    kapibara["oko_prawe"] = mina["eyeBlinkLeft"] > PROG_MRUGNIECIA     # ✅ 1
    # 🧑‍💻 2 · nadęte policzki → dymek „Puff!”, a potem dwa Wasze powiedzonka na inne miny
    if mina["cheekPuff"] > 0.3:                                         # ✅ 2 · dymki
        kapibara["dymek"] = "Puff!"
    if mina["mouthPucker"] > 0.5:
        kapibara["dymek"] = "Cmok!"
    if kapibara["brwi"] and kapibara["pysk"] > 0.5:
        kapibara["dymek"] = "Co?!"
    # kapibara["pysk"] = 1 - kapibara["pysk"]                          # ✅ odwrotna marionetka (odkomentuj)
    return kapibara

WIDOKI_L4 = ["zdjecie", "siatka", "bryla"]
stanL4 = {"widok": 0, "podglad": True, "mandarynka": MANDARYNKA_NA_GLOWIE, "foto": nastepne_foto(None)}

def live4(klatka, klawisz):
    if klawisz == ord("s"):
        stanL4["widok"] = (stanL4["widok"] + 1) % len(WIDOKI_L4)
    for klucz, nazwa in ((ord("p"), "podglad"), (ord("m"), "mandarynka")):
        if klawisz == klucz:
            stanL4[nazwa] = not stanL4[nazwa]
    if klawisz == ord("f"):
        stanL4["foto"] = nastepne_foto(stanL4["foto"])
    punkty, mina = wykryj_twarz(klatka)
    kapibara = mapuj_mine(mina) if punkty is not None else {}
    kapibara["mandarynka"] = stanL4["mandarynka"]
    obraz = kapibara_3d(punkty, kapibara, WYGLADZANIE, widok=WIDOKI_L4[stanL4["widok"]], foto=stanL4["foto"])
    if stanL4["podglad"]:
        podglad_kamery(obraz, klatka)
    podpis_zdjecia(obraz)
    skret, kiw, przech = _fk["poza"][:3]
    napisz(obraz, f"{FOTY[stanL4['foto']]['opis']} · widok: {WIDOKI_L4[stanL4['widok']]}", 12, 12, 16, obrys=2)
    napisz(obraz, f"skręt {skret:+.0f}°  kiwnięcie {kiw:+.0f}°  przechylenie {przech:+.0f}°", 12, 34, 14, ZOLTY, obrys=2)
    return obraz

if gotowe(mapuj_mine, proba=lambda: [dict(mapuj_mine(m)) for m in _miny_proby()]):
    uruchom(live4, "LIVE 4 - kapibara 3D", pomoc="f – zdjęcie | s – widok | p – podgląd | m – mandarynka")

### ⭐ 4b. Sowa 3D 🦉

Sowa nie ma żadnego zdjęcia: to bryły, chmura i światło liczone w każdej klatce. Głowa obraca się mocniej niż Wasza, jak u prawdziwej sowy.

⌨️ `s` widok: pióra → siatka → bryła · `p` podgląd · `m` mandarynka

In [ ]:
SOWA_WZMOCNIENIE = (3.0, 1.2, 1.8)   # Wasz skręt / kiwnięcie / przechylenie × te liczby = ruch głowy sowy · ✅ (-3.0, …) = w drugą stronę

def reakcje_sowy(sowa, mina):
    # 🧑‍💻 1 · szeroko otwarte usta → sowa huka: dymek „Hu hu!”
    if mina["jawOpen"] > 0.6:                          # ✅ 1
        sowa["dymek"] = "Hu hu!"
    # 🧑‍💻 2 · uśmiech → sowa mruży oba oczy z zadowolenia
    if sowa["usmiech"]:                                # ✅ 2
        sowa["oko_lewe"] = sowa["oko_prawe"] = True
    return sowa

WIDOKI_L4B = ["piora", "siatka", "bryla"]
stanL4b = {"widok": 0, "podglad": True, "mandarynka": MANDARYNKA_NA_GLOWIE}

def live4b(klatka, klawisz):
    if klawisz == ord("s"):
        stanL4b["widok"] = (stanL4b["widok"] + 1) % len(WIDOKI_L4B)
    for klucz, nazwa in ((ord("p"), "podglad"), (ord("m"), "mandarynka")):
        if klawisz == klucz:
            stanL4b[nazwa] = not stanL4b[nazwa]
    punkty, mina = wykryj_twarz(klatka)
    sowa = reakcje_sowy(mapuj_mine(mina), mina) if punkty is not None else {}
    sowa["mandarynka"] = stanL4b["mandarynka"]
    obraz = sowa_3d(punkty, sowa, globals().get("WYGLADZANIE", 0.6), widok=WIDOKI_L4B[stanL4b["widok"]])
    if stanL4b["podglad"]:
        podglad_kamery(obraz, klatka)
    skret, kiw, przech = _sowa["poza"][:3]
    napisz(obraz, f"Puchacz 3D · widok: {WIDOKI_L4B[stanL4b['widok']]}", 12, 12, 16, obrys=2)
    napisz(obraz, f"skręt {skret:+.0f}°  kiwnięcie {kiw:+.0f}°  przechylenie {przech:+.0f}°", 12, 34, 14, ZOLTY, obrys=2)
    return obraz

if gotowe(mapuj_mine, reakcje_sowy, proba=lambda: [dict(reakcje_sowy(dict(mapuj_mine(m)), m)) for m in _miny_proby()]):
    uruchom(live4b, "LIVE 4b - sowa 3D", pomoc="s – widok | p – podgląd | m – mandarynka")

### 5. Studio Kapibara TV 📺

Absurdalny, fikcyjny news z Waszą kapibarą jako prezenterem. Najpierw wpiszcie swój tytuł i pasek wiadomości, potem razem dopisujemy grafikę jak w telewizji.

⌨️ `1` foto · `2` sowa · `3` osoba · `f` zdjęcie · `r` nagrywanie start/stop (maks. 15 s)

In [ ]:
GRUPA = "grupa1"                  # nazwa pliku z nagraniem (bez spacji i polskich znaków)
KANAL = "KAPIBARA TV"
NAGLOWEK = "PILNE"
TYTUL = "Kapibary przejęły Politechnikę Bydgoską"
PASEK = "WTIiE is the best • Wykłady przeniesione do jacuzzi • Studenci zachowują spokój przed sesją"
TRYB = "foto"                     # "foto", "sowa" albo "osoba"
FOTO = "garnitur"                 # w trybie "foto": "garnitur" albo "trawa"
KLUCZ = "zielony"                 # w trybie "osoba": "zielony" albo "oba" (AI wytnie też ścianę obok — wolniej)
TLO_STUDIA = None                 # None = studio (w trybie "sowa": nocny las) albo np. "tlo.jpg"

def moja_nakladka(obraz):
    # 🧑‍💻 1 · pogoda w prawym górnym rogu: „POGODA: 25°C”
    napisz(obraz, "POGODA: 25°C", 650, 80, 20, BIALY, obrys=2)                         # ✅ 1
    # 🧑‍💻 2 · migający napis „BREAKING NEWS”: pół sekundy jest, pół sekundy go nie ma
    # 🧑‍💻 3 · potem razem: licznik widzów, który rośnie z czasem
    if int(time.time() * 2) % 2:                                                         # ✅ 2 · miga co pół sekundy
        napisz(obraz, "BREAKING NEWS", 650, 110, 18, CZERWONY, obrys=2)
    napisz(obraz, f"widzowie: {1000 + int(time.time() * 3) % 5000}", 650, 140, 16, BIALY, obrys=2)   # ✅ 3
    return obraz

tlo_studia = wczytaj_tlo(TLO_STUDIA)
stanL5 = {"tryb": TRYB, "foto": FOTO if FOTO in dostepne_foty() else nastepne_foto(FOTO)}

def live5(klatka, klawisz):
    stanL5["tryb"] = {ord("1"): "foto", ord("2"): "sowa", ord("3"): "osoba"}.get(klawisz, stanL5["tryb"])
    if klawisz == ord("f"):
        stanL5["foto"] = nastepne_foto(stanL5["foto"])
    if stanL5["tryb"] == "osoba":
        klatka, alfa = wytnij_osobe(klatka, KLUCZ)
        obraz = zloz(klatka, tlo_studia, alfa)
    else:
        punkty, mina = wykryj_twarz(klatka)
        postac = mapuj_mine(mina) if punkty is not None else {}
        if stanL5["tryb"] == "foto":
            obraz = kapibara_3d(punkty, postac, globals().get("WYGLADZANIE", 0.6), foto=stanL5["foto"], tlo=tlo_studia)
        else:
            obraz = sowa_3d(punkty, postac, globals().get("WYGLADZANIE", 0.6), tlo=tlo_studia if TLO_STUDIA else None)
    obraz = moja_nakladka(nakladki_tv(obraz))
    if stanL5["tryb"] == "foto":
        podpis_zdjecia(obraz)
    return obraz

if gotowe(mapuj_mine, moja_nakladka, proba=lambda: ([dict(mapuj_mine(m)) for m in _miny_proby()], sprawdz_obraz(
        moja_nakladka(nakladki_tv(_klatka_proby())), _klatka_proby(), "moja_nakladka"))):
    uruchom(live5, "LIVE 5 - Kapibara TV", pomoc="1 foto | 2 sowa | 3 osoba | f zdjęcie", nagrywanie=True, nazwa=GRUPA)

In [ ]:
# ▶ Odtworzenie ostatniego nagrania (q — koniec). Konkretny plik: odtworz("nagrania/nazwa.mp4")
odtworz()

### 6. Detektyw AI 🔎

Detektor to **wstępnie nauczona sieć neuronowa**: uczyła się na dziesiątkach tysięcy zdjęć prawdziwych i zrobionych przez AI. Piszemy, jak program bierze filmy z folderu `probki/` i klatki z każdego filmu.

⌨️ `a` / `d` poprzednia / następna strona · `q` koniec

In [ ]:
FOLDER_PROBEK = "probki"
KLATEK_Z_FILMU = 8                

def znajdz_filmy(folder):
    filmy = []
    for nazwa in sorted(os.listdir(folder)):
        # 🧑‍💻 1 · do listy tylko filmy .mp4, z pełną ścieżką do pliku
        if nazwa.lower().endswith(".mp4"):                     # ✅ 1 (bez lower() pominie plik .MP4)
            filmy.append(os.path.join(folder, nazwa))
    return filmy

def wczytaj_klatki(sciezka, ile):
    wideo = cv2.VideoCapture(sciezka)
    co_ile = max(1, int(wideo.get(cv2.CAP_PROP_FRAME_COUNT)) // ile)   # żeby objąć cały film
    klatki, nr = [], 0
    while len(klatki) < ile:
        ok, klatka = wideo.read()                                      # ok = False, gdy film się skończył
        if not ok:
            break
        # 🧑‍💻 2 · do listy co co_ile-tą klatkę
        if nr % co_ile == 0:                                   # ✅ 2
            klatki.append(klatka)
        nr += 1
    wideo.release()
    return klatki

def czy_trafil(nazwa, model_mowi_ai):
    # 🧑‍💻 3 · nazwa zaczyna się od ai_ albo prawda_ — True, gdy model ma rację
    return nazwa.startswith("ai_") == model_mowi_ai           # ✅ 3

if gotowe(znajdz_filmy, wczytaj_klatki, czy_trafil, proba=lambda: czy_trafil("ai_proba.mp4", True)):
    zbadaj_probki(FOLDER_PROBEK, znajdz_filmy, wczytaj_klatki, KLATEK_Z_FILMU, czy_trafil)

### 7. Filtr na twarz 🤡

Własny filtr jak na Snapchacie albo TikToku: okulary, wąsy i coś, co reaguje na minę. Wymyślcie postać — kosmita, pirat, kapibara…

⌨️ `n` numery punktów · `s` siatka

In [ ]:
def moj_filtr(obraz, punkty, mina):
    # 🧑‍💻 1 · okulary: szkła na tęczówkach (punkty 468 i 473), a między nimi linia
    kolko(obraz, punkty[468], 25, BIALY, 3)                                 # ✅ 1 · okulary
    kolko(obraz, punkty[473], 25, BIALY, 3)
    linia(obraz, punkty[468], punkty[473], BIALY, 3)
    # 🧑‍💻 2 · wąsy od kącików ust (punkty 61 i 291) w bok i w górę
    linia(obraz, punkty[61], punkty[61] + (-40, -15), CZARNY, 5)            # ✅ 2 · wąsy
    linia(obraz, punkty[291], punkty[291] + (40, -15), CZARNY, 5)
    # 🧑‍💻 3 · coś, co reaguje na minę: rumieńce przy uśmiechu, czułki kosmity, gwiazdki nad głową…
    if mina["mouthSmileLeft"] > 0.5:                                        # ✅ 3 · rumieńce przy uśmiechu
        kolko(obraz, punkty[50], 18, (150, 120, 255))
        kolko(obraz, punkty[280], 18, (150, 120, 255))
    # linia(obraz, punkty[103], punkty[103] + (-30, -70), (80, 220, 80), 4)   # ✅ inny pomysł: czułki kosmity
    # linia(obraz, punkty[332], punkty[332] + (30, -70), (80, 220, 80), 4)
    return obraz

stanL7 = {"numery": True, "siatka": True}

def live7(klatka, klawisz):
    if klawisz == ord("n"):
        stanL7["numery"] = not stanL7["numery"]
    if klawisz == ord("s"):
        stanL7["siatka"] = not stanL7["siatka"]
    punkty, mina = wykryj_twarz(klatka)
    obraz = (klatka * 0.55).astype(np.uint8) if stanL7["siatka"] else klatka.copy()
    if punkty is None:
        napisz(obraz, "Nie widzę twarzy…", 20, 20, 26, obrys=2)
        return obraz
    if stanL7["siatka"]:
        rysuj_siatke(obraz, punkty, numery=stanL7["numery"])
    obraz = sprawdz_obraz(moj_filtr(obraz, punkty, mina), klatka, "moj_filtr")
    napisz(obraz, f"{len(punkty)} punktów", 12, 12, 20, ZOLTY, obrys=2)
    return obraz

if gotowe(moj_filtr, proba=lambda: [sprawdz_obraz(moj_filtr(_klatka_proby(), _punkty_proby(), m), _klatka_proby(), "moj_filtr")
                                     for m in _miny_proby()]):
    uruchom(live7, "LIVE 7 - filtr na twarz", pomoc="n – numery punktów | s – siatka")

### 8. Licznik mrugnięć 👁️

Pierwsze deepfake'i prawie nie mrugały badacze wykrywali je właśnie po tym. Piszemy licznik mrugnięć i ostrzeżenie, gdy ktoś mruga podejrzanie rzadko.

⌨️ `x` liczymy od nowa

In [ ]:
PROG_MRUGNIECIA = 0.45            # licznik ma liczyć mrugnięcia, a nie zmrużone oczy

def oczy_zamkniete(mina):
    # 🧑‍💻 1 · oba oczy zamknięte: obie liczby mrugnięcia większe od progu
    return mina["eyeBlinkLeft"] > PROG_MRUGNIECIA and mina["eyeBlinkRight"] > PROG_MRUGNIECIA   # ✅ 1 (z „or” liczy też puszczanie oka)

def ostrzezenia(obraz, mina, na_minute):
    # 🧑‍💻 2 · mniej niż 5 mrugnięć na minutę → czerwony napis „PODEJRZANIE MAŁO MRUGNIĘĆ”
    if na_minute < 5:                                                         # ✅ 2
        napisz(obraz, "PODEJRZANIE MAŁO MRUGNIĘĆ", 12, 70, 18, CZERWONY, obrys=2)
    return obraz

stanL8 = {"mrugniecia": 0, "zamkniete": False, "start": time.time()}

def live8(klatka, klawisz):
    if klawisz == ord("x"):
        stanL8.update(mrugniecia=0, start=time.time())
    obraz = klatka.copy()
    punkty, mina = wykryj_twarz(klatka)
    if punkty is None:
        napisz(obraz, "Nie widzę twarzy…", 20, 20, 26, obrys=2)
        return obraz
    rysuj_siatke(obraz, punkty, numery=False, siatka=False)
    panel_min(obraz, mina)
    zamkniete = oczy_zamkniete(mina)
    if zamkniete and not stanL8["zamkniete"]:
        stanL8["mrugniecia"] += 1
    stanL8["zamkniete"] = zamkniete
    minuty = max((time.time() - stanL8["start"]) / 60, 1 / 60)
    na_minute = stanL8["mrugniecia"] / minuty
    napisz(obraz, f"mrugnięcia: {stanL8['mrugniecia']} · na minutę: {na_minute:.0f}", 12, 12, 20, ZOLTY, obrys=2)
    napisz(obraz, "oczy: ZAMKNIĘTE" if zamkniete else "oczy: otwarte", 12, 40, 16, obrys=2)
    return sprawdz_obraz(ostrzezenia(obraz, mina, na_minute), klatka, "ostrzezenia")

if gotowe(oczy_zamkniete, ostrzezenia, proba=lambda: [(bool(oczy_zamkniete(m)), sprawdz_obraz(
        ostrzezenia(_klatka_proby(), m, ile), _klatka_proby(), "ostrzezenia")) for m in _miny_proby() for ile in (3.0, 20.0)]):
    stanL8.update(mrugniecia=0, zamkniete=False, start=time.time())
    uruchom(live8, "LIVE 8 - mrugniecia", pomoc="x – liczymy od nowa")

### 9. Teleport 🟩

Relacja „na żywo” z miejsca, w którym nikt nie był: green screen z kalibracją i tło, które rysujemy kodem.

⌨️ `k` kalibracja · `+`/`-` próg · `m` widok · `z` poświata · `a` pomoc AI (wolniej)

In [ ]:
USUWAJ_POSWIATE = True            # False = zielona obwódka na włosach wraca
POMOC_AI = False                  # True = AI wytnie też ścianę obok materiału (wolniej)

def moje_tlo():
    obraz = np.full((WYS, SZER, 3), (235, 190, 120), np.uint8)    # niebo w kolorze (B, G, R)
    # 🧑‍💻 1 · trawa na dole ekranu i słońce w prawym górnym rogu
    prostokat(obraz, 0, WYS - 140, SZER, WYS, (70, 160, 60))          # ✅ 1 · trawa (kolory: B, G, R)
    kolko(obraz, (SZER - 130, 110), 60, (40, 210, 255))               # ✅ 1 · słońce
    # 🧑‍💻 2 · napis „NA ŻYWO Z PLANETY ARBUZ”, a potem coś od siebie: chmury, góry, drugie słońce…
    napisz(obraz, "NA ŻYWO Z PLANETY ARBUZ", 24, 60, 34, BIALY, obrys=3)   # ✅ 2
    for x in (140, 360):                                              # ✅ 2 · chmury z trzech kółek
        kolko(obraz, (x, 175), 30, BIALY)
        kolko(obraz, (x + 35, 163), 38, BIALY)
        kolko(obraz, (x + 72, 175), 28, BIALY)
    return obraz

WIDOKI_L9 = ["wynik", "zieloność", "maska"]
stanL9 = {"widok": 0, "poswiata": USUWAJ_POSWIATE, "ai": POMOC_AI, "info": "", "info_do": 0.0}
ZIELEN["prog"] = None

def live9(klatka, klawisz):
    if klawisz == ord("k"):
        ok = kalibruj_zielen(klatka)
        stanL9["info"] = "kalibracja OK" if ok else "nie widzę zielonego materiału – stańcie przed nim"
        stanL9["info_do"] = time.time() + 2.5
    if klawisz in (ord("+"), ord("=")):
        ZIELEN["prog"] = min(200, (ZIELEN["prog"] or 40) + 3)
    if klawisz == ord("-"):
        ZIELEN["prog"] = max(3, (ZIELEN["prog"] or 40) - 3)
    if klawisz == ord("m"):
        stanL9["widok"] = (stanL9["widok"] + 1) % len(WIDOKI_L9)
    if klawisz == ord("z"):
        stanL9["poswiata"] = not stanL9["poswiata"]
    if klawisz == ord("a"):
        stanL9["ai"] = not stanL9["ai"]

    widok = WIDOKI_L9[stanL9["widok"]]
    if widok == "zieloność":
        obraz = pokaz_zielonosc(zielonosc(klatka))
    else:
        klatka, alfa = wytnij_osobe(klatka, "oba" if stanL9["ai"] else "zielony", stanL9["poswiata"])
        if widok == "maska":
            obraz = pokaz_maske(alfa, "żółty = osoba · fioletowy = tło (zielony materiał)")
        else:
            obraz = zloz(klatka, tlo_l9, alfa)
    prog = ZIELEN["prog"] or 40
    napisz(obraz, f"próg zieloności: {prog:.0f} · poświata: {'usuwana' if stanL9['poswiata'] else 'zostaje'}"
                  f" · pomoc AI: {'tak' if stanL9['ai'] else 'nie'}", 12, 12, 18, obrys=2)
    if time.time() < stanL9["info_do"]:
        napisz(obraz, stanL9["info"], 12, 40, 18, ZOLTY, obrys=2)
    return obraz

if gotowe(moje_tlo, proba=lambda: sprawdz_obraz(moje_tlo(), _klatka_proby(), "moje_tlo")):
    tlo_l9 = dopasuj_kadr(moje_tlo())
    uruchom(live9, "LIVE 9 - teleport", pomoc="k – kalibracja | +/- – próg | m – widok | z – poświata | a – pomoc AI")